In [ ]:
# Escondendo Warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# EDA – Base das Irregularidades (1 célula por irregularidade) – otimizado para não crashar

Este notebook foi ajustado para **não estourar memória**:

- **Cálculo das irregularidades**: carrega e processa **somente as colunas necessárias** (base mínima).
- **Saída (DF + Parquet + PDF)**: para cada irregularidade, materializa um **dataset completo** com **todas as colunas do parquet original** + **a coluna da irregularidade** (e a coluna `dia_operacional` para análise).

Estrutura de saída:
- `outputs_irregularidades/pdfs/` → 1 PDF por irregularidade (mesmos gráficos)
- `outputs_irregularidades/dfs/` → 1 parquet por irregularidade



---

### Lista de Verificações e Irregularidades

1. Verificar registros de colaboradores com ocorrência “999” utilizada como trajeto (**irregularidade\_999\_como\_trajeto.pdf**)
2. Verificar ausência das horas presenciais mínimas no período selecionado (**irregularidade\_ausencia\_das\_horas\_presenciais\_minimas.pdf**)
3. Verificar ausência de descanso mínimo entre jornadas de trabalho (**irregularidade\_ausencia\_de\_descanso\_entre\_jornadas\_de\_trabalho.pdf**)
4. Verificar executivos com histórico de uso de “999” como trajeto (**irregularidade\_executivos\_com\_historico\_de\_999\_como\_trajeto.pdf**)
5. Verificar pessoas que não entraram no refeitório, mas passaram pelo grill (**irregularidade\_nao\_entrou\_no\_refeitorio\_mas\_passou\_no\_grill.pdf**)
6. Verificar pessoas sem cumprir os 3 dias presenciais mínimos em semanas com feriado prolongado (**irregularidade\_pessoas\_sem\_cumprir\_os\_3\_dias\_presenciais\_em\_semana\_de\_feriado\_prolongado.pdf**)
7. Verificar pessoas trabalhando além da jornada de trabalho permitida (**irregularidade\_pessoas\_trabalhando\_alem\_da\_jornada\_de\_trabalho.pdf**)
8. Verificar pessoas trabalhando sem registro de ponto (**irregularidade\_pessoas\_trabalhando\_sem\_ganhar.pdf**)
9. Verificar registro de ponto sem passagem pela portaria (**irregularidade\_ponto\_sem\_portaria.pdf**) 
10. Verificar registros de entrada/saída identificados na fábrica sem passagem pela portaria (**irregularidade\_registro\_sem\_portaria.pdf**) 
11. Verificar registros de pessoas desligadas que ainda aparecem como presentes na fábrica (**irregularidade\_registros\_de\_pessoas\_deligadas.pdf**)

> Solução: Fazer com que os cartões PKI’s parem de registrar presença; utilizar biometria para entrada na empresa e reconhecimento facial para entrada de refeitórios e pontos dentro da fábrica.

> Solução: Remover o Home-Office.

> Solução: Implementar uma nova coluna na base 'ronda' que mostre a localização da pessoa no Home-Office, ver se ele realmente está trabalhando na sua residência, como previsto pelo 'Termo de Adesão ao Trabalho Remoto.pdf'.


---

#### Contexto (o que está sendo analisado na auditoria)

* **registro/ponto dentro da fábrica sem evidência de portaria**
* **ponto “999” perto do período presencial** (trajeto/abertura/fechamento remoto)
* **uso indevido de crachá por terceiros** (proxy attendance)
* **descanso interjornada < 11h** (CLT art. 66) ([JusBrasil][1])
* **registro de desligados**
* e **pessoas trabalhando sem ponto** (válido para horistas, **não para executivos** — você está certo)

---

## Solução 1 — PKI deixa de “validar presença” + biometria/face no acesso e pontos internos

##### Business Case (bem direto)

Trocar “posse do crachá” por **prova forte de identidade** em momentos-chave:

* **Entrou na planta** (portaria/catraca) = biometria/face
* **Consumiu acesso interno** (refeitório/área/ponto) = face
  Isso derruba o vetor “**terceiro usando crachá**”, reduz “**registro/ponto sem portaria**” e melhora muito a **trilha auditável**. A ANPD trata biometria/face como tecnologia com riscos e necessidade de mitigação (privacidade, segurança, finalidade). ([Serviços e Informações do Brasil][2])
  E na LGPD, dado biométrico vinculado à pessoa é **dado sensível**, então precisa governança de tratamento. ([Planalto][3])

#### Escopo Anchieta (o seu)

* **5 entradas / 15 catracas**
* **8 refeitórios / 18 catracas**
* **10 pontos**
  ➡️ Total: **43 pontos de controle**

#### Preço (ordem de grandeza, sem prometer milagre)

##### 1) Hardware unitário (pra você ter referência “na ponta da língua”)

* **Terminal facial (ex.: Control iD iDFace / similares): ~R$ 1,1k–1,2k** ([Mercado Livre][4])
* **Catraca facial completa (quando precisa trocar): ~R$ 4,6k–7,2k** ([canalautomacao.com.br][5])
* **Relógio de ponto facial (ex.: Henry Prisma SF Facial): existe como REP-C/homologado** (preço varia por canal; use como linha de produto, não como orçamento fechado) ([Loja do Ponto][6])
* **REP-P facial (variações mais simples de mercado): ~R$ 1,1k–2,8k** ([canalautomacao.com.br][7])

> Se “ponto” for **registro de jornada**, tem que respeitar Portaria 671 (categorias REP e regras). ([Serviços e Informações do Brasil][8])

##### 2) Projeto “instalado e funcionando” (o que pesa de verdade)

Em ambiente industrial, o custo real inclui: infra (rede/energia), instalação, integração, servidores/monitoramento, contingência, cadastro de usuários e suporte.

Para dar referência de variação, uma matéria do ge cita pacote completo (catracas/servidores/módulos) **na faixa de R$ 1M–3M** dependendo do que já existe e da quantidade de equipamentos. ([ge][9])
**Fábrica não é estádio**, mas a lógica de custo é igual: *“já tem catraca/infra?”* muda tudo.

**Estimativa prática por ponto (instalado)** para você usar em business case:

* **R$ 10k–18k por ponto de controle** (retrofit + integração + operação inicial)

➡️ **Anchieta (43 pontos): ~R$ 430 mil a R$ 774 mil**
Se precisar **trocar muitas catracas**, tende a subir (pode passar de **R$ 1M**).

##### 3) OPEX (mensal)

* Tipicamente entra: suporte, manutenção, monitoramento, reposição, operação de cadastro e ajustes.
  Sem seus SLAs não dá pra cravar, mas **planeje algo como 2%–4% do CAPEX ao mês** em contratos mais completos (ou menor se for só manutenção e TI absorver parte).

#### Prazo (instalar e “funcionar de verdade”)

##### Instalação até go-live (Anchieta / 43 pontos / 3 turnos)

* **Levantamento + desenho + TI/rede:** 1–2 semanas
* **Infra + instalação física:** 2–4 semanas
* **Integração + testes + contingência + treinamento:** 2–3 semanas
  ✅ **Go-live típico: 5–9 semanas**

##### Estabilização (pra parar de dar dor)

* **2–4 semanas pós go-live** (fila em pico, falso-rejeito, exceções, suporte)

➡️ **Kickoff → estável: 7–13 semanas**

#### O que essa solução ataca (mapeado nas suas irregularidades)

**Ataca forte (melhora muito):**

* Registro/ponto **sem portaria coerente**
* Uso indevido de crachá por terceiros
* Presença interna sem evidência de acesso

**Ataca parcialmente (depende de política de 999):**

* “999 como trajeto” — você precisa **governança de elegibilidade** e/ou bloquear 999 para perfis não elegíveis

**Importante (governança do seu analytics):**

* “Trabalhando sem ganhar” **não usar para executivos** (porque não batem ponto) — use só para horistas.

### Solução 2 — Remover Home Office (matar 999 por política)

##### O que resolve

* **Corta rápido** a discussão e a incidência de “999 como trajeto” (efeito imediato no seu detector)

##### O que NÃO resolve (irregularidade dentro da planta continua)

* Crachá com terceiros
* Ponto batido por outra pessoa
* Registros internos sem portaria (se o controle físico continuar fraco)

##### Prazo e custo

* **CAPEX:** ~zero
* **Prazo de implantação:** 2–6 semanas (decisão + comunicação + exceções)
* **Tempo pra estabilizar:** 1–3 meses (aderência, impactos de clima/turnover/negociação)

### Recomendação objetiva para Anchieta (com o seu escopo)

1. **Solução 1 como base** (controle/identidade forte) + política clara de 999
2. **Solução 2** só como complemento se a empresa quiser reduzir especificamente o vetor “999”, mas ela **não substitui** controle de identidade dentro da planta.

#### “Pesquisa de campo” (o mínimo pra fechar orçamento e evitar erro de dimensionamento)

Em 48h você valida:

1. **Pico por turno** em cada uma das 5 entradas (pessoas/minuto)
2. Onde dá **retrofit** vs onde precisa **trocar catraca**
3. O que os “10 pontos” são: **REP (jornada)** ou **apenas controle de acesso** (muda regra e compliance) ([Serviços e Informações do Brasil][8])
4. Regra de **contingência** (sem rede / falha de leitura) + auditoria das exceções
5. Plano de **cadastro por turno** (senão trava operação)


## 1) Bibliotecas e diretórios de saída


In [ ]:
# Importando bibliotecas
from __future__ import annotations

from pathlib import Path
import re
import unicodedata
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


In [ ]:
# Diretórios de saída
OUT_ROOT = Path("./outputs_irregularidades")
PDF_DIR = OUT_ROOT / "pdfs"
DFS_DIR = OUT_ROOT / "dfs"

PDF_DIR.mkdir(parents=True, exist_ok=True)
DFS_DIR.mkdir(parents=True, exist_ok=True)

# Entrada (parquet da Ronda)
PARQUET_PATH = Path(r".\df_ronda.parquet")  # ajuste se necessário

# Se o seu notebook ainda estiver pesado, troque para False
MANTER_DF_EM_MEMORIA = False

print("OUT_ROOT:", OUT_ROOT)
print("PDF_DIR :", PDF_DIR)
print("DFS_DIR :", DFS_DIR)
print("PARQUET :", PARQUET_PATH)


## 2) Parâmetros, regex e semanas de feriado prolongado


In [ ]:
# Parâmetros (mantendo a lógica)
janela_portaria_min = 120
tolerancia_entrada_min = 30
tolerancia_saida_min = 30
janela_999_horas = 6

hora_corte_operacional = 11

descanso_min_horas = 11
jornada_min_horas = 10
jornada_min_minutos = 50

usar_jornada_por_portaria = True
max_jornada_portaria_horas = 20

# Regex
regex_portaria = r"\bPORT(?:ARIA)?\b|\bCANCELA\b|\bCANCELA\s*-\s*PORT\b|\bCANCELA VEICULO\s+[A-Z0-9]+\b"
regex_refeitorio = r"\bREFEIT(?:ÓRIO|ORIO)?\b|\bREFEITORIO\b"
regex_grill = r"\bGRILL\b"
regex_ponto = r"\bMARCA\b|\bPONTO\b|\bPONT\b"  # Ponto = MARCA / PONTO / PONT

# Semanas "feriado prolongado"
semanas_feriado_prolongado = [
    (pd.Timestamp("2025-04-28"), pd.Timestamp("2025-05-04")),
    (pd.Timestamp("2025-06-16"), pd.Timestamp("2025-06-22")),
    (pd.Timestamp("2025-11-17"), pd.Timestamp("2025-11-23")),
    (pd.Timestamp("2025-12-22"), pd.Timestamp("2025-12-28")),
]

# Timedeltas
t_jan_port = pd.Timedelta(minutes=janela_portaria_min)
t_tol_ent = pd.Timedelta(minutes=tolerancia_entrada_min)
t_tol_sai = pd.Timedelta(minutes=tolerancia_saida_min)
t_jan_999 = pd.Timedelta(hours=janela_999_horas)

t_desc_min = pd.Timedelta(hours=descanso_min_horas)
t_jornada_min = pd.Timedelta(hours=jornada_min_horas, minutes=jornada_min_minutos)

t_gap_work = pd.Timedelta(hours=2)
t_max_jornada_port = pd.Timedelta(hours=max_jornada_portaria_horas)


## 3) Leitura mínima (somente colunas necessárias para calcular as irregularidades)


In [ ]:
# Colunas mínimas para cálculo (não é para relatório nem export final)
MIN_COLS = [
    "cadastro",
    "nome",
    "data",
    "hora",
    "dir",
    "planta",
    "descricao coletor",
    "cargo",
    "área",
    "heranca",
    "desligado",
    "Data_Dem",
]

# Validação rápida
# (Se faltar alguma dessas, o parquet não atende a lógica atual)
try:
    try:
        # usa backend Arrow para reduzir pico de memória em colunas texto
        df_min_raw = pd.read_parquet(PARQUET_PATH, columns=MIN_COLS, dtype_backend="pyarrow")
    except TypeError:
        df_min_raw = pd.read_parquet(PARQUET_PATH, columns=MIN_COLS)
except Exception as e:
    raise RuntimeError(
        "Falha ao ler parquet com colunas mínimas. "
        "Verifique se o arquivo existe e se o engine de parquet está instalado (pyarrow/fastparquet)."
    ) from e

missing = [c for c in MIN_COLS if c not in df_min_raw.columns]
if missing:
    raise ValueError(f"Parquet não tem colunas mínimas necessárias: {missing}")

# Remove duplicatas
df_min_raw = df_min_raw.drop_duplicates().reset_index(drop=True)

# ID posicional para materialização com iloc no parquet completo
df_min_raw["_row_id"] = np.arange(len(df_min_raw), dtype=np.int64)

print("Linhas (parquet):", len(df_min_raw))
print("Colunas carregadas (mínimo):", len(df_min_raw.columns))
df_min_raw.head(3)


## 4) Pré-processamento mínimo (flags e dia operacional)


In [ ]:
# Copiando base
df = df_min_raw.copy()

# cadastro como string
df["cadastro"] = df["cadastro"].astype(str).str.strip()

# descrição
desc = df["descricao coletor"].fillna("").astype(str)

df["Portaria"] = desc.str.contains(regex_portaria, case=False, na=False, regex=True)
df["Ponto"] = desc.str.contains(regex_ponto, case=False, na=False, regex=True)
df["_REFEITORIO"] = desc.str.contains(regex_refeitorio, case=False, na=False, regex=True)
df["_GRILL"] = desc.str.contains(regex_grill, case=False, na=False, regex=True)

# planta == 999 (robusto)
planta_num = pd.to_numeric(df["planta"], errors="coerce")
df["999"] = planta_num.eq(999)

# Presencial: planta válida e não 999
df["Presencial"] = planta_num.notna() & (~df["999"])

# Executivo: cargo == 'E'
cargo_norm = df["cargo"].fillna("").astype(str).str.strip().str.upper()
df["_is_executivo"] = cargo_norm.eq("E")

# dir normalizado
dir_norm = df["dir"].fillna("").astype(str).str.strip().str.upper()
dir_norm = dir_norm.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
dir_norm = dir_norm.where(~dir_norm.str.startswith("E"), "E")
dir_norm = dir_norm.where(~dir_norm.str.startswith("S"), "S")
df["_dir_norm"] = dir_norm

# dt_ronda robusto (data + hora)
data_dt = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")
hora_s = df["hora"].fillna("00:00").astype(str).str.strip()
hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
hora_td = pd.to_timedelta(hora_s, errors="coerce")

df["_dia_key"] = data_dt.dt.normalize()
df["_dt_ronda"] = df["_dia_key"] + hora_td

# Dia operacional base (corte às 11h)
df["_dia_operacional"] = (df["_dt_ronda"] - pd.Timedelta(hours=hora_corte_operacional)).dt.normalize()

# Mantém df sem ordenar globalmente para não “embaralhar” o _row_id,
# mas cria uma visão ordenada quando necessário.
print("OK: pré-processamento mínimo aplicado.")
df.head(3)


## 5) Ajuste de dia operacional por jornada Portaria (E->S) – mantém a lógica


In [ ]:
# Ajuste de dia operacional por jornada (Portaria E->S) - versão otimizada
if usar_jornada_por_portaria:
    cols_min = ["cadastro", "_dt_ronda", "Portaria", "_dir_norm", "999", "_dia_operacional"]
    df_min = df[cols_min].copy()

    # ordena e cria índice posicional (evita loc caro no loop)
    df_sorted = (
        df_min.sort_values(["cadastro", "_dt_ronda"], kind="mergesort")
              .reset_index(drop=False)   # guarda índice original em "index"
    )

    orig_idx = df_sorted["index"].to_numpy()
    times    = df_sorted["_dt_ronda"].to_numpy()  # datetime64[ns]
    port     = df_sorted["Portaria"].fillna(False).to_numpy(dtype=bool)
    dirn     = df_sorted["_dir_norm"].to_numpy()
    is999    = df_sorted["999"].fillna(False).to_numpy(dtype=bool)

    # array alvo (escrita rápida)
    dia_op = df_sorted["_dia_operacional"].to_numpy(copy=True)

    # garante que é gravável
    if not dia_op.flags.writeable:
        dia_op = dia_op.copy()
        dia_op.setflags(write=True)

    # converte janelas para numpy timedelta64[ns]
    tmax = np.timedelta64(int(pd.Timedelta(t_max_jornada_port).value), "ns")
    tjan = np.timedelta64(int(pd.Timedelta(t_jan_999).value), "ns")

    # iterar grupos por posições (mais rápido que fatiar DF)
    grp_pos = df_sorted.groupby("cadastro", sort=False).indices

    for cad, pos in grp_pos.items():
        t = times[pos]
        valid = ~np.isnat(t)
        if not valid.any():
            continue

        posv  = pos[valid]
        tv    = t[valid]
        portv = port[posv]
        dirv  = dirn[posv]
        is9v  = is999[posv]

        m_ent = portv & (dirv == "E")
        m_sai = portv & (dirv == "S")

        ent = tv[m_ent]
        sai = tv[m_sai]

        if ent.size == 0 or sai.size == 0:
            continue

        i = j = 0
        while i < ent.size and j < sai.size:
            dt_ent = ent[i]
            dt_sai = sai[j]

            if dt_sai < dt_ent:
                j += 1
                continue

            if (dt_sai - dt_ent) > tmax:
                i += 1
                continue

            # dia operacional = dia da entrada
            dia = dt_ent.astype("datetime64[D]").astype("datetime64[ns]")

            # intervalo core [ent, sai]
            l = np.searchsorted(tv, dt_ent, side="left")
            r = np.searchsorted(tv, dt_sai, side="right")
            if r > l:
                dia_op[posv[l:r]] = dia

            # 999 antes (ent - janela, ent)
            l2 = np.searchsorted(tv, dt_ent - tjan, side="left")
            r2 = np.searchsorted(tv, dt_ent, side="left")
            if r2 > l2:
                sl = slice(l2, r2)
                m9 = is9v[sl]
                if m9.any():
                    dia_op[posv[sl][m9]] = dia

            # 999 depois (sai, sai + janela]
            l3 = np.searchsorted(tv, dt_sai, side="right")
            r3 = np.searchsorted(tv, dt_sai + tjan, side="right")
            if r3 > l3:
                sl = slice(l3, r3)
                m9 = is9v[sl]
                if m9.any():
                    dia_op[posv[sl][m9]] = dia

            i += 1
            j += 1

    # grava de volta no df original pelo índice original
    s = pd.Series(dia_op, index=orig_idx, name="_dia_operacional").reindex(df.index)
    df["_dia_operacional"] = s.to_numpy()


# limpa temporários pesados desta etapa
for _v in ("df_min", "df_sorted", "orig_idx", "times", "port", "dirn", "is999", "dia_op", "grp_pos"):
    if _v in globals():
        del globals()[_v]
gc.collect()

print("OK: _dia_operacional ajustado por jornada (Portaria E->S) [otimizado].")


## 6) Tabela day (features por cadastro + dia_operacional) – mínima


In [ ]:
# =========================
# Considera apenas registros com dt válido (sem cópia profunda)
# =========================

# (Se _dt_ronda já for datetime64, ótimo. Se for object, converta antes uma vez:)
# df["_dt_ronda"] = pd.to_datetime(df["_dt_ronda"], errors="coerce")

mask_dt = df["_dt_ronda"].notna()

COLS_KEEP = [
    "cadastro", "_dia_operacional", "_dt_ronda",
    "Presencial", "Ponto", "Portaria",
    "_REFEITORIO", "_GRILL", "_is_executivo",
    "_dir_norm",
    "999",
]

# SEM .copy() aqui (era isso que explodia a RAM)
df_valid = df.loc[mask_dt, COLS_KEEP]

# Min/Max por cadastro (anti-truncamento)
cad_minmax = (
    df_valid.groupby("cadastro", sort=False)["_dt_ronda"]
      .agg(min_dt_cad="min", max_dt_cad="max")
      .reset_index()
)

# day_any (qualquer presencial)
pres_any = df_valid.loc[df_valid["Presencial"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda"]]
day_any = (
    pres_any.groupby(["cadastro", "_dia_operacional"], as_index=False, sort=False)
            .agg(
                first_any_pres=("_dt_ronda", "min"),
                last_any_pres=("_dt_ronda", "max"),
                n_any_pres=("_dt_ronda", "size"),
            )
)
day_any["dur_any_pres"] = day_any["last_any_pres"] - day_any["first_any_pres"]

# day_ponto (ponto presencial)
pres_ponto = df_valid.loc[
    df_valid["Ponto"].eq(True) & df_valid["Presencial"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda"]
]
day_ponto = (
    pres_ponto.groupby(["cadastro", "_dia_operacional"], as_index=False, sort=False)
              .agg(first_ponto_pres=("_dt_ronda", "min"), last_ponto_pres=("_dt_ronda", "max"))
)

# presença de ponto (qualquer) no dia
day_has_ponto_any = (
    df_valid.loc[df_valid["Ponto"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_PONTO_ANY=True)
)

# presença de portaria no dia (qualquer)
day_has_port = (
    df_valid.loc[df_valid["Portaria"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_PORTARIA_ANY=True)
)

# refeitório/grill no dia
day_has_ref = (
    df_valid.loc[df_valid["_REFEITORIO"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_REFEITORIO_ANY=True)
)

day_has_grill = (
    df_valid.loc[df_valid["_GRILL"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_GRILL_ANY=True)
)

# executivo no dia
day_is_exec = (
    df_valid.loc[df_valid["_is_executivo"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(IS_EXECUTIVO_DAY=True)
)

# Monta day base
day = day_any.merge(day_ponto, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_ponto_any, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_port, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_ref, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_grill, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_is_exec, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(cad_minmax, on="cadastro", how="left")

for c in ["HAS_PONTO_ANY", "HAS_PORTARIA_ANY", "HAS_REFEITORIO_ANY", "HAS_GRILL_ANY", "IS_EXECUTIVO_DAY"]:
    day[c] = day[c].fillna(False).astype(bool)

# SEM_COBERTURA_CONTEXTO
day["w_start_needed"] = day["first_any_pres"] - t_jan_port
day["w_end_needed"] = day["last_any_pres"] + t_jan_port
day["SEM_COBERTURA_CONTEXTO"] = False
mask_mm = day["min_dt_cad"].notna() & day["max_dt_cad"].notna()
day.loc[mask_mm, "SEM_COBERTURA_CONTEXTO"] = (day["w_start_needed"] < day["min_dt_cad"]) | (day["w_end_needed"] > day["max_dt_cad"])

# ok_entry / ok_exit (portaria coerente)
port = df_valid.loc[df_valid["Portaria"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]].copy(deep=False)
if port.empty:
    day["ok_entry"] = False
    day["ok_exit"] = False
else:
    port = port.merge(
        day[["cadastro", "_dia_operacional", "w_start_needed", "w_end_needed", "first_any_pres", "last_any_pres"]],
        on=["cadastro", "_dia_operacional"],
        how="inner",
    )

    in_window = (port["_dt_ronda"] >= port["w_start_needed"]) & (port["_dt_ronda"] <= port["w_end_needed"])

    port["ok_entry"] = (
        in_window
        & port["_dir_norm"].eq("E")
        & (port["_dt_ronda"] <= (port["first_any_pres"] + t_tol_ent))
    )

    port["ok_exit"] = (
        in_window
        & port["_dir_norm"].eq("S")
        & (port["_dt_ronda"] >= (port["last_any_pres"] - t_tol_sai))
    )

    ok = port.groupby(["cadastro", "_dia_operacional"], sort=False)[["ok_entry", "ok_exit"]].any().reset_index()
    day = day.merge(ok, on=["cadastro", "_dia_operacional"], how="left")
    day[["ok_entry", "ok_exit"]] = day[["ok_entry", "ok_exit"]].fillna(False)

# SEM_PORTARIA_COERENTE
day["SEM_PORTARIA_COERENTE"] = np.where(
    day["IS_EXECUTIVO_DAY"],
    ~day["HAS_PORTARIA_ANY"],
    ~(day["ok_entry"] | day["ok_exit"]),
)

# SEM_COBERTURA_CONTEXTO desliga a regra
day.loc[day["SEM_COBERTURA_CONTEXTO"], "SEM_PORTARIA_COERENTE"] = False

# =========================
# Regra grill/refeitório — REFEITÓRIO ANTES DO PRIMEIRO GRILL
# =========================

first_grill = (
    df_valid.loc[df_valid["_GRILL"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda"]]
      .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
      .min()
      .rename("first_grill")
      .reset_index()
)

ref_rows = (
    df_valid.loc[df_valid["_REFEITORIO"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda"]]
      .merge(first_grill, on=["cadastro", "_dia_operacional"], how="inner")
)

ref_before = (
    ref_rows.loc[ref_rows["_dt_ronda"] < ref_rows["first_grill"], ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_REFEITORIO_BEFORE_GRILL=True)
)

day = day.merge(ref_before, on=["cadastro", "_dia_operacional"], how="left")
day["HAS_REFEITORIO_BEFORE_GRILL"] = day["HAS_REFEITORIO_BEFORE_GRILL"].fillna(False).astype(bool)

day["Irregularidade - Não entrou no refeitório, mas passou no grill"] = (
    day["HAS_GRILL_ANY"].fillna(False).astype(bool)
    & (~day["HAS_REFEITORIO_BEFORE_GRILL"])
)

print("OK: day construído.")
day.head(3)

## 7) Helpers: slugify, relatório PDF e export por row_id


In [ ]:
# Limpa df pesado para liberar memória (se necessário)
BATCH_SIZE = 250_000  # usado em loops de chunk/leitura/escrita nesta célula

def slugify(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return text.lower()

IRREGULARIDADES_SEM_PLANTA_999 = {
    "irregularidade_ausencia_das_horas_presenciais_minimas",
    "irregularidade_pessoas_trabalhando_sem_ganhar",
    "irregularidade_ponto_sem_portaria",
    "irregularidade_registro_sem_portaria",
}

def _deve_excluir_planta_999(
    irregularidade_col: str | None = None,
    titulo: str | None = None,
    pdf_path: Path | None = None,
) -> bool:
    candidatos = []

    if irregularidade_col:
        candidatos.append(slugify(irregularidade_col))

    if titulo:
        candidatos.append(slugify(titulo))

    if pdf_path is not None:
        try:
            candidatos.append(slugify(Path(pdf_path).stem))
        except Exception:
            pass
        candidatos.append(slugify(str(pdf_path)))

    return any(
        alvo in cand
        for cand in candidatos
        for alvo in IRREGULARIDADES_SEM_PLANTA_999
    )

def _mask_planta_not_999(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip()
    s_num = pd.to_numeric(s, errors="coerce")
    mask_999 = s_num.eq(999) | s.str.match(r"^999(\D|$)", na=False)
    return ~mask_999

def _filtrar_df_sem_planta_999(df_: pd.DataFrame | None) -> pd.DataFrame | None:
    if not isinstance(df_, pd.DataFrame):
        return df_
    if df_.empty:
        return df_
    if "planta" not in df_.columns:
        return df_
    return df_.loc[_mask_planta_not_999(df_["planta"])].copy()

def _filtrar_row_ids_sem_planta_999(row_ids: np.ndarray) -> np.ndarray:
    if row_ids is None:
        return np.array([], dtype=np.int64)

    row_ids = np.asarray(row_ids, dtype=np.int64)
    if row_ids.size == 0:
        return row_ids

    if "planta" not in df_min_raw.columns:
        return row_ids

    row_ids = row_ids[(row_ids >= 0) & (row_ids < len(df_min_raw))]
    if row_ids.size == 0:
        return row_ids

    cache_name = "_PLANTA_NUM_CACHE"
    cache = globals().get(cache_name)

    if not isinstance(cache, np.ndarray) or len(cache) != len(df_min_raw):
        cache = pd.to_numeric(df_min_raw["planta"], errors="coerce").to_numpy()
        globals()[cache_name] = cache

    plantas = cache[row_ids]
    keep = ~pd.Series(plantas).eq(999).to_numpy()
    return row_ids[keep]

def _pick_col(df_: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df_.columns:
            return c
    return None


def _render_capa_padrao_pdf(
    pdf,
    titulo: str,
    gen_date: str,
    metricas: list[tuple[str, str]],
    periodo: str | None = None,
    concentracoes: list[tuple[str, str]] | None = None,
    top3_piores: list[str] | None = None,
    top3_atuais: list[str] | None = None,
    janela_info: str | None = None,
    extras: list[str] | None = None,
) -> None:
    fig = plt.figure(figsize=(11.69, 8.27))
    ax = fig.add_subplot(111)
    ax.axis("off")

    ax.text(0.02, 0.96, titulo, fontsize=22, fontweight="bold", va="top")
    ax.text(0.02, 0.91, f"Gerado em: {gen_date}", fontsize=12, va="top")

    y = 0.85
    for rotulo, valor in metricas:
        ax.text(0.02, y, f"• {rotulo}: {valor}", fontsize=14, va="top")
        y -= 0.045

    if periodo:
        ax.text(0.02, y, f"• Período: {periodo}", fontsize=12.5, va="top")
        y -= 0.05

    if concentracoes:
        for rotulo, valor in concentracoes:
            ax.text(0.02, y, f"• {rotulo}: {valor}", fontsize=12.5, va="top")
            y -= 0.043

    ax.text(0.02, y, "• Top 3 piores (quantidade e % do total):", fontsize=12.5, fontweight="bold", va="top")
    y -= 0.04
    if top3_piores:
        for i, linha in enumerate(top3_piores[:3], start=1):
            ax.text(0.03, y, f"{i}. {linha}", fontsize=11.5, va="top")
            y -= 0.037
    else:
        ax.text(0.03, y, "Sem dados suficientes.", fontsize=11.5, va="top")
        y -= 0.037

    janela_txt = janela_info if janela_info else "base indisponível"
    ax.text(
        0.02,
        y,
        f"• Top 3 atuais (últimos 30 dias) — base da janela: {janela_txt}",
        fontsize=12.2,
        fontweight="bold",
        va="top",
    )
    y -= 0.04
    if top3_atuais:
        for i, linha in enumerate(top3_atuais[:3], start=1):
            ax.text(0.03, y, f"{i}. {linha}", fontsize=11.0, va="top")
            y -= 0.034
    else:
        ax.text(0.03, y, "Sem dados suficientes na janela.", fontsize=11.0, va="top")
        y -= 0.034

    if extras:
        y -= 0.01
        for extra in extras:
            ax.text(0.02, y, f"• {extra}", fontsize=10.8, va="top")
            y -= 0.032

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

def gerar_relatorio_pdf_semanal_feriado(df_semanal: pd.DataFrame, titulo: str, pdf_path: Path) -> None:
    import datetime as _dt
    import numpy as np
    from matplotlib.backends.backend_pdf import PdfPages
    from matplotlib.colors import LinearSegmentedColormap
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    gen_date = _dt.date.today().strftime("%Y-%m-%d")

    if df_semanal is None or df_semanal.empty:
        with PdfPages(pdf_path) as pdf:
            _render_capa_padrao_pdf(
                pdf=pdf,
                titulo=titulo,
                gen_date=gen_date,
                metricas=[
                    ("Pessoa-semanas irregulares", "0"),
                    ("Pessoas únicas", "0"),
                    ("Semanas com ocorrência", "0"),
                ],
                periodo=None,
                concentracoes=[
                    ("Concentração Top 1", "0 (0.0%)"),
                    ("Concentração Top 3", "0 (0.0%)"),
                    ("Concentração Top 10", "0 (0.0%)"),
                ],
                top3_piores=[],
                top3_atuais=[],
                janela_info="0 (0.0% do total)",
                extras=["Sem linhas para gerar o relatório semanal."],
            )
        return

    dfw = df_semanal.copy()

    # -----------------------------
    # Normalização mínima
    # -----------------------------
    if "_semana_feriado_ini" in dfw.columns and "semana_inicio" not in dfw.columns:
        dfw = dfw.rename(columns={"_semana_feriado_ini": "semana_inicio"})

    if "área" in dfw.columns and "area" not in dfw.columns:
        dfw = dfw.rename(columns={"área": "area"})

    if "semana_inicio" not in dfw.columns:
        raise ValueError("df_semanal precisa ter a coluna 'semana_inicio'.")

    dfw["semana_inicio"] = pd.to_datetime(dfw["semana_inicio"], errors="coerce").dt.normalize()
    dfw = dfw[dfw["semana_inicio"].notna()].copy()

    if "semana_fim" not in dfw.columns:
        dfw["semana_fim"] = dfw["semana_inicio"] + pd.Timedelta(days=6)
    else:
        dfw["semana_fim"] = pd.to_datetime(dfw["semana_fim"], errors="coerce").dt.normalize()

    if "nome" not in dfw.columns:
        dfw["nome"] = "Sem Nome"
    dfw["nome"] = dfw["nome"].fillna("Sem Nome").astype(str).str.strip()

    if "area" not in dfw.columns:
        dfw["area"] = "Sem Área"
    dfw["area"] = dfw["area"].fillna("Sem Área").astype(str).str.strip()

    if "dias_com_registro" not in dfw.columns:
        dfw["dias_com_registro"] = np.nan

    if "dias_presenciais" not in dfw.columns:
        dfw["dias_presenciais"] = 0
    if "horas_presenciais" not in dfw.columns:
        dfw["horas_presenciais"] = 0.0

    dfw["dias_presenciais"] = pd.to_numeric(dfw["dias_presenciais"], errors="coerce").fillna(0).astype(int)
    dfw["horas_presenciais"] = pd.to_numeric(dfw["horas_presenciais"], errors="coerce").fillna(0.0)

    if "cadastro" in dfw.columns:
        dfw["cadastro"] = dfw["cadastro"].astype(str).str.strip()
    else:
        raise ValueError("df_semanal precisa ter a coluna 'cadastro'.")

    if "nao_cumpriu_3" not in dfw.columns:
        dfw["nao_cumpriu_3"] = (dfw["dias_presenciais"] < 3)
    else:
        dfw["nao_cumpriu_3"] = dfw["nao_cumpriu_3"].fillna(False).astype(bool)

    # -----------------------------
    # Base analítica (pessoa-semana)
    # -----------------------------
    pessoa_semanas_total = int(len(dfw))
    pessoas_unicas = int(dfw["cadastro"].nunique())
    areas_unicas = int(dfw["area"].nunique())
    semanas_unicas = int(dfw["semana_inicio"].nunique())

    dt_min = dfw["semana_inicio"].min()
    dt_max = dfw["semana_fim"].max()

    area_agg = (
        dfw.groupby("area", as_index=False)
           .agg(
               pessoa_semanas=("cadastro", "size"),
               pessoas_unicas=("cadastro", "nunique"),
               media_dias_presenciais=("dias_presenciais", "mean"),
               media_horas_presenciais=("horas_presenciais", "mean"),
           )
    )
    area_agg["score_risco"] = area_agg["pessoa_semanas"] * (3 - area_agg["media_dias_presenciais"].clip(lower=0, upper=3))

    semana_agg = (
        dfw.groupby("semana_inicio", as_index=False)
           .agg(
               pessoa_semanas=("cadastro", "size"),
               pessoas_unicas=("cadastro", "nunique"),
               media_dias_presenciais=("dias_presenciais", "mean"),
               media_horas_presenciais=("horas_presenciais", "mean"),
           )
           .sort_values("semana_inicio")
    )

    pessoa_stats = (
        dfw.groupby(["cadastro", "nome", "area"], as_index=False)
           .agg(
               semanas_irregulares=("semana_inicio", "nunique"),
               dias_presenciais_total=("dias_presenciais", "sum"),
               horas_presenciais_total=("horas_presenciais", "sum"),
               media_dias_presenciais=("dias_presenciais", "mean"),
               media_horas_presenciais=("horas_presenciais", "mean"),
           )
    )

    def make_first_name_labels(series_names: pd.Series) -> pd.Series:
        first = (
            series_names.astype(str)
            .str.strip()
            .str.split()
            .str[0]
            .replace({"": "Semnome"})
        )
        counts = {}
        labels = []
        for fn in first.tolist():
            counts[fn] = counts.get(fn, 0) + 1
            labels.append(fn if counts[fn] == 1 else f"{fn} ({counts[fn]})")
        return pd.Series(labels, index=series_names.index)

    # -----------------------------
    # Estilo
    # -----------------------------
    VW_NAVY   = "#001E50"
    VW_BLUE   = "#0B5FA5"
    VW_BLUE2  = "#7FB0D6"
    VW_GREY   = "#C9D1D9"
    VW_DARK   = "#1B2A41"
    ACCENT    = "#FFB000"
    BG_LIGHT  = "#F6F8FB"

    plt.rcParams.update({
        "figure.facecolor": BG_LIGHT,
        "axes.facecolor": "white",
        "axes.edgecolor": VW_GREY,
        "axes.labelcolor": VW_DARK,
        "xtick.color": VW_DARK,
        "ytick.color": VW_DARK,
        "text.color": VW_DARK,
        "font.size": 10,
    })

    VW_CMAP = LinearSegmentedColormap.from_list(
        "vw_blues",
        [BG_LIGHT, "#CFE2F3", VW_BLUE, VW_NAVY]
    )

    def style_card(ax):
        for s in ax.spines.values():
            s.set_visible(True)
            s.set_color(VW_GREY)
            s.set_linewidth(1.2)
        ax.set_facecolor("white")

    def add_desc(ax, text):
        ax.text(
            0.01, -0.34, text,
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=8.8, color="#334155",
            wrap=True,
            bbox=dict(
                boxstyle="round,pad=0.40",
                facecolor=BG_LIGHT,
                edgecolor=VW_GREY,
                linewidth=0.9
            )
        )

    DESC = {
        "A": "Combina volume de pessoa-semana com severidade média (menos dias presenciais = maior severidade).",
        "B": "Mostra onde está a maior massa de semanas irregulares. É o melhor corte para priorização rápida.",
        "C": "Ajuda a ver se o problema foi pontual ou se repetiu entre as semanas/áreas analisadas.",
        "D": "Compara o volume de semanas irregulares com a média de dias presenciais da semana.",
        "E": "Mostra quão perto do limite mínimo de 3 dias estão os casos. Quanto mais perto de 0 dia, mais severo.",
        "F": "Evidencia quem reaparece em mais de uma semana irregular.",
        "G": "Mostra as áreas com maior base afetada em pessoas únicas, não só em volume bruto.",
        "H": "Destaca os recorrentes com menor média de dias presenciais nas semanas irregulares.",
    }

    with PdfPages(pdf_path) as pdf:
        # =========================================
        # CAPA
        # =========================================
        cad_nome_map = (
            pessoa_stats.drop_duplicates(subset=["cadastro"], keep="first")
                       .set_index("cadastro")["nome"]
            if (not pessoa_stats.empty) and ("cadastro" in pessoa_stats.columns)
            else pd.Series(dtype=str)
        )
        labels_pessoa = dfw["cadastro"].astype(str).map(
            lambda c: f"{c} - {cad_nome_map.get(c, '')}".strip(" -")
        )
        top_pessoas_semana = labels_pessoa.value_counts()
        top1_n = int(top_pessoas_semana.head(1).sum()) if not top_pessoas_semana.empty else 0
        top3_n = int(top_pessoas_semana.head(3).sum()) if not top_pessoas_semana.empty else 0
        top10_n = int(top_pessoas_semana.head(10).sum()) if not top_pessoas_semana.empty else 0

        top1_pct = (top1_n / pessoa_semanas_total) if pessoa_semanas_total else 0.0
        top3_pct = (top3_n / pessoa_semanas_total) if pessoa_semanas_total else 0.0
        top10_pct = (top10_n / pessoa_semanas_total) if pessoa_semanas_total else 0.0

        top3_piores_capa = []
        for k, v in top_pessoas_semana.head(3).items():
            pct = (int(v) / pessoa_semanas_total) if pessoa_semanas_total else 0.0
            top3_piores_capa.append(f"{str(k)} — {int(v)} ({pct:.1%})")

        dt_ref = pd.to_datetime(dfw["semana_inicio"], errors="coerce").dropna()
        dt_ref_max = dt_ref.max().normalize() if not dt_ref.empty else pd.NaT
        if pd.notna(dt_ref_max):
            dt_cut = dt_ref_max - pd.Timedelta(days=29)
            recent = dfw.loc[pd.to_datetime(dfw["semana_inicio"], errors="coerce") >= dt_cut]
        else:
            recent = dfw.head(0)

        total_recent = int(len(recent))
        if total_recent > 0:
            labels_recent = recent["cadastro"].astype(str).map(
                lambda c: f"{c} - {cad_nome_map.get(c, '')}".strip(" -")
            )
            top_recent = labels_recent.value_counts().head(3)
        else:
            top_recent = pd.Series(dtype=int)

        top3_atuais_capa = []
        for k, v in top_recent.items():
            pct = (int(v) / total_recent) if total_recent else 0.0
            top3_atuais_capa.append(f"{str(k)} — {int(v)} ({pct:.1%} da janela)")

        janela_info = (
            f"{total_recent:,} ({(total_recent / pessoa_semanas_total):.1%} do total)".replace(",", ".")
            if pessoa_semanas_total > 0 else "0 (0.0% do total)"
        )

        metricas_capa = [
            ("Pessoa-semanas irregulares", f"{pessoa_semanas_total:,}".replace(",", ".")),
            ("Pessoas únicas", f"{pessoas_unicas:,}".replace(",", ".")),
            ("Semanas com ocorrência", f"{semanas_unicas:,}".replace(",", ".")),
        ]

        periodo_capa = None
        if pd.notna(dt_min) and pd.notna(dt_max):
            periodo_capa = f"{pd.to_datetime(dt_min).date()} → {pd.to_datetime(dt_max).date()}"

        concentracoes_capa = [
            ("Concentração Top 1", f"{top1_n:,} ({top1_pct:.1%})".replace(",", ".")),
            ("Concentração Top 3", f"{top3_n:,} ({top3_pct:.1%})".replace(",", ".")),
            ("Concentração Top 10", f"{top10_n:,} ({top10_pct:.1%})".replace(",", ".")),
        ]

        extras_capa = [
            f"Áreas com ocorrência: {areas_unicas:,}".replace(",", "."),
            "Critério: menos de 3 dias presenciais na semana de feriado prolongado",
        ]

        _render_capa_padrao_pdf(
            pdf=pdf,
            titulo=titulo,
            gen_date=gen_date,
            metricas=metricas_capa,
            periodo=periodo_capa,
            concentracoes=concentracoes_capa,
            top3_piores=top3_piores_capa,
            top3_atuais=top3_atuais_capa,
            janela_info=janela_info,
            extras=extras_capa,
        )

        # =========================================
        # DASHBOARD 4x2
        # =========================================
        fig = plt.figure(figsize=(18, 24))
        gs = fig.add_gridspec(4, 2, hspace=0.92, wspace=0.28)

        # (A) Prioridade — Volume x Severidade por área
        axA = fig.add_subplot(gs[0, 0]); style_card(axA)

        dfA = area_agg.copy()
        topA = dfA.sort_values("score_risco", ascending=False).head(8).copy()

        axA.scatter(
            dfA["pessoa_semanas"],
            dfA["media_dias_presenciais"],
            s=(dfA["pessoas_unicas"] * 20).clip(50, 1000),
            alpha=0.65, color=VW_BLUE2, edgecolors=VW_NAVY, linewidths=0.7
        )
        axA.scatter(
            topA["pessoa_semanas"],
            topA["media_dias_presenciais"],
            s=(topA["pessoas_unicas"] * 24).clip(70, 1100),
            alpha=0.95, color=ACCENT, edgecolors=VW_NAVY, linewidths=0.9
        )

        for _, r in topA.iterrows():
            axA.annotate(
                str(r["area"]),
                xy=(r["pessoa_semanas"], r["media_dias_presenciais"]),
                xytext=(12, 10),
                textcoords="offset points",
                ha="left", va="bottom", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor=VW_GREY, linewidth=0.6)
            )

        axA.set_title("A) Prioridade — Volume × severidade por Área", pad=10, color=VW_NAVY, fontweight="bold")
        axA.set_xlabel("Volume (pessoa-semanas)")
        axA.set_ylabel("Média de dias presenciais")
        axA.grid(True, linestyle="--", alpha=0.18)
        axA.legend(handles=[
            Line2D([0], [0], marker='o', color='w', label='Áreas (geral)',
                   markerfacecolor=VW_BLUE2, markeredgecolor=VW_NAVY, markersize=9),
            Line2D([0], [0], marker='o', color='w', label='Top 8 prioridade',
                   markerfacecolor=ACCENT, markeredgecolor=VW_NAVY, markersize=9),
        ], loc="upper right", frameon=True, facecolor="white", edgecolor=VW_GREY)
        add_desc(axA, DESC["A"])

        # (B) Pareto — pessoa-semanas por área
        axB = fig.add_subplot(gs[0, 1]); style_card(axB)
        dfB = area_agg.sort_values("pessoa_semanas", ascending=False).head(12).copy()

        axB.bar(dfB["area"].astype(str), dfB["pessoa_semanas"], color=VW_NAVY, alpha=0.92)
        axB.set_title("B) Pareto — Massa de semanas irregulares por Área", pad=10, color=VW_NAVY, fontweight="bold")
        axB.set_xlabel("Área")
        axB.set_ylabel("Qtd de pessoa-semanas")
        axB.grid(axis="y", linestyle="--", alpha=0.18)
        axB.tick_params(axis="x", rotation=35)

        max_bar = float(dfB["pessoa_semanas"].max()) if not dfB.empty else 1.0
        axB.set_ylim(0, max_bar * 1.30 if max_bar > 0 else 1)
        for i, r in enumerate(dfB.itertuples(index=False)):
            axB.text(
                i,
                r.pessoa_semanas + max_bar * 0.03,
                f"{int(r.pessoa_semanas)}\n({float(r.media_dias_presenciais):.2f} dias)",
                ha="center", va="bottom", fontsize=8
            )
        add_desc(axB, DESC["B"])

        # (C) Heatmap — área x semana
        axC = fig.add_subplot(gs[1, 0]); style_card(axC)
        top_areas_heat = area_agg.sort_values("pessoa_semanas", ascending=False).head(10)["area"].astype(str).tolist()

        aw = (
            dfw[dfw["area"].astype(str).isin(top_areas_heat)]
            .groupby(["area", "semana_inicio"], as_index=False)
            .size()
            .rename(columns={"size": "qtd"})
        )

        if aw.empty:
            axC.axis("off")
            axC.text(0.02, 0.95, "C) Estabilidade — Área × semana", fontsize=14, fontweight="bold", va="top")
            axC.text(0.02, 0.82, "Sem dados suficientes.", fontsize=11, va="top")
        else:
            pivot = aw.pivot_table(index="area", columns="semana_inicio", values="qtd", aggfunc="sum").fillna(0)
            pivot = pivot.reindex(top_areas_heat)
            pivot = pivot.reindex(sorted(pivot.columns), axis=1)
            mat = pivot.values

            im = axC.imshow(mat, aspect="auto", cmap=VW_CMAP)
            axC.set_title("C) Estabilidade — Pessoa-semanas por Área × semana", pad=10, color=VW_NAVY, fontweight="bold")
            axC.set_xlabel("Semana (início)")
            axC.set_ylabel("Área")

            xlabels = [pd.to_datetime(c).strftime("%Y-%m-%d") for c in pivot.columns]
            axC.set_xticks(np.arange(len(xlabels)))
            axC.set_xticklabels(xlabels, rotation=45, ha="right")
            axC.set_yticks(np.arange(pivot.shape[0]))
            axC.set_yticklabels(pivot.index.astype(str))

            for i in range(mat.shape[0]):
                for j in range(mat.shape[1]):
                    v = mat[i, j]
                    axC.text(j, i, f"{int(v)}", ha="center", va="center",
                             fontsize=8, color=("white" if v >= np.nanmax(mat) * 0.60 and np.nanmax(mat) > 0 else VW_DARK))

            cbar = fig.colorbar(im, ax=axC, fraction=0.046, pad=0.04)
            cbar.set_label("Qtd pessoa-semana")

        add_desc(axC, DESC["C"])

        # (D) Tendência semanal
        axD = fig.add_subplot(gs[1, 1]); style_card(axD)
        dfD = semana_agg.copy()
        dfD["semana"] = dfD["semana_inicio"].dt.strftime("%Y-%m-%d")

        axD.plot(dfD["semana"], dfD["pessoa_semanas"], marker="o", color=VW_NAVY, linewidth=2.4, label="Pessoa-semanas")
        axD.set_title("D) Evolução semanal — Volume vs média de dias", pad=10, color=VW_NAVY, fontweight="bold")
        axD.set_xlabel("Semana (início)")
        axD.set_ylabel("Qtd pessoa-semana")
        axD.grid(True, linestyle="--", alpha=0.18)
        axD.tick_params(axis="x", rotation=45)

        axD2 = axD.twinx()
        axD2.plot(dfD["semana"], dfD["media_dias_presenciais"], marker="o", color=VW_BLUE, linewidth=2.2, label="Média dias")
        axD2.set_ylabel("Média dias presenciais")

        lines1, labels1 = axD.get_legend_handles_labels()
        lines2, labels2 = axD2.get_legend_handles_labels()
        axD.legend(lines1 + lines2, labels1 + labels2, loc="upper right", frameon=True, facecolor="white", edgecolor=VW_GREY)

        for x, y in zip(dfD["semana"], dfD["pessoa_semanas"]):
            axD.text(x, y + max(dfD["pessoa_semanas"].max() * 0.03, 0.2), f"{int(y)}", ha="center", fontsize=8)

        add_desc(axD, DESC["D"])

        # (E) Severidade — faixas de horas
        axE = fig.add_subplot(gs[2, 0]); style_card(axE)

        bins = [-0.001, 1.0, 2.0, 3.0001]
        labels = ["0 dia", "1 dia", "2 dias"]
        dfw["faixa_dias"] = pd.cut(dfw["dias_presenciais"], bins=bins, labels=labels, include_lowest=True, right=False)

        dist = (
            dfw.groupby("faixa_dias", dropna=False)
               .size()
               .rename("qtd")
               .reset_index()
        )
        dist["faixa_dias"] = dist["faixa_dias"].astype(str)
        dist = dist[dist["faixa_dias"].isin(labels)].copy()
        dist = dist.set_index("faixa_dias").reindex(labels).fillna(0).reset_index()
        dist["pct"] = dist["qtd"] / dist["qtd"].sum() * 100 if dist["qtd"].sum() > 0 else 0.0

        axE.bar(dist["faixa_dias"], dist["pct"], color=VW_BLUE, alpha=0.92)
        axE.set_title("E) Severidade — Distribuição por faixa de dias", pad=10, color=VW_NAVY, fontweight="bold")
        axE.set_xlabel("Dias presenciais na semana")
        axE.set_ylabel("% de pessoa-semanas")
        axE.grid(axis="y", linestyle="--", alpha=0.18)

        ymaxE = float(dist["pct"].max()) * 1.25 if not dist.empty else 1.0
        axE.set_ylim(0, ymaxE if ymaxE > 0 else 1)
        for i, r in dist.iterrows():
            axE.text(i, float(r["pct"]) + ymaxE * 0.02 if ymaxE > 0 else 0.1, f"{float(r['pct']):.1f}%", ha="center", fontsize=8)

        add_desc(axE, DESC["E"])

        # (F) Recorrência por pessoa
        axF = fig.add_subplot(gs[2, 1]); style_card(axF)

        topF = (
            pessoa_stats.sort_values(["semanas_irregulares", "horas_presenciais_total"], ascending=[False, True])
                       .head(12)
                       .copy()
        )
        topF["primeiro_nome"] = make_first_name_labels(topF["nome"])

        valsF = topF["semanas_irregulares"].to_numpy()
        axF.bar(topF["primeiro_nome"], valsF, color=VW_NAVY, alpha=0.92)
        axF.set_title("F) Investigação — Recorrência por pessoa", pad=10, color=VW_NAVY, fontweight="bold")
        axF.set_xlabel("Primeiro nome")
        axF.set_ylabel("Qtd de semanas irregulares")
        axF.grid(axis="y", linestyle="--", alpha=0.18)
        axF.tick_params(axis="x", rotation=45)
        for lab in axF.get_xticklabels():
            lab.set_ha("right")

        if len(valsF) > 0:
            axF.set_ylim(0, max(valsF.max() * 1.30, valsF.max() + 1))
            for i, r in enumerate(topF.itertuples(index=False)):
                axF.text(i, r.semanas_irregulares + 0.05, f"{str(r.area)}\n{float(r.media_dias_presenciais):.2f} dias",
                         ha="center", va="bottom", fontsize=8)

        add_desc(axF, DESC["F"])

        # (G) Base afetada — pessoas únicas por área
        axG = fig.add_subplot(gs[3, 0]); style_card(axG)

        dfG = area_agg.sort_values(["pessoas_unicas", "pessoa_semanas"], ascending=[False, False]).head(12).copy()
        valsG = dfG["pessoas_unicas"].to_numpy()

        axG.bar(dfG["area"].astype(str), valsG, color=VW_BLUE, alpha=0.92)
        axG.set_title("G) Base afetada — Pessoas únicas por Área", pad=10, color=VW_NAVY, fontweight="bold")
        axG.set_xlabel("Área")
        axG.set_ylabel("Pessoas únicas")
        axG.grid(axis="y", linestyle="--", alpha=0.18)
        axG.tick_params(axis="x", rotation=35)

        if len(valsG) > 0:
            axG.set_ylim(0, max(valsG.max() * 1.25, valsG.max() + 1))
            for i, r in enumerate(dfG.itertuples(index=False)):
                axG.text(i, r.pessoas_unicas + 0.05, f"{int(r.pessoa_semanas)} sem.", ha="center", va="bottom", fontsize=8)

        add_desc(axG, DESC["G"])

        # (H) Menor média de horas — recorrentes
        axH = fig.add_subplot(gs[3, 1]); style_card(axH)

        baseH = pessoa_stats[pessoa_stats["semanas_irregulares"] >= 2].copy()
        topH = (
            baseH.sort_values(["media_dias_presenciais", "semanas_irregulares"], ascending=[True, False])
                 .head(12)
                 .copy()
        )
        topH["primeiro_nome"] = make_first_name_labels(topH["nome"])

        valsH = topH["media_dias_presenciais"].to_numpy()
        axH.bar(topH["primeiro_nome"], valsH, color=VW_NAVY, alpha=0.85)
        axH.set_title("H) Recorrentes — Menor média de dias presenciais", pad=10, color=VW_NAVY, fontweight="bold")
        axH.set_xlabel("Primeiro nome")
        axH.set_ylabel("Média de dias presenciais")
        axH.grid(axis="y", linestyle="--", alpha=0.18)
        axH.tick_params(axis="x", rotation=45)
        for lab in axH.get_xticklabels():
            lab.set_ha("right")

        if len(valsH) > 0:
            axH.set_ylim(0, max(valsH.max() * 1.35, 1))
            for i, r in enumerate(topH.itertuples(index=False)):
                axH.text(i, r.media_dias_presenciais + 0.03, f"{int(r.semanas_irregulares)} sem.\n{str(r.area)}",
                         ha="center", va="bottom", fontsize=8)

        add_desc(axH, DESC["H"])

        fig.suptitle(
            "Painel semanal — Semana de Feriado Prolongado (pessoa-semana)",
            fontsize=20, fontweight="bold", color=VW_NAVY, y=0.99
        )

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # =========================================
        # LISTAS FINAIS
        # =========================================
        fig = plt.figure(figsize=(11.69, 8.27))
        ax = fig.add_subplot(111)
        ax.axis("off")

        top_pessoas = (
            pessoa_stats.sort_values(["semanas_irregulares", "media_dias_presenciais"], ascending=[False, True])
                       .head(15)
                       .loc[:, ["cadastro", "nome", "area", "semanas_irregulares", "media_dias_presenciais"]]
                       .copy()
        )
        top_pessoas["media_dias_presenciais"] = top_pessoas["media_dias_presenciais"].round(2)

        ax.text(0.02, 0.97, "Top 15 pessoas — semanas irregulares", fontsize=15, fontweight="bold", va="top")

        if top_pessoas.empty:
            ax.text(0.02, 0.90, "Sem dados suficientes.", fontsize=11, va="top")
        else:
            tabela = ax.table(
                cellText=top_pessoas.values,
                colLabels=["Cadastro", "Nome", "Área", "Semanas", "Média dias"],
                cellLoc="left",
                colLoc="left",
                bbox=[0.02, 0.08, 0.96, 0.82]
            )
            tabela.auto_set_font_size(False)
            tabela.set_fontsize(9)
            tabela.scale(1, 1.15)

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

def gerar_relatorio_pdf(df_irregularidade: pd.DataFrame, titulo: str, pdf_path: Path) -> None:
    if (
        isinstance(df_irregularidade, pd.DataFrame)
        and not df_irregularidade.empty
        and {"cadastro", "semana_inicio", "dias_presenciais", "horas_presenciais"}.issubset(df_irregularidade.columns)
    ):
        gerar_relatorio_pdf_semanal_feriado(df_irregularidade, titulo=titulo, pdf_path=pdf_path)
        return

    import datetime as _dt
    import numpy as np

    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 150,
        "font.size": 11,
        "axes.titlesize": 16,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    })

    def _wrap(s: str, width: int = 44) -> str:
        s = "" if s is None else str(s)
        s = s.strip()
        if len(s) <= width:
            return s
        words = s.split()
        out, line = [], ""
        for w in words:
            if len(line) + len(w) + 1 <= width:
                line = (line + " " + w).strip()
            else:
                if line:
                    out.append(line)
                line = w
        if line:
            out.append(line)
        if len(out) <= 4:
            return "\n".join(out)
        return "\n".join(out[:4]) + "…"

    def _as_series(df_: pd.DataFrame, col: str) -> pd.Series:
        x = df_.loc[:, col]
        if isinstance(x, pd.DataFrame):
            x = x.iloc[:, 0]
        return x

    def _pick_col_local(df_: pd.DataFrame, candidates: list[str]) -> str | None:
        for c in candidates:
            if c in df_.columns:
                return c
        return None

    def _norm_text(s: str) -> str:
        s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
        return re.sub(r"\s+", " ", s).strip().lower()

    def _match_any(text_norm: str, patterns_norm: list[str]) -> bool:
        return any(p in text_norm for p in patterns_norm)

    def _annotate_barh(ax, total_base: int | None = None, fs: int = 9):
        xmax = ax.get_xlim()[1] if ax.get_xlim()[1] > 0 else 1.0
        pad = xmax * 0.012
        for p in ax.patches:
            v = float(p.get_width())
            if v <= 0:
                continue
            y = p.get_y() + p.get_height() / 2
            label = f"{int(v)} ({(v / total_base):.1%})" if (total_base and total_base > 0) else f"{int(v)}"
            x = min(v + pad, xmax * 0.985)
            ha = "left" if x < xmax * 0.96 else "right"
            ax.text(x, y, label, va="center", ha=ha, fontsize=fs)

    def _annotate_bar(ax, total_base: int | None = None, fs: int = 9):
        ymax = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1.0
        pad = ymax * 0.015
        for p in ax.patches:
            v = float(p.get_height())
            if v <= 0:
                continue
            x = p.get_x() + p.get_width() / 2
            label = f"{int(v)}\n({(v / total_base):.1%})" if (total_base and total_base > 0) else f"{int(v)}"
            ax.text(x, v + pad, label, ha="center", va="bottom", fontsize=fs)

    def _pie_with_legend(ax, values, legend_labels, title, base_text):
        values = [int(v) for v in values]
        total_vals = sum(values)
        ax.set_title(title, pad=16, fontsize=12)
        if total_vals <= 0:
            ax.axis("off")
            ax.text(0.02, 0.90, "Sem dados suficientes.", fontsize=10.5, va="top")
            return

        wedges, _, autotexts = ax.pie(
            values,
            labels=None,
            autopct=lambda p: f"{p:.0f}%" if p > 0 else "",
            startangle=90,
            pctdistance=0.72,
            radius=0.88,
            wedgeprops={"linewidth": 0.8, "edgecolor": "white"},
            textprops={"fontsize": 9},
        )
        for t in autotexts:
            t.set_fontsize(9)

        ax.legend(
            wedges,
            legend_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.98),
            ncol=min(2, len(legend_labels)),
            fontsize=8.5,
            frameon=False,
            borderaxespad=0.0,
        )
        ax.text(0.5, 0.04, base_text, ha="center", va="bottom", transform=ax.transAxes, fontsize=8.8)
        ax.set_aspect("equal")

    def _find_dim_colab_source(required_cols: list[str]) -> pd.DataFrame | None:
        candidates = [
            "dim_colaborador", "df_dim_colaborador", "df_colaboradores", "df_colaborador",
            "vw_colaboradores", "df_colaborador_hr", "vw_colaborador_hr", "df_colaboradorRH"
        ]
        for name in candidates:
            obj = globals().get(name)
            if isinstance(obj, pd.DataFrame) and all(c in obj.columns for c in required_cols):
                return obj
        return None

    def _get_dim_map(area_col_use: str | None, col_heranca: str | None, col_cargo: str | None = None) -> pd.DataFrame | None:
        if (area_col_use is None) or (col_heranca is None):
            return None

        cache_key = ("_DIM_MAP_CACHE", area_col_use, col_heranca, col_cargo or "")
        if cache_key in globals():
            cached = globals().get(cache_key)
            if isinstance(cached, pd.DataFrame) and not cached.empty and "cadastro" in cached.columns:
                return cached

        required = ["cadastro", area_col_use, col_heranca]
        required_cargo = required + ([col_cargo] if col_cargo else [])

        src = _find_dim_colab_source(required_cargo)
        if src is not None:
            dim = src.loc[:, required_cargo].copy(deep=False).drop_duplicates(subset=["cadastro"], keep="last")
            dim["cadastro"] = dim["cadastro"].astype(str).str.strip()
            dim["_area_norm"] = dim[area_col_use].fillna("").astype(str).str.strip()
            dim["_her_norm"] = dim[col_heranca].fillna("").astype(str).str.strip()
            if col_cargo and (col_cargo in dim.columns):
                dim["_cargo_norm"] = dim[col_cargo].fillna("").astype(str).str.strip().str.upper()
            globals()[cache_key] = dim[["cadastro", "_area_norm", "_her_norm"] + (["_cargo_norm"] if "_cargo_norm" in dim.columns else [])]
            return globals()[cache_key]

        base_all = globals().get("df_min_raw")
        if not isinstance(base_all, pd.DataFrame):
            return None
        if not all(c in base_all.columns for c in required_cargo):
            return None

        seen: set[str] = set()
        parts = []
        n = len(base_all)
        cols = required_cargo

        for i in range(0, n, BATCH_SIZE):
            chunk = base_all.iloc[i:i + BATCH_SIZE][cols].copy(deep=False)
            if chunk.empty:
                continue
            chunk = chunk.drop_duplicates(subset=["cadastro"], keep="last")
            cad_str = chunk["cadastro"].astype(str).str.strip()
            new_mask = ~cad_str.isin(seen)
            if not new_mask.any():
                continue
            chunk = chunk.loc[new_mask]
            cad_new = cad_str.loc[new_mask]
            seen.update(cad_new.tolist())
            chunk = chunk.assign(cadastro=cad_new.to_numpy(copy=False))
            parts.append(chunk)
            if len(seen) >= 400_000:
                break

        if not parts:
            return None

        dim = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["cadastro"], keep="last")
        dim["_area_norm"] = dim[area_col_use].fillna("").astype(str).str.strip()
        dim["_her_norm"] = dim[col_heranca].fillna("").astype(str).str.strip()
        if col_cargo and (col_cargo in dim.columns):
            dim["_cargo_norm"] = dim[col_cargo].fillna("").astype(str).str.strip().str.upper()

        globals()[cache_key] = dim[["cadastro", "_area_norm", "_her_norm"] + (["_cargo_norm"] if "_cargo_norm" in dim.columns else [])]
        return globals()[cache_key]

    def _build_dt_from_data_hora(df_: pd.DataFrame) -> pd.Series:
        if "_dt_ronda" in df_.columns:
            return pd.to_datetime(df_["_dt_ronda"], errors="coerce")
        if ("data" in df_.columns) and ("hora" in df_.columns):
            data_dt = pd.to_datetime(df_["data"], dayfirst=True, errors="coerce")
            hora_s = df_["hora"].fillna("00:00").astype(str).str.strip()
            hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
            hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
            hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
            hora_td = pd.to_timedelta(hora_s, errors="coerce")
            return data_dt.dt.normalize() + hora_td
        return pd.Series(pd.NaT, index=df_.index)

    gen_date = _dt.date.today().strftime("%Y-%m-%d")

    if df_irregularidade is None or df_irregularidade.empty:
        with PdfPages(pdf_path) as pdf:
            _render_capa_padrao_pdf(
                pdf=pdf,
                titulo=titulo,
                gen_date=gen_date,
                metricas=[
                    ("Ocorrências (pessoa-dia)", "0"),
                    ("Pessoas (cadastro)", "0"),
                    ("Dias com ocorrência", "0"),
                ],
                periodo=None,
                concentracoes=[
                    ("Concentração Top 1", "0 (0.0%)"),
                    ("Concentração Top 3", "0 (0.0%)"),
                    ("Concentração Top 10", "0 (0.0%)"),
                ],
                top3_piores=[],
                top3_atuais=[],
                janela_info="0 (0.0% do total)",
                extras=["Sem linhas para gerar o relatório."],
            )
        return

    if getattr(df_irregularidade, "columns", None) is not None:
        df_irregularidade = df_irregularidade.loc[:, ~df_irregularidade.columns.duplicated()]

    titulo_norm = _norm_text(titulo)
    path_norm = _norm_text(str(pdf_path))

    excluir_planta_999_nesta_irregularidade = _deve_excluir_planta_999(
        titulo=titulo,
        pdf_path=pdf_path,
    )

    sem_corte_30 = _match_any(
        titulo_norm + " " + path_norm,
        [
            "semana de feriado prolongado",
            "feriado prolongado",
            "3 dias presenciais na semana",
            "irregularidade_pessoas_sem_cumprir_os_3_dias_presenciais_em_semana_de_feriado_prolongado",
            "pessoas_sem_cumprir_os_3_dias_presenciais_em_semana_de_feriado_prolongado",
            "executivos com historico de 999 como trajeto",
            "irregularidade_ausencia_das_horas_presenciais_minimas",
            "ausencia das horas presenciais minimas",
            "irregularidade_nao_entrou_no_refeitorio_mas_passou_no_grill",
            "nao entrou no refeitorio mas passou no grill",
        ],
    )
    somente_executivos_atuais = ("executivos com historico de 999 como trajeto" in titulo_norm)

    excluir_exec_nessas = _match_any(
        titulo_norm + " " + path_norm,
        [
            "irregularidade_registro_sem_portaria",
            "registro sem portaria",
            "irregularidade_ponto_sem_portaria",
            "ponto sem portaria",
            "irregularidade_pessoas_trabalhando_sem_ganhar",
            "pessoas trabalhando sem ganhar",
            "irregularidade_pessoas_trabalhando_alem_da_jornada_de_trabalho",
            "pessoas trabalhando alem da jornada de trabalho",
        ],
    )
    analisar_proxy_ponto = _match_any(
        titulo_norm + " " + path_norm,
        ["irregularidade_ponto_sem_portaria", "ponto sem portaria"],
    )
    mostrar_periodo_desligado = _match_any(
        titulo_norm + " " + path_norm,
        [
            "irregularidade_registros_de_pessoas_desligadas",
            "registros de pessoas desligadas",
        ],
    )

    col_cad = _pick_col_local(df_irregularidade, ["cadastro"])
    col_nome = _pick_col_local(df_irregularidade, ["nome"])
    col_areas = "área" if "área" in df_irregularidade.columns else None
    col_area_fallback = _pick_col_local(df_irregularidade, ["area"])
    col_planta = "planta" if "planta" in df_irregularidade.columns else None
    col_heranca = "herança" if "herança" in df_irregularidade.columns else ("heranca" if "heranca" in df_irregularidade.columns else None)
    col_cargo = _pick_col_local(df_irregularidade, ["cargo", "cargo_atual"])
    col_data_dem = _pick_col_local(df_irregularidade, ["Data_Dem", "data_desligado", "DataDemissao", "data_demissao"])

    if "_dia_operacional" in df_irregularidade.columns:
        dia_op = pd.to_datetime(_as_series(df_irregularidade, "_dia_operacional"), errors="coerce")
    elif "dia_operacional" in df_irregularidade.columns:
        dia_op = pd.to_datetime(_as_series(df_irregularidade, "dia_operacional"), errors="coerce")
    else:
        data_dt = (
            pd.to_datetime(_as_series(df_irregularidade, "data"), dayfirst=True, errors="coerce")
            if "data" in df_irregularidade.columns
            else pd.Series(pd.NaT, index=df_irregularidade.index)
        )
        if "hora" not in df_irregularidade.columns:
            dia_op = data_dt.dt.normalize()
        else:
            hora_s = _as_series(df_irregularidade, "hora").fillna("00:00").astype(str).str.strip()
            hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
            hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
            hora_td = pd.to_timedelta(hora_s, errors="coerce")
            dia_op = (data_dt.dt.normalize() + hora_td - pd.Timedelta(hours=11)).dt.normalize()

    cols_keep = ["_dia_operacional"]
    if col_cad: cols_keep.append(col_cad)
    if col_nome: cols_keep.append(col_nome)
    if col_areas: cols_keep.append(col_areas)
    elif col_area_fallback: cols_keep.append(col_area_fallback)
    if col_planta: cols_keep.append(col_planta)
    if col_heranca: cols_keep.append(col_heranca)
    if col_cargo: cols_keep.append(col_cargo)
    if col_data_dem: cols_keep.append(col_data_dem)

    dfv = df_irregularidade.loc[:, [c for c in cols_keep if c != "_dia_operacional"]].copy(deep=False)
    dfv["_dia_operacional"] = pd.to_datetime(dia_op, errors="coerce")

    if excluir_planta_999_nesta_irregularidade and col_planta and (col_planta in dfv.columns):
        dfv = dfv.loc[_mask_planta_not_999(dfv[col_planta])].copy()

    if col_cad and (col_cad in dfv.columns):
        dfv[col_cad] = dfv[col_cad].astype(str).str.strip()
        dfv = dfv[dfv[col_cad].ne("")]

    if somente_executivos_atuais and col_cargo and (col_cargo in dfv.columns):
        cargo_norm = dfv[col_cargo].fillna("").astype(str).str.strip().str.upper()
        dfv = dfv.loc[cargo_norm.eq("E")]

    if excluir_exec_nessas and col_cargo and (col_cargo in dfv.columns):
        cargo_norm2 = dfv[col_cargo].fillna("").astype(str).str.strip().str.upper()
        dfv = dfv.loc[~cargo_norm2.eq("E")]

    key_cols = [col_cad, "_dia_operacional"] if (col_cad and col_cad in dfv.columns) else ["_dia_operacional"]
    df_occ = (
        dfv.dropna(subset=["_dia_operacional"])
           .drop_duplicates(subset=key_cols, keep="first")
    )

    if df_occ.empty:
        with PdfPages(pdf_path) as pdf:
            _render_capa_padrao_pdf(
                pdf=pdf,
                titulo=titulo,
                gen_date=gen_date,
                metricas=[
                    ("Ocorrências (pessoa-dia)", "0"),
                    ("Pessoas (cadastro)", "0"),
                    ("Dias com ocorrência", "0"),
                ],
                periodo=None,
                concentracoes=[
                    ("Concentração Top 1", "0 (0.0%)"),
                    ("Concentração Top 3", "0 (0.0%)"),
                    ("Concentração Top 10", "0 (0.0%)"),
                ],
                top3_piores=[],
                top3_atuais=[],
                janela_info="0 (0.0% do total)",
                extras=["Sem ocorrências após normalização de dia operacional."],
            )
        return

    MIN_OCORRENCIAS_PESSOA_DIA = 30
    if (not sem_corte_30) and col_cad and (col_cad in df_occ.columns):
        _cnt_pd = df_occ[col_cad].astype(str).str.strip().value_counts()
        _elig = _cnt_pd[_cnt_pd >= MIN_OCORRENCIAS_PESSOA_DIA].index
        if _elig.empty:
            with PdfPages(pdf_path) as pdf:
                _render_capa_padrao_pdf(
                    pdf=pdf,
                    titulo=titulo,
                    gen_date=gen_date,
                    metricas=[
                        ("Ocorrências (pessoa-dia)", "0"),
                        ("Pessoas (cadastro)", "0"),
                        ("Dias com ocorrência", "0"),
                    ],
                    periodo=None,
                    concentracoes=[
                        ("Concentração Top 1", "0 (0.0%)"),
                        ("Concentração Top 3", "0 (0.0%)"),
                        ("Concentração Top 10", "0 (0.0%)"),
                    ],
                    top3_piores=[],
                    top3_atuais=[],
                    janela_info="0 (0.0% do total)",
                    extras=["Sem dados suficientes (mínimo 30 ocorrências por pessoa-dia)."],
                )
            return
        df_occ = df_occ.loc[df_occ[col_cad].astype(str).str.strip().isin(_elig)]
        if (col_cad in dfv.columns) and (not dfv.empty):
            dfv = dfv.loc[dfv[col_cad].astype(str).str.strip().isin(_elig)]

    total = int(len(df_occ))
    n_pessoas = int(df_occ[col_cad].nunique()) if (col_cad and col_cad in df_occ.columns) else 0
    n_dias = int(df_occ["_dia_operacional"].nunique())
    dt_min = df_occ["_dia_operacional"].min()
    dt_max = df_occ["_dia_operacional"].max()

    nome_map = {}
    if col_cad and col_nome and (col_cad in dfv.columns) and (col_nome in dfv.columns):
        tmp = dfv[[col_cad, col_nome]].copy()
        tmp[col_nome] = tmp[col_nome].fillna("").astype(str).str.strip()
        tmp = tmp[tmp[col_nome].ne("")]
        if not tmp.empty:
            cnt = (
                tmp.groupby([col_cad, col_nome], sort=False)
                   .size()
                   .reset_index(name="n")
                   .sort_values([col_cad, "n"], ascending=[True, False], kind="mergesort")
            )
            nome_map = cnt.drop_duplicates(subset=[col_cad], keep="first").set_index(col_cad)[col_nome].to_dict()

    periodo_desligado_map: dict[str, str] = {}
    if mostrar_periodo_desligado and col_data_dem and col_cad and (col_data_dem in dfv.columns) and (col_cad in dfv.columns):
        dem_aux = dfv[[col_cad, "_dia_operacional", col_data_dem]].copy()
        dem_aux["_dia_operacional"] = pd.to_datetime(dem_aux["_dia_operacional"], errors="coerce").dt.normalize().astype("datetime64[ns]")
        dem_aux[col_data_dem] = pd.to_datetime(dem_aux[col_data_dem], dayfirst=True, errors="coerce").dt.normalize().astype("datetime64[ns]")
        dem_aux = dem_aux.dropna(subset=[col_data_dem, "_dia_operacional"])
        if not dem_aux.empty:
            dem_aux[col_cad] = dem_aux[col_cad].astype(str).str.strip()
            dem_aux = dem_aux.loc[dem_aux[col_cad].ne("")]
            dem_aux = dem_aux.loc[dem_aux["_dia_operacional"] >= dem_aux[col_data_dem]]
            if not dem_aux.empty:
                agg_dem = (
                    dem_aux.groupby(col_cad)
                           .agg(dt_ini=(col_data_dem, "min"), dt_fim=("_dia_operacional", "max"))
                           .reset_index()
                )
                for r in agg_dem.itertuples(index=False):
                    cadk = str(getattr(r, col_cad)).strip()
                    if cadk:
                        dt_ini = pd.to_datetime(r.dt_ini, errors="coerce")
                        dt_fim = pd.to_datetime(r.dt_fim, errors="coerce")
                        if pd.notna(dt_ini) and pd.notna(dt_fim):
                            periodo_desligado_map[cadk] = f"{dt_ini.date()} → {dt_fim.date()}"
    if col_cad and (col_cad in df_occ.columns):
        cad_series = df_occ[col_cad].astype(str).str.strip()
        cad_counts = cad_series.value_counts()
        eligible_cads = cad_counts.index if sem_corte_30 else cad_counts[cad_counts >= MIN_OCORRENCIAS_PESSOA_DIA].index

        if len(eligible_cads) == 0:
            top_pessoas = pd.Series(dtype=int)
            top_pessoas_cad_counts = pd.Series(dtype=int)
        else:
            cad_series_elig = cad_series[cad_series.isin(eligible_cads)]
            pessoa_label = cad_series_elig.map(lambda c: f"{c} - {nome_map.get(c, '')}".strip(" -")) if nome_map else cad_series_elig
            top_pessoas = pessoa_label.value_counts()
            top_pessoas_cad_counts = cad_counts.loc[eligible_cads]
    else:
        top_pessoas = pd.Series(dtype=int)
        top_pessoas_cad_counts = pd.Series(dtype=int)

    top1_n = int(top_pessoas.head(1).sum()) if (not top_pessoas.empty) else 0
    top3_n = int(top_pessoas.head(3).sum()) if (not top_pessoas.empty) else 0
    top10_n = int(top_pessoas.head(10).sum()) if (not top_pessoas.empty) else 0
    top1_pct = (top1_n / total) if total else 0.0
    top3_pct = (top3_n / total) if total else 0.0
    top10_pct = (top10_n / total) if total else 0.0

    mes = df_occ["_dia_operacional"].dt.to_period("M").dt.to_timestamp()
    mensal = mes.value_counts().sort_index()

    dow_map = {0: "Seg", 1: "Ter", 2: "Qua", 3: "Qui", 4: "Sex", 5: "Sáb", 6: "Dom"}
    dow = (
        df_occ["_dia_operacional"].dt.dayofweek.map(dow_map)
            .value_counts()
            .reindex(["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"])
            .fillna(0).astype(int)
    )

    dt_max_norm = pd.to_datetime(dt_max).normalize()
    window_days = 30
    if pd.notna(dt_min) and pd.notna(dt_max):
        span = (pd.to_datetime(dt_max) - pd.to_datetime(dt_min)).days
        if span < 45:
            window_days = 14
        if span < 21:
            window_days = 7

    dt_cut = dt_max_norm - pd.Timedelta(days=window_days)
    recent = df_occ.loc[df_occ["_dia_operacional"] >= dt_cut]
    total_recent = int(len(recent))
    if col_cad and (col_cad in df_occ.columns) and total_recent > 0:
        cad_recent = recent[col_cad].astype(str).str.strip()
        if not sem_corte_30 and (top_pessoas_cad_counts is not None) and (not top_pessoas_cad_counts.empty):
            cad_recent = cad_recent[cad_recent.isin(top_pessoas_cad_counts.index.astype(str))]
        label_recent = cad_recent.map(lambda c: f"{c} - {nome_map.get(c, '')}".strip(" -")) if nome_map else cad_recent
        top_recent = label_recent.value_counts()
    else:
        top_recent = pd.Series(dtype=int)

    def _rep_value_per_occ(df_in: pd.DataFrame, key_cols2: list[str], val_col: str) -> pd.Series:
        if (val_col is None) or (val_col not in df_in.columns):
            return pd.Series(dtype=str)
        tmp2 = df_in[key_cols2 + [val_col]].copy()
        tmp2[val_col] = tmp2[val_col].fillna("").astype(str).str.strip()
        tmp2 = tmp2[tmp2[val_col].ne("")]
        if tmp2.empty:
            return pd.Series(dtype=str)
        cnt = (
            tmp2.groupby(key_cols2 + [val_col], sort=False)
                .size()
                .reset_index(name="n")
                .sort_values(key_cols2 + ["n"], ascending=[True] * len(key_cols2) + [False], kind="mergesort")
        )
        rep = cnt.drop_duplicates(subset=key_cols2, keep="first").set_index(key_cols2)[val_col]
        return rep

    area_col_use = col_areas if col_areas else col_area_fallback
    dim_map = _get_dim_map(area_col_use, col_heranca, col_cargo)

    area_map: dict[str, str] = {}
    if isinstance(dim_map, pd.DataFrame) and (not dim_map.empty) and ("cadastro" in dim_map.columns) and ("_area_norm" in dim_map.columns):
        _area_tmp = dim_map[["cadastro", "_area_norm"]].copy(deep=False)
        _area_tmp["cadastro"] = _area_tmp["cadastro"].astype(str).str.strip()
        _area_tmp["_area_norm"] = _area_tmp["_area_norm"].fillna("").astype(str).str.strip()
        _area_tmp = _area_tmp.loc[_area_tmp["cadastro"].ne("")]
        if not _area_tmp.empty:
            area_map = _area_tmp.drop_duplicates(subset=["cadastro"], keep="last").set_index("cadastro")["_area_norm"].to_dict()

    area_rep = pd.Series(dtype=str)
    if area_col_use and col_cad and (col_cad in dfv.columns):
        area_rep = _rep_value_per_occ(dfv, [col_cad, "_dia_operacional"], area_col_use)

    planta_rep = pd.Series(dtype=str)
    if col_planta and col_cad and (col_cad in dfv.columns):
        planta_rep = _rep_value_per_occ(dfv, [col_cad, "_dia_operacional"], col_planta)
        if not planta_rep.empty:
            _pl_str = planta_rep.astype(str).str.strip()
            _pl_num = pd.to_numeric(_pl_str, errors="coerce")
            _mask_999 = _pl_num.eq(999) | _pl_str.str.match(r"^999(\D|$)", na=False)
            planta_rep = planta_rep.loc[~_mask_999]

    combo_rep = pd.Series(dtype=str)
    if (not area_rep.empty) and (not planta_rep.empty):
        idx = area_rep.index.intersection(planta_rep.index)
        if len(idx) > 0:
            combo_rep = (area_rep.loc[idx].astype(str) + " | " + planta_rep.loc[idx].astype(str))

    top_areas = area_rep.value_counts() if not area_rep.empty else pd.Series(dtype=int)
    top_plantas = planta_rep.value_counts() if not planta_rep.empty else pd.Series(dtype=int)
    top_combo = combo_rep.value_counts() if not combo_rep.empty else pd.Series(dtype=int)

    # =========================
    # PROXY (Ponto sem Portaria):
    # - mesmo dia operacional
    # - janela de até 1 minuto
    # - planta/família do coletor usadas como reforço, não exigência absoluta
    # =========================
    proxy_map: dict[str, dict] = {}
    proxy_top3_cads: list[str] = []
    proxy_diag = {
        "candidatos_avaliados": 0,
        "candidatos_com_eventos_elegiveis": 0,
        "eventos_elegiveis_candidatos": 0,
        "pares_proximos_tempo": 0,
        "pares_com_reforco_planta_ou_familia": 0,
        "pares_aceitos_suspeitos": 0,
        "pessoas_com_indicio": 0,
        "pessoas_com_recorrencia": 0,
    }

    if analisar_proxy_ponto and col_cad and (col_cad in df_occ.columns):
        base_all = globals().get("df_valid")
        if not isinstance(base_all, pd.DataFrame):
            base_all = df_irregularidade

        desc_col = None
        if "descricao coletor" in base_all.columns:
            desc_col = "descricao coletor"
        else:
            for c in ["descricao_coletor", "desc_coletor", "coletor", "descricao"]:
                if c in base_all.columns:
                    desc_col = c
                    break

        # Se df_valid não tiver descrição do coletor, tenta a base da própria irregularidade.
        if desc_col is None and isinstance(df_irregularidade, pd.DataFrame):
            base_alt = df_irregularidade
            if "descricao coletor" in base_alt.columns:
                desc_col = "descricao coletor"
            else:
                for c in ["descricao_coletor", "desc_coletor", "coletor", "descricao"]:
                    if c in base_alt.columns:
                        desc_col = c
                        break
            if desc_col is not None and ("cadastro" in base_alt.columns):
                base_all = base_alt

        if desc_col is not None and ("cadastro" in base_all.columns):
            MAX_PROXY_CANDIDATOS = 60
            JANELA_SEGUNDOS = 1 * 60
            MAX_EVENTOS_POR_CANDIDATO = 3000

            if (top_pessoas_cad_counts is not None) and (not top_pessoas_cad_counts.empty):
                proxy_candidate_cads = list(top_pessoas_cad_counts.head(MAX_PROXY_CANDIDATOS).index.astype(str))
            else:
                proxy_candidate_cads = list(df_occ[col_cad].astype(str).str.strip().dropna().unique()[:MAX_PROXY_CANDIDATOS])

            proxy_diag["candidatos_avaliados"] = int(len(proxy_candidate_cads))

            base = base_all.copy(deep=False)
            cad_base = base["cadastro"].astype(str).str.strip()

            if "_dt_ronda" in base.columns:
                dt_base = pd.to_datetime(base["_dt_ronda"], errors="coerce")
            else:
                dt_base = pd.to_datetime(_build_dt_from_data_hora(base), errors="coerce")

            if "_dia_operacional" in base.columns:
                dia_op_base = pd.to_datetime(base["_dia_operacional"], errors="coerce").dt.normalize()
            else:
                dia_op_base = (dt_base - pd.Timedelta(hours=11)).dt.normalize()

            if "planta" in base.columns:
                pl_base = pd.to_numeric(base["planta"], errors="coerce")
            else:
                pl_base = pd.Series(np.nan, index=base.index)

            desc_base = base[desc_col].fillna("").astype(str).str.strip()

            def _proxy_desc_family_series(s: pd.Series) -> pd.Series:
                fam = s.fillna("").astype(str).str.strip().str.upper()
                fam = fam.apply(lambda v: unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii"))
                fam = fam.str.replace(r"\b(CATRACA|COLETOR|RELOGIO|RELOGIOPONTO|PONTO|PORTARIA|ENTRADA|SAIDA)\b", " ", regex=True)
                fam = fam.str.replace(r"\d+", " ", regex=True)
                fam = fam.str.replace(r"[^A-Z]+", " ", regex=True)
                fam = fam.str.replace(r"\s+", " ", regex=True).str.strip()
                return fam

            fam_base = _proxy_desc_family_series(desc_base)

            point_mask = pd.Series(False, index=base.index)
            for bc in ["Ponto", "PONTO", "_PONTO", "_PONTO_PRESENCIAL", "PONTO_PRES", "HAS_PONTO_PRES", "PontoPresencial"]:
                if bc in base.columns and pd.api.types.is_bool_dtype(base[bc]):
                    point_mask = point_mask | base[bc].fillna(False)
            if not point_mask.any():
                point_mask = desc_base.str.contains(r"\bponto\b|rel[oó]gio|marca", case=False, na=False, regex=True)

            base_mask = point_mask & dt_base.notna() & dia_op_base.notna() & cad_base.ne("")

            if base_mask.any():
                ev = pd.DataFrame({
                    "cad": cad_base.loc[base_mask].astype(str).str.strip().to_numpy(),
                    "dt": pd.to_datetime(dt_base.loc[base_mask], errors="coerce").to_numpy(dtype="datetime64[ns]"),
                    "dia_op": pd.to_datetime(dia_op_base.loc[base_mask], errors="coerce").to_numpy(dtype="datetime64[ns]"),
                    "planta": pd.to_numeric(pl_base.loc[base_mask], errors="coerce").to_numpy(dtype=float),
                    "fam": fam_base.loc[base_mask].fillna("").astype(str).str.strip().to_numpy(dtype=object),
                })

                ev = ev.dropna(subset=["dt", "dia_op"]).copy()
                if not ev.empty:
                    ev["sec"] = (ev["dt"].astype("int64") // 1_000_000_000).astype(np.int64)
                    ev["dia_i"] = pd.to_datetime(ev["dia_op"], errors="coerce").to_numpy(dtype="datetime64[D]").astype(np.int64)
                    ev["planta_i"] = pd.to_numeric(ev["planta"], errors="coerce")
                    ev["fam"] = ev["fam"].fillna("").astype(str).str.strip()

                    day_groups: dict[int, dict[str, np.ndarray]] = {}
                    for day_i, g in ev.groupby("dia_i", sort=False):
                        g = g.sort_values("sec", kind="mergesort")
                        day_groups[int(day_i)] = {
                            "sec": g["sec"].to_numpy(dtype=np.int64, copy=False),
                            "cad": g["cad"].to_numpy(dtype=object, copy=False),
                            "planta": pd.to_numeric(g["planta_i"], errors="coerce").to_numpy(dtype=float, copy=False),
                            "fam": g["fam"].to_numpy(dtype=object, copy=False),
                        }

                    counts: dict[str, dict[str, int]] = {str(c): {} for c in proxy_candidate_cads}

                    for cad0 in proxy_candidate_cads:
                        ev0 = ev.loc[ev["cad"].eq(str(cad0)), ["dia_i", "sec", "planta_i", "fam"]]
                        if ev0.empty:
                            continue

                        proxy_diag["candidatos_com_eventos_elegiveis"] += 1
                        proxy_diag["eventos_elegiveis_candidatos"] += int(len(ev0))

                        if len(ev0) > MAX_EVENTOS_POR_CANDIDATO:
                            ev0 = ev0.sort_values("sec", kind="mergesort").head(MAX_EVENTOS_POR_CANDIDATO)

                        dct = counts.setdefault(str(cad0), {})

                        for day_i, g0 in ev0.groupby("dia_i", sort=False):
                            day_data = day_groups.get(int(day_i))
                            if day_data is None:
                                continue

                            sec_day = day_data["sec"]
                            cad_day = day_data["cad"]
                            pl_day = day_data["planta"]
                            fam_day = day_data["fam"]

                            for row in g0.itertuples(index=False):
                                sec0 = int(row.sec)
                                l = int(np.searchsorted(sec_day, sec0 - JANELA_SEGUNDOS, side="left"))
                                r = int(np.searchsorted(sec_day, sec0 + JANELA_SEGUNDOS, side="right"))
                                if r <= l:
                                    continue

                                sec_sub = sec_day[l:r]
                                cad_sub = cad_day[l:r]
                                pl_sub = pl_day[l:r]
                                fam_sub = fam_day[l:r]

                                mask_other = np.array([str(c).strip() != str(cad0) for c in cad_sub], dtype=bool)
                                if not mask_other.any():
                                    continue

                                sec_other = sec_sub[mask_other]
                                cad_other = cad_sub[mask_other]
                                pl_other = pl_sub[mask_other]
                                fam_other = fam_sub[mask_other]

                                dif = np.abs(sec_other - sec0)
                                proxy_diag["pares_proximos_tempo"] += int(len(dif))

                                pl0 = pd.to_numeric(row.planta_i, errors="coerce")
                                fam0 = str(row.fam).strip()

                                same_planta = (~np.isnan(pl_other)) & (not pd.isna(pl0)) & (pl_other == float(pl0))
                                same_fam = np.array(
                                    [(str(f).strip() != "") and (str(f).strip() == fam0) and (fam0 != "") for f in fam_other],
                                    dtype=bool,
                                )

                                reforco = same_planta | same_fam
                                proxy_diag["pares_com_reforco_planta_ou_familia"] += int(reforco.sum())

                                # Aceita somente pares com diferença máxima de 1 minuto.
                                accepted = dif <= JANELA_SEGUNDOS
                                if not accepted.any():
                                    continue

                                proxy_diag["pares_aceitos_suspeitos"] += int(accepted.sum())
                                for cad_other_i in cad_other[accepted]:
                                    cad_o = str(cad_other_i).strip()
                                    if cad_o == "" or cad_o == str(cad0):
                                        continue
                                    dct[cad_o] = dct.get(cad_o, 0) + 1

                    for cad0, dct in counts.items():
                        if not dct:
                            continue

                        top_other = sorted(dct.items(), key=lambda x: (-x[1], x[0]))[:10]
                        total_hits = int(sum(dct.values()))
                        max_hits_mesmo_terceiro = int(top_other[0][1]) if top_other else 0

                        proxy_map[str(cad0)] = {
                            "top_other": [(k, int(v)) for k, v in top_other],
                            "score_total_proxy": total_hits,
                            "qtd_terceiros": int(len(dct)),
                            "max_hits_mesmo_terceiro": max_hits_mesmo_terceiro,
                        }

                    proxy_diag["pessoas_com_indicio"] = int(len(proxy_map))
                    proxy_diag["pessoas_com_recorrencia"] = int(
                        sum(1 for _cad, info in proxy_map.items() if int(info.get("max_hits_mesmo_terceiro", 0)) >= 2)
                    )

                    if proxy_map:
                        def _cad_volume(cad: str) -> int:
                            if (top_pessoas_cad_counts is None) or top_pessoas_cad_counts.empty:
                                return 0
                            try:
                                return int(top_pessoas_cad_counts.get(cad, 0))
                            except Exception:
                                return 0

                        proxy_top3_cads = [
                            cad for cad, _info in sorted(
                                proxy_map.items(),
                                key=lambda kv: (
                                    -int(kv[1].get("score_total_proxy", 0)),        # 1) total de ocorrências suspeitas
                                    -int(kv[1].get("max_hits_mesmo_terceiro", 0)), # 2) repetição do mesmo terceiro
                                    -_cad_volume(kv[0]),                           # 3) volume total da pessoa
                                    str(kv[0]),
                                )
                            )[:3]
                        ]

    # =========================
    # PDF
    # =========================
    with PdfPages(pdf_path) as pdf:
        # CAPA (padrão único em tópicos)
        top3_piores_capa = []
        for k, v in top_pessoas.head(3).items():
            pct = (int(v) / total) if total else 0.0
            top3_piores_capa.append(f"{_wrap(k, 60)} — {int(v)} ({pct:.1%})")

        top3_atuais_capa = []
        for k, v in top_recent.head(3).items():
            pct = (int(v) / total_recent) if total_recent else 0.0
            top3_atuais_capa.append(f"{_wrap(k, 60)} — {int(v)} ({pct:.1%} da janela)")

        metricas_capa = [
            ("Ocorrências (pessoa-dia)", f"{total:,}".replace(",", ".")),
            ("Pessoas (cadastro)", f"{n_pessoas:,}".replace(",", ".")),
            ("Dias com ocorrência", f"{n_dias:,}".replace(",", ".")),
        ]
        periodo_capa = (
            f"{pd.to_datetime(dt_min).date()} → {pd.to_datetime(dt_max).date()}"
            if pd.notna(dt_min) and pd.notna(dt_max) else None
        )
        concentracoes_capa = [
            ("Concentração Top 1", f"{top1_n:,} ({top1_pct:.1%})".replace(",", ".")),
            ("Concentração Top 3", f"{top3_n:,} ({top3_pct:.1%})".replace(",", ".")),
            ("Concentração Top 10", f"{top10_n:,} ({top10_pct:.1%})".replace(",", ".")),
        ]
        janela_info = (
            f"{total_recent:,} ({(total_recent / total):.1%} do total)".replace(",", ".")
            if total > 0 else "0 (0.0% do total)"
        )

        _render_capa_padrao_pdf(
            pdf=pdf,
            titulo=titulo,
            gen_date=gen_date,
            metricas=metricas_capa,
            periodo=periodo_capa,
            concentracoes=concentracoes_capa,
            top3_piores=top3_piores_capa,
            top3_atuais=top3_atuais_capa,
            janela_info=janela_info,
        )

        # GERAIS
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11.69, 8.27), gridspec_kw={"height_ratios": [2.1, 1.3]})
        if mensal.empty:
            ax1.axis("off")
            ax1.text(0.02, 0.95, "Tendência mensal", fontsize=16, fontweight="bold", va="top")
            ax1.text(0.02, 0.80, "Sem datas válidas para agregação mensal.", fontsize=12, va="top")
        else:
            ax1.plot(mensal.index, mensal.values, marker="o", linewidth=2.1)
            ax1.fill_between(mensal.index, mensal.values, alpha=0.15)
            ax1.set_title("Tendência mensal (ocorrências pessoa-dia)")
            ax1.set_ylabel("Ocorrências")
            ax1.grid(alpha=0.25)

        ax2.set_title("Distribuição por dia da semana")
        ax2.bar(dow.index, dow.values)
        _annotate_bar(ax2, total_base=int(dow.values.sum()) if dow.values.size else None, fs=8.5)
        ax2.set_ylabel("Ocorrências")
        ax2.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        def _page_top_series(title: str, ser: pd.Series, max_items: int = 12):
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)
            if ser is None or ser.empty:
                ax.axis("off")
                ax.text(0.02, 0.95, title, fontsize=16, fontweight="bold", va="top")
                ax.text(0.02, 0.80, "Sem dados suficientes.", fontsize=12, va="top")
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
                return

            s = ser.head(max_items)
            y_pos = np.arange(len(s))
            labels = [_wrap(x, 60) for x in s.index.astype(str)]
            vals = s.values.astype(float)

            ax.barh(y_pos, vals, height=0.8)
            ax.set_yticks(y_pos)
            ax.set_yticklabels(labels)
            ax.invert_yaxis()
            ax.set_title(title)
            ax.set_xlabel("Ocorrências (pessoa-dia)")
            ax.grid(axis="x", alpha=0.25)

            xmax = max(float(vals.max()) * 1.22, 1.0)
            ax.set_xlim(0, xmax)
            _annotate_barh(ax, total_base=int(ser.sum()), fs=9)
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

        def _nome_abreviado(nome: str) -> str:
            nome = " ".join(str(nome).strip().split())
            if not nome:
                return "-"
            parts = [p for p in nome.split(" ") if p]
            if len(parts) == 1:
                return parts[0]
            return f"{parts[0]} {parts[-1][0]}."

        def _cad_label(cad: str) -> str:
            cad = str(cad).strip()
            return f"{cad} - {nome_map.get(cad, '')}".strip(" -") if nome_map else cad

        def _cad_proxy_label(cad: str) -> str:
            cad = str(cad).strip()
            nome_abv = _nome_abreviado(nome_map.get(cad, "")) if nome_map else "-"
            area_txt = (area_map.get(cad, "") if area_map else "")
            area_txt = str(area_txt).strip() if area_txt is not None else ""
            if area_txt == "":
                area_txt = "-"
            return f"{cad} | {nome_abv} | {area_txt}"

        def _page_top3_volume_pessoas():
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)

            if (top_pessoas_cad_counts is None) or top_pessoas_cad_counts.empty:
                ax.axis("off")
                ax.text(0.02, 0.95, "Top 3 pessoas — volume de ocorrências", fontsize=16, fontweight="bold", va="top")
                ax.text(0.02, 0.80, "Sem dados suficientes.", fontsize=12, va="top")
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
                return

            s = top_pessoas_cad_counts.head(3).copy()
            s.index = [_cad_label(c) for c in s.index.astype(str)]

            y_pos = np.arange(len(s))
            vals = s.values.astype(float)

            ax.barh(y_pos, vals, height=0.7)
            ax.set_yticks(y_pos)
            ax.set_yticklabels([_wrap(x, 60) for x in s.index.astype(str)])
            ax.invert_yaxis()
            ax.set_title("Top 3 pessoas — volume de ocorrências")
            ax.set_xlabel("Ocorrências (pessoa-dia)")
            ax.grid(axis="x", alpha=0.25)

            xmax = max(float(vals.max()) * 1.20, 1.0)
            ax.set_xlim(0, xmax)

            for i, v in enumerate(vals):
                pct = (float(v) / total * 100.0) if total else 0.0
                ax.text(min(v + xmax * 0.015, xmax * 0.98), i, f"{int(v)} ({pct:.1f}%)", va="center", fontsize=9)

            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

        def _page_top3_percentual_pessoas():
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)

            if (top_pessoas_cad_counts is None) or top_pessoas_cad_counts.empty or total <= 0:
                ax.axis("off")
                ax.text(0.02, 0.95, "Top 3 pessoas — percentual sobre o total", fontsize=16, fontweight="bold", va="top")
                ax.text(0.02, 0.80, "Sem dados suficientes.", fontsize=12, va="top")
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
                return

            s_counts = top_pessoas_cad_counts.head(3).copy()
            s_pct = (s_counts / total * 100.0).round(2)
            s_pct.index = [_cad_label(c) for c in s_pct.index.astype(str)]

            y_pos = np.arange(len(s_pct))
            vals = s_pct.values.astype(float)

            ax.barh(y_pos, vals, height=0.7)
            ax.set_yticks(y_pos)
            ax.set_yticklabels([_wrap(x, 60) for x in s_pct.index.astype(str)])
            ax.invert_yaxis()
            ax.set_title("Top 3 pessoas — percentual sobre o total")
            ax.set_xlabel("% do total de ocorrências")
            ax.grid(axis="x", alpha=0.25)

            xmax = max(float(vals.max()) * 1.20, 1.0)
            ax.set_xlim(0, xmax)

            for i, (pct_v, count_v) in enumerate(zip(vals, s_counts.values.astype(int))):
                ax.text(min(pct_v + xmax * 0.015, xmax * 0.98), i, f"{pct_v:.1f}% ({int(count_v)} ocorr.)", va="center", fontsize=9)

            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

        def _page_top3_proxy_pessoas():
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)

            diag_lines = [
                f"Candidatos avaliados: {int(proxy_diag.get('candidatos_avaliados', 0))}",
                f"Candidatos com eventos elegíveis: {int(proxy_diag.get('candidatos_com_eventos_elegiveis', 0))}",
                f"Eventos elegíveis (candidatos): {int(proxy_diag.get('eventos_elegiveis_candidatos', 0))}",
                f"Pares próximos no tempo (<=1 min): {int(proxy_diag.get('pares_proximos_tempo', 0))}",
                f"Pares com reforço planta/família: {int(proxy_diag.get('pares_com_reforco_planta_ou_familia', 0))}",
                f"Pares aceitos como suspeitos: {int(proxy_diag.get('pares_aceitos_suspeitos', 0))}",
                f"Pessoas com indício: {int(proxy_diag.get('pessoas_com_indicio', 0))}",
                f"Pessoas com recorrência (mesmo terceiro >=2): {int(proxy_diag.get('pessoas_com_recorrencia', 0))}",
            ]

            if (not analisar_proxy_ponto) or (not proxy_top3_cads):
                ax.axis("off")
                ax.text(0.02, 0.95, "Top 3 pessoas — maior indício de terceiro batendo ponto", fontsize=16, fontweight="bold", va="top")
                ax.text(0.02, 0.84, "Sem casos suficientes para ranking Top 3.", fontsize=12, va="top")
                ax.text(0.02, 0.76, "Funil quantitativo:", fontsize=12, fontweight="bold", va="top")
                ax.text(0.03, 0.72, "\n".join([f"• {x}" for x in diag_lines]), fontsize=10.5, va="top")
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
                return

            data_proxy = []
            for cad in proxy_top3_cads[:3]:
                info = proxy_map.get(str(cad), {})
                data_proxy.append({
                    "cad": str(cad),
                    "label": _cad_label(str(cad)),
                    "score": int(info.get("score_total_proxy", 0)),  # total de ocorrências de terceiros
                    "qtd_terceiros": int(info.get("qtd_terceiros", 0)),
                    "max_mesmo_terceiro": int(info.get("max_hits_mesmo_terceiro", 0)),
                })

            df_proxy = pd.DataFrame(data_proxy)

            if df_proxy.empty:
                ax.axis("off")
                ax.text(0.02, 0.95, "Top 3 pessoas — maior indício de terceiro batendo ponto", fontsize=16, fontweight="bold", va="top")
                ax.text(0.02, 0.84, "Sem dados suficientes.", fontsize=12, va="top")
                ax.text(0.02, 0.76, "Funil quantitativo:", fontsize=12, fontweight="bold", va="top")
                ax.text(0.03, 0.72, "\n".join([f"• {x}" for x in diag_lines]), fontsize=10.5, va="top")
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
                return

            y_pos = np.arange(len(df_proxy))
            vals = df_proxy["score"].to_numpy(dtype=float)

            ax.barh(y_pos, vals, height=0.7)
            ax.set_yticks(y_pos)
            ax.set_yticklabels([_wrap(x, 60) for x in df_proxy["label"].astype(str)])
            ax.invert_yaxis()
            ax.set_title("Top 3 pessoas — maior indício de terceiro batendo ponto")
            ax.set_xlabel("Qtd de ocorrências suspeitas por terceiros")
            ax.grid(axis="x", alpha=0.25)

            xmax = max(float(vals.max()) * 1.20, 1.0)
            ax.set_xlim(0, xmax)

            for i, r in enumerate(df_proxy.itertuples(index=False)):
                txt = (
                    f"ocorrências suspeitas={int(r.score)} | "
                    f"máx. mesmo terceiro={int(r.max_mesmo_terceiro)} | "
                    f"terceiros distintos={int(r.qtd_terceiros)}"
                )
                ax.text(min(float(r.score) + xmax * 0.015, xmax * 0.98), i, txt, va="center", fontsize=8.8)

            ax.text(
                0.02,
                0.02,
                "Funil: " + " | ".join(diag_lines[:6]),
                transform=ax.transAxes,
                fontsize=8.8,
                va="bottom",
                ha="left",
            )

            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

        _page_top_series("Top áreas (representatividade por pessoa-dia)", top_areas)
        _page_top_series("Top plantas (representatividade por pessoa-dia) — exclui planta 999", top_plantas)
        _page_top_series("Top área | planta (representatividade por pessoa-dia) — exclui planta 999", top_combo)

        _page_top3_volume_pessoas()
        _page_top3_percentual_pessoas()
        if analisar_proxy_ponto:
            _page_top3_proxy_pessoas()

        # TOP 3 (painel original por volume)
        if col_cad and (col_cad in df_occ.columns) and (top_pessoas_cad_counts is not None) and (not top_pessoas_cad_counts.empty):
            top3_cads_plot = list(top_pessoas_cad_counts.head(3).index.astype(str))
            top10_cads_plot = list(top_pessoas_cad_counts.head(10).index.astype(str))
            top10_total = int(top_pessoas_cad_counts.loc[top10_cads_plot].sum()) if top10_cads_plot else 0
            cad_irreg_set = set(df_occ[col_cad].astype(str).str.strip().unique())

            dt_end = pd.to_datetime(dt_max).normalize() if pd.notna(dt_max) else pd.Timestamp.today().normalize()
            dt_start_30 = dt_end - pd.Timedelta(days=29)
            idx_30 = pd.date_range(dt_start_30, dt_end, freq="D")

            for cad in top3_cads_plot:
                occ_p = df_occ.loc[df_occ[col_cad].astype(str).str.strip().eq(str(cad)), ["_dia_operacional"]]
                if occ_p.empty:
                    continue

                total_p = int(len(occ_p))
                pct_p_total = (total_p / total) if total else 0.0
                nome = nome_map.get(str(cad), "") if nome_map else ""
                label = f"{cad} - {nome}".strip(" -")

                days = occ_p["_dia_operacional"].dt.normalize()
                week_start = (days - pd.to_timedelta(days.dt.dayofweek, unit="D")).dt.normalize()
                weekly_days = week_start.value_counts().sort_index()
                mensal_semana = (
                    weekly_days.groupby(weekly_days.index.to_period("M").to_timestamp(), sort=False)
                               .mean()
                               .sort_index()
                    if not weekly_days.empty else pd.Series(dtype=float)
                )

                if ("_dia_operacional" in dfv.columns) and (col_cad in dfv.columns):
                    dfv_p = dfv.loc[
                        dfv[col_cad].astype(str).str.strip().eq(str(cad))
                        & dfv["_dia_operacional"].between(dt_start_30, dt_end),
                        ["_dia_operacional"],
                    ]
                    daily_30 = (
                        dfv_p.groupby(dfv_p["_dia_operacional"].dt.normalize(), sort=False)
                             .size()
                             .reindex(idx_30, fill_value=0)
                    )
                else:
                    daily_30 = (
                        occ_p.groupby(occ_p["_dia_operacional"].dt.normalize(), sort=False)
                             .size()
                             .reindex(idx_30, fill_value=0)
                    )

                pct_vs_top10 = (total_p / top10_total) if top10_total else 0.0

                area_val = None
                her_val = None
                if isinstance(dim_map, pd.DataFrame) and (not dim_map.empty) and ("cadastro" in dim_map.columns):
                    row_dim = dim_map.loc[dim_map["cadastro"].eq(str(cad))]
                    if not row_dim.empty:
                        area_val = row_dim["_area_norm"].iloc[0] or None
                        her_val = row_dim["_her_norm"].iloc[0] or None

                grp_total = grp_with = grp_without = None
                if (area_val is not None) and (her_val is not None) and isinstance(dim_map, pd.DataFrame) and (not dim_map.empty):
                    grp_cads = dim_map.loc[
                        dim_map["_area_norm"].eq(str(area_val)) & dim_map["_her_norm"].eq(str(her_val)),
                        "cadastro"
                    ].to_numpy()
                    grp_total = int(len(grp_cads))
                    if grp_total > 0:
                        grp_with = int(np.isin(grp_cads, list(cad_irreg_set)).sum())
                        grp_without = int(grp_total - grp_with)

                other_days = int(max(total - total_p, 0))

                fig = plt.figure(figsize=(11.69, 8.27))
                gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.0], wspace=0.38, hspace=0.48)

                ax1 = fig.add_subplot(gs[0, 0])
                ax2 = fig.add_subplot(gs[0, 1])
                ax3 = fig.add_subplot(gs[0, 2])
                ax4 = fig.add_subplot(gs[1, 0])
                ax5 = fig.add_subplot(gs[1, 1])
                ax6 = fig.add_subplot(gs[1, 2])
                ax6.axis("off")

                fig.suptitle(
                    f"Top 3 — {_wrap(label, 70)} | Total (pessoa-dia): {total_p} ({pct_p_total:.1%} do total)",
                    fontsize=15,
                    fontweight="bold",
                    y=0.985,
                )

                if mensal_semana.empty:
                    ax1.axis("off")
                    ax1.text(0.02, 0.92, "Dias com ocorrência na semana (mensal)", fontsize=11.5, fontweight="bold", va="top")
                    ax1.text(0.02, 0.78, "Sem dados suficientes.", fontsize=10, va="top")
                else:
                    ax1.plot(mensal_semana.index, mensal_semana.values, marker="o", linewidth=2.0)
                    ax1.fill_between(mensal_semana.index, mensal_semana.values, alpha=0.12)
                    ax1.set_title("Dias com ocorrência na semana\n(média semanal no mês)", fontsize=11.5)
                    ax1.set_xlabel("Mês")
                    ax1.set_ylabel("Dias/semana")
                    ax1.grid(alpha=0.25)
                    ax1.tick_params(axis="x", rotation=35)
                    ax1.set_ylim(0, 7)

                if daily_30.empty:
                    ax2.axis("off")
                    ax2.text(0.02, 0.92, "Últimos 30 dias (ocorrências/dia)", fontsize=11.5, fontweight="bold", va="top")
                    ax2.text(0.02, 0.78, "Sem dados suficientes.", fontsize=10, va="top")
                else:
                    ax2.plot(daily_30.index, daily_30.values, marker="o", linewidth=1.7)
                    ax2.fill_between(daily_30.index, daily_30.values, alpha=0.12)
                    ax2.set_title("Últimos 30 dias\n(ocorrências por dia)", fontsize=11.5)
                    ax2.set_xlabel("Dia")
                    ax2.set_ylabel("Ocorrências")
                    ax2.grid(alpha=0.25)
                    ax2.tick_params(axis="x", rotation=35)

                ax3.set_title("% de ocorrências vs total do Top 10", fontsize=11.5)
                ax3.barh(["Pessoa"], [pct_vs_top10], height=0.55)
                ax3.set_xlim(0, 1)
                ax3.set_xlabel("% do Top 10")
                ax3.grid(axis="x", alpha=0.25)
                ax3.text(
                    min(pct_vs_top10 + 0.02, 0.97), 0,
                    f"{pct_vs_top10:.1%}",
                    va="center",
                    ha="left" if pct_vs_top10 < 0.9 else "right",
                    fontsize=10,
                )

                _pie_with_legend(
                    ax4,
                    [grp_with if grp_with is not None else 0, grp_without if grp_without is not None else 0],
                    [
                        f"Com irregularidade ({0 if grp_with is None else grp_with})",
                        f"Sem irregularidade ({0 if grp_without is None else grp_without})",
                    ],
                    "Área + herança:\nmesmo grupo da pessoa",
                    f"Base: {0 if grp_total is None else grp_total} pessoas",
                )

                _pie_with_legend(
                    ax5,
                    [total_p, other_days],
                    [f"Pessoa ({total_p})", f"Demais ({other_days})"],
                    "Dias com ocorrência:\npessoa vs demais",
                    f"Base: {total} ocorrências pessoa-dia",
                )

                periodo_analisado_txt = f"{pd.to_datetime(dt_min).date()} → {pd.to_datetime(dt_max).date()}"
                periodo_desligado_txt = periodo_desligado_map.get(str(cad), "-") if mostrar_periodo_desligado else None

                ax6.text(0.00, 0.95, "Resumo", fontsize=13, fontweight="bold", va="top")
                ax6.text(0.00, 0.82, f"Cadastro: {cad}", fontsize=10.5, va="top")
                ax6.text(0.00, 0.72, f"Nome: {_wrap(nome if nome else '-', 28)}", fontsize=10.5, va="top")
                ax6.text(0.00, 0.58, f"Participação no total: {pct_p_total:.1%}", fontsize=10.5, va="top")
                ax6.text(0.00, 0.48, f"Participação no Top 10: {pct_vs_top10:.1%}", fontsize=10.5, va="top")
                ax6.text(0.00, 0.38, f"Área: {_wrap(area_val if area_val else '-', 28)}", fontsize=10.5, va="top")
                ax6.text(0.00, 0.30, f"Herança: {_wrap(her_val if her_val else '-', 28)}", fontsize=10.5, va="top")
                ax6.text(0.00, 0.22, f"Período: {periodo_analisado_txt}", fontsize=10.2, va="top")
                if mostrar_periodo_desligado:
                    ax6.text(0.00, 0.14, f"Período já desligado: {periodo_desligado_txt}", fontsize=10.0, va="top")

                if analisar_proxy_ponto:
                    info = proxy_map.get(str(cad))
                    ax6.text(0.00, 0.08, "Possíveis terceiros (mesmo dia operacional; <=1 min; chapa | nome abreviado | área)", fontsize=9.4, fontweight="bold", va="top")
                    if (not info) or (not info.get("top_other")):
                        ax6.text(0.00, 0.02, "Sem correspondências suficientes.", fontsize=9.0, va="top")
                    else:
                        lines = [f"- {_cad_proxy_label(k)} ({v})" for k, v in info["top_other"]]
                        ax6.text(0.00, 0.02, "\n".join(lines[:5]), fontsize=8.8, va="top")

                fig.tight_layout(rect=[0, 0, 1, 0.95])
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

        # 999 trajeto (heatmap) — mantido
        titulo_l = str(titulo).lower()
        if ("999" in titulo_l) and ("trajeto" in titulo_l or "999" in str(pdf_path).lower()):
            required_cols = ["cadastro", "data", "hora", "dir", "planta", "descricao coletor"]
            if all(c in df_irregularidade.columns for c in required_cols):
                cols_999 = required_cols + ([col_cargo] if (somente_executivos_atuais and col_cargo) else [])
                dfx = df_irregularidade.loc[:, [c for c in cols_999 if c in df_irregularidade.columns]].copy(deep=False)
                dfx["cadastro"] = dfx["cadastro"].astype(str).str.strip()

                if somente_executivos_atuais and col_cargo and (col_cargo in dfx.columns):
                    cargo_norm_999 = dfx[col_cargo].fillna("").astype(str).str.strip().str.upper()
                    dfx = dfx.loc[cargo_norm_999.eq("E")]

                data_dt = pd.to_datetime(dfx["data"], dayfirst=True, errors="coerce")
                hora_s = dfx["hora"].fillna("00:00").astype(str).str.strip()
                hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
                hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
                hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
                hora_td = pd.to_timedelta(hora_s, errors="coerce")
                dfx["_dt"] = data_dt.dt.normalize() + hora_td
                hora_corte = int(globals().get("hora_corte_operacional", 11))
                dfx["_dia_key"] = (dfx["_dt"] - pd.Timedelta(hours=hora_corte)).dt.normalize()

                _re_port = globals().get("regex_portaria", r"PORTARIA|PORT")
                desc = dfx["descricao coletor"].fillna("").astype(str)
                dfx["_is_port"] = desc.str.contains(_re_port, case=False, na=False, regex=True)

                dirn = dfx["dir"].fillna("").astype(str).str.strip().str.upper()
                dirn = dirn.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
                dfx["_is_E"] = dirn.str.startswith("E")
                dfx["_is_S"] = dirn.str.startswith("S")

                planta_num = pd.to_numeric(dfx["planta"], errors="coerce")
                dfx["_is_999"] = planta_num.eq(999)

                port = dfx.loc[dfx["_is_port"] & dfx["_dt"].notna(), ["cadastro", "_dia_key", "_dt", "_is_E", "_is_S"]]
                fig = plt.figure(figsize=(11.69, 8.27))
                ax = fig.add_subplot(111)

                if port.empty:
                    ax.axis("off")
                    ax.text(0.02, 0.92, "999 como trajeto — concentração por período", fontsize=18, fontweight="bold", va="top")
                    ax.text(0.02, 0.80, "Sem eventos de portaria válidos para classificar.", fontsize=13, va="top")
                    pdf.savefig(fig, bbox_inches="tight")
                    plt.close(fig)
                else:
                    ent = (
                        port.loc[port["_is_E"]]
                            .groupby(["cadastro", "_dia_key"], sort=False)["_dt"].min()
                            .rename("t_ent")
                            .reset_index()
                    )
                    sai = (
                        port.loc[port["_is_S"]]
                            .groupby(["cadastro", "_dia_key"], sort=False)["_dt"].max()
                            .rename("t_sai")
                            .reset_index()
                    )
                    pe = ent.merge(sai, on=["cadastro", "_dia_key"], how="inner")

                    ev = dfx.loc[dfx["_is_999"] & dfx["_dt"].notna(), ["cadastro", "_dia_key", "_dt"]]
                    if (not ev.empty) and (not pe.empty):
                        ev = ev.merge(pe, on=["cadastro", "_dia_key"], how="inner")
                        ev["_before"] = ev["_dt"] < ev["t_ent"]
                        ev["_after"] = ev["_dt"] > ev["t_sai"]

                        day_flags = (
                            ev.groupby(["cadastro", "_dia_key"], sort=False)[["_before", "_after"]]
                              .any()
                              .reset_index()
                        )
                        day_flags["cat"] = np.select(
                            [
                                day_flags["_before"] & day_flags["_after"],
                                day_flags["_before"] & (~day_flags["_after"]),
                                (~day_flags["_before"]) & day_flags["_after"],
                            ],
                            ["Ambos", "Antes", "Depois"],
                            default="(sem class.)",
                        )
                        day_flags = day_flags[day_flags["cat"].isin(["Antes", "Depois", "Ambos"])]

                        if day_flags.empty:
                            ax.axis("off")
                            ax.text(0.02, 0.92, "999 como trajeto — concentração por período", fontsize=18, fontweight="bold", va="top")
                            ax.text(0.02, 0.80, "Sem eventos 999 classificados após aplicar as regras.", fontsize=13, va="top")
                            pdf.savefig(fig, bbox_inches="tight")
                            plt.close(fig)
                        else:
                            agg = (
                                day_flags.groupby(["cadastro", "cat"], sort=False)
                                         .size()
                                         .rename("n_dias")
                                         .reset_index()
                            )
                            tot2 = agg.groupby("cadastro", sort=False)["n_dias"].sum().rename("n_total").reset_index()
                            agg = agg.merge(tot2, on="cadastro", how="left")
                            agg["pct"] = agg["n_dias"] / agg["n_total"].where(agg["n_total"] > 0, np.nan)

                            top_n_heatmap = min(10, len(tot2))
                            top_cads = (
                                tot2.sort_values(["n_total", "cadastro"], ascending=[False, True])
                                    .head(top_n_heatmap)["cadastro"].astype(str).tolist()
                            )

                            cats = ["Antes", "Depois", "Ambos"]
                            mat = (
                                agg.loc[agg["cadastro"].isin(top_cads)]
                                   .pivot_table(index="cadastro", columns="cat", values="pct", aggfunc="first")
                                   .reindex(columns=cats)
                                   .fillna(0.0)
                            )

                            if nome_map:
                                y_labels = [f"{c} - {nome_map.get(str(c), '')}".strip(" -") for c in mat.index]
                            else:
                                y_labels = list(mat.index)

                            ax.set_title(f"999 como trajeto — concentração por período (Top {top_n_heatmap})", fontsize=16)
                            ax.set_xlabel("Categoria de concentração (dia operacional)")
                            ax.set_xticks([0, 1, 2])
                            ax.set_xticklabels(["Antes de entrar", "Depois de sair", "Ambos"])
                            ax.set_yticks(range(len(y_labels)))
                            ax.set_yticklabels([_wrap(y, 58) for y in y_labels])

                            im = ax.imshow(mat.values, aspect="auto", cmap="Blues", vmin=0.0, vmax=1.0)
                            for yi in range(mat.shape[0]):
                                for xi in range(mat.shape[1]):
                                    p = float(mat.iloc[yi, xi])
                                    ax.text(xi, yi, f"{p * 100:.0f}%", ha="center", va="center", fontsize=9.5)

                            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                            cbar.set_label("% (linha = 100%)")

                            pdf.savefig(fig, bbox_inches="tight")
                            plt.close(fig)
                    else:
                        ax.axis("off")
                        ax.text(0.02, 0.92, "999 como trajeto — concentração por período", fontsize=18, fontweight="bold", va="top")
                        ax.text(0.02, 0.80, "Sem eventos 999 válidos para classificar.", fontsize=13, va="top")
                        pdf.savefig(fig, bbox_inches="tight")
                        plt.close(fig)

def row_ids_from_keys(df_min: pd.DataFrame, keys: pd.DataFrame) -> np.ndarray:
    if keys is None or keys.empty:
        return np.array([], dtype=np.int64)

    keys2 = keys.loc[:, ["cadastro", "_dia_operacional"]].drop_duplicates().copy()
    keys2["_dia_operacional"] = pd.to_datetime(keys2["_dia_operacional"], errors="coerce")

    if ("_ROWID_CACHE_HASH" not in globals()) or (len(globals()["_ROWID_CACHE_HASH"]) != len(df_min)):
        globals()["_ROWID_CACHE_ROWID"] = df_min["_row_id"].to_numpy(dtype=np.int64, copy=False)
        globals()["_ROWID_CACHE_HASH"] = pd.util.hash_pandas_object(
            df_min[["cadastro", "_dia_operacional"]], index=False
        ).to_numpy(dtype=np.uint64, copy=False)
        globals()["_ROWID_CACHE_HASH_INDEX"] = pd.Index(globals()["_ROWID_CACHE_HASH"])

    key_hash = pd.util.hash_pandas_object(
        keys2[["cadastro", "_dia_operacional"]], index=False
    ).to_numpy(dtype=np.uint64, copy=False)

    if key_hash.size == 0:
        return np.array([], dtype=np.int64)

    mask = globals()["_ROWID_CACHE_HASH_INDEX"].isin(key_hash)
    return globals()["_ROWID_CACHE_ROWID"][mask].astype(np.int64, copy=True)

# Mapa dia_operacional por row_id (para colocar na saída)
n_total = len(df_min_raw)
dia_op_arr = np.full(n_total, np.datetime64("NaT"), dtype="datetime64[ns]")
valid_map = df.loc[df["_dt_ronda"].notna(), ["_row_id", "_dia_operacional"]]
dia_op_arr[valid_map["_row_id"].to_numpy()] = valid_map["_dia_operacional"].to_numpy()

def materializar_e_exportar_por_rowid(
    row_ids: np.ndarray,
    irregularidade_col: str,
    titulo: str,
    df_relatorio: pd.DataFrame | None = None,
) -> pd.DataFrame:
    slug = slugify(irregularidade_col)

    parquet_path = DFS_DIR / f"{slug}.parquet"
    pdf_path = PDF_DIR / f"{slug}.pdf"

    excluir_planta_999_nesta_irregularidade = _deve_excluir_planta_999(
        irregularidade_col=irregularidade_col,
        titulo=titulo,
        pdf_path=pdf_path,
    )

    params_export = {
        "param_janela_portaria_min": globals().get("janela_portaria_min"),
        "param_tolerancia_entrada_min": globals().get("tolerancia_entrada_min"),
        "param_tolerancia_saida_min": globals().get("tolerancia_saida_min"),
        "param_janela_999_horas": globals().get("janela_999_horas"),
        "param_hora_corte_operacional": globals().get("hora_corte_operacional"),
        "param_descanso_min_horas": globals().get("descanso_min_horas"),
        "param_jornada_min_horas": globals().get("jornada_min_horas"),
        "param_jornada_min_minutos": globals().get("jornada_min_minutos"),
        "param_usar_jornada_por_portaria": globals().get("usar_jornada_por_portaria"),
        "param_max_jornada_portaria_horas": globals().get("max_jornada_portaria_horas"),
    }

    base_relatorio = df_relatorio if isinstance(df_relatorio, pd.DataFrame) else None

    if excluir_planta_999_nesta_irregularidade:
        row_ids = _filtrar_row_ids_sem_planta_999(row_ids)
        base_relatorio = _filtrar_df_sem_planta_999(base_relatorio)

    if row_ids.size == 0:
        df_out = df_min_raw.head(0).copy()
        df_out["dia_operacional"] = pd.to_datetime([], errors="coerce")
        df_out[irregularidade_col] = False
        for _k, _v in params_export.items():
            df_out[_k] = _v

        df_out.to_parquet(parquet_path, index=False)
        gerar_relatorio_pdf(base_relatorio if base_relatorio is not None else df_out, titulo=titulo, pdf_path=pdf_path)
        gc.collect()
        print("Parquet:", parquet_path)
        print("PDF   :", pdf_path)
        return df_out

    row_ids = row_ids[(row_ids >= 0) & (row_ids < len(df_min_raw))]

    MAX_ROWS_IN_MEMORY = 2_000_000
    REPORT_SAMPLE_ROWS = 2_000

    if row_ids.size > MAX_ROWS_IN_MEMORY:
        import pyarrow as pa
        import pyarrow.parquet as pq

        row_ids = np.sort(row_ids.astype(np.int64, copy=False))
        chunk_size = BATCH_SIZE

        if parquet_path.exists():
            parquet_path.unlink()

        writer = None
        report_parts = []
        report_count = 0

        for i in range(0, row_ids.size, chunk_size):
            ids = row_ids[i:i + chunk_size]
            part = df_min_raw.iloc[ids].copy()
            part["dia_operacional"] = pd.to_datetime(dia_op_arr[ids], errors="coerce")
            part[irregularidade_col] = True
            for _k, _v in params_export.items():
                part[_k] = _v
            part.drop(columns=["_row_id"], inplace=True, errors="ignore")

            table = pa.Table.from_pandas(part, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(str(parquet_path), table.schema, compression="snappy")
            writer.write_table(table)

            if report_count < REPORT_SAMPLE_ROWS:
                take = min(REPORT_SAMPLE_ROWS - report_count, len(part))
                report_parts.append(part.head(take).copy())
                report_count += take

            del ids, part, table
            gc.collect()

        if writer is not None:
            writer.close()

        if report_parts:
            df_report_fallback = pd.concat(report_parts, ignore_index=True)
        else:
            df_report_fallback = df_min_raw.head(0).copy()
            df_report_fallback["dia_operacional"] = pd.to_datetime([], errors="coerce")
            df_report_fallback[irregularidade_col] = False
            for _k, _v in params_export.items():
                df_report_fallback[_k] = _v

        if excluir_planta_999_nesta_irregularidade:
            df_report_fallback = _filtrar_df_sem_planta_999(df_report_fallback)

        gerar_relatorio_pdf(base_relatorio if base_relatorio is not None else df_report_fallback, titulo=titulo, pdf_path=pdf_path)
        gc.collect()
        print("Parquet:", parquet_path)
        print("PDF   :", pdf_path)
        return df_report_fallback

    df_out = df_min_raw.iloc[row_ids].copy()
    df_out["dia_operacional"] = pd.to_datetime(dia_op_arr[row_ids], errors="coerce")
    df_out[irregularidade_col] = True
    for _k, _v in params_export.items():
        df_out[_k] = _v
    df_out.drop(columns=["_row_id"], inplace=True, errors="ignore")

    if excluir_planta_999_nesta_irregularidade:
        df_out = _filtrar_df_sem_planta_999(df_out)

    df_out.to_parquet(parquet_path, index=False)
    gerar_relatorio_pdf(base_relatorio if base_relatorio is not None else df_out, titulo=titulo, pdf_path=pdf_path)

    gc.collect()
    print("Parquet:", parquet_path)
    print("PDF   :", pdf_path)
    return df_out

# Mantém referência dos helpers modernos para evitar regressão por redefinições posteriores.
_gerar_relatorio_pdf_moderno = gerar_relatorio_pdf
_row_ids_from_keys_moderno = row_ids_from_keys
_materializar_e_exportar_por_rowid_moderno = materializar_e_exportar_por_rowid

In [ ]:
# =========================
# IRREGULARIDADE: 999 como Trajeto
# =========================
irregularidade_col = "Irregularidade - 999 como Trajeto"
titulo = "Irregularidade – 999 como Trajeto"

# Base: precisa ter presencial e 999 no mesmo dia
day999_counts = (
    df_valid.loc[df_valid["999"].fillna(False), ["cadastro", "_dia_operacional"]]
      .groupby(["cadastro", "_dia_operacional"])
      .size()
      .rename("n_999")
      .reset_index()
)

day_ = day.merge(day999_counts, on=["cadastro", "_dia_operacional"], how="left")
day_["n_999"] = day_["n_999"].fillna(0).astype(int)

# Anti-truncamento específico da janela 999
day_["ANTI_TRUNC_999"] = False
mask_mm = day_["min_dt_cad"].notna() & day_["max_dt_cad"].notna()
day_.loc[mask_mm, "ANTI_TRUNC_999"] = (
    (day_.loc[mask_mm, "first_any_pres"] - t_jan_999) < day_.loc[mask_mm, "min_dt_cad"]
) | (
    (day_.loc[mask_mm, "last_any_pres"] + t_jan_999) > day_.loc[mask_mm, "max_dt_cad"]
)

# Conta 999 antes/depois do presencial
mask_999 = df_valid["999"].fillna(False).to_numpy()
if mask_999.any():
    idx_999 = np.flatnonzero(mask_999)
    ev_999 = df_valid.iloc[idx_999][["cadastro", "_dia_operacional", "_dt_ronda"]].copy()
else:
    ev_999 = pd.DataFrame(columns=["cadastro", "_dia_operacional", "_dt_ronda"])
del mask_999
ev_999 = ev_999.merge(
    day_.loc[:, ["cadastro", "_dia_operacional", "first_any_pres", "last_any_pres"]],
    on=["cadastro", "_dia_operacional"],
    how="inner",
)

ev_999["in_before"] = (ev_999["_dt_ronda"] >= (ev_999["first_any_pres"] - t_jan_999)) & (ev_999["_dt_ronda"] < ev_999["first_any_pres"])
ev_999["in_after"]  = (ev_999["_dt_ronda"] >  ev_999["last_any_pres"]) & (ev_999["_dt_ronda"] <= (ev_999["last_any_pres"] + t_jan_999))

cnt = (
    ev_999.groupby(["cadastro", "_dia_operacional"], as_index=False)
          .agg(cnt_before=("in_before", "sum"), cnt_after=("in_after", "sum"))
)

day_ = day_.merge(cnt, on=["cadastro", "_dia_operacional"], how="left")
day_[["cnt_before", "cnt_after"]] = day_[["cnt_before", "cnt_after"]].fillna(0).astype(int)

odd_before = (day_["cnt_before"] % 2 == 1) & (day_["cnt_before"] > 0)
odd_after  = (day_["cnt_after"] % 2 == 1) & (day_["cnt_after"] > 0)

day_[irregularidade_col] = (
    (day_["n_999"] > 0)
    & day_["HAS_PORTARIA_ANY"]
    & (odd_before | odd_after)
    & (~day_["ANTI_TRUNC_999"])
)

# ==========================================================
# 4.1) CORREÇÃO "FALSO VERDADEIRO" (ponto fechado) -> zera o dia
#     (INSERIDO AQUI — altera apenas o necessário)
# ==========================================================
def _norm_dir_series(s: pd.Series) -> pd.Series:
    x = s.fillna("").astype(str).str.strip().str.upper()
    x = x.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
    x = x.where(~x.str.startswith("E"), "E")
    x = x.where(~x.str.startswith("S"), "S")
    return x

def _dia_e_falso_verdadeiro(g: pd.DataFrame) -> bool:
    # g deve conter: _dt_ronda, _is_portaria, _dir_norm, _is_ponto_marca
    g = g.sort_values("_dt_ronda", kind="mergesort")
    port = g[g["_is_portaria"]]
    if port.empty:
        return False

    first_port = port.iloc[0]
    last_port  = port.iloc[-1]

    if first_port["_dir_norm"] != "E":
        return False
    if last_port["_dir_norm"] != "S":
        return False

    t0 = first_port["_dt_ronda"]
    t1 = last_port["_dt_ronda"]
    if pd.isna(t0) or pd.isna(t1) or t1 <= t0:
        return False

    # Se houver 999 valido na janela antes/depois da portaria, nao descartar como falso verdadeiro.
    is_999 = g.get("999", pd.Series(False, index=g.index)).fillna(False)
    if is_999.any():
        t999 = g.loc[is_999, "_dt_ronda"]
        has_999_before = ((t999 >= (t0 - t_jan_999)) & (t999 < t0)).any()
        has_999_after = ((t999 > t1) & (t999 <= (t1 + t_jan_999))).any()
        if has_999_before or has_999_after:
            return False

    pontos_entre = g[
        g["_is_ponto_marca"]
        & (g["_dt_ronda"] >= t0)
        & (g["_dt_ronda"] <= t1)
    ]
    return len(pontos_entre) >= 2

keys_true = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].drop_duplicates()

grupos_descartar = set()
if not keys_true.empty:
    # traz só linhas necessárias do df_valid para validar falsos positivos
    cols_need = ["cadastro", "_dia_operacional", "_dt_ronda", "999"]
    if "Portaria" in df_valid.columns: cols_need.append("Portaria")
    if "dir" in df_valid.columns: cols_need.append("dir")
    if "_dir_norm" in df_valid.columns: cols_need.append("_dir_norm")
    if "descricao coletor" in df_valid.columns: cols_need.append("descricao coletor")

    cols_base = [c for c in cols_need if c in df_valid.columns]

    # reduz o universo por cadastro antes do merge por dia
    cads_true = keys_true["cadastro"].drop_duplicates()
    mask_cad = df_valid["cadastro"].isin(cads_true)
    df_check = df_valid.loc[mask_cad, cols_base]
    del mask_cad

    df_check = df_check.merge(keys_true, on=["cadastro", "_dia_operacional"], how="inner")

    # portaria flag
    if "Portaria" in df_check.columns:
        df_check["_is_portaria"] = df_check["Portaria"].fillna(False).astype(bool)
    else:
        if "descricao coletor" in df_check.columns:
            descu = df_check["descricao coletor"].fillna("").astype(str).str.upper()
        else:
            descu = pd.Series("", index=df_check.index, dtype="string")
        df_check["_is_portaria"] = descu.str.contains(r"PORT", regex=True, na=False)

    # direção normalizada
    if "_dir_norm" in df_check.columns:
        df_check["_dir_norm"] = _norm_dir_series(df_check["_dir_norm"])
    elif "dir" in df_check.columns:
        df_check["_dir_norm"] = _norm_dir_series(df_check["dir"])
    else:
        df_check["_dir_norm"] = ""

    # ponto/marca (?:MARCA|PONTO|PONT) e NÃO portaria
    if "descricao coletor" in df_check.columns:
        descu = df_check["descricao coletor"].fillna("").astype(str).str.upper()
    else:
        descu = pd.Series("", index=df_check.index, dtype="string")
    df_check["_is_ponto_marca"] = (
        descu.str.contains(r"(?:MARCA|PONTO|PONT)", regex=True, na=False)
        & (~df_check["_is_portaria"])
    )

    grp_flag = (
        df_check.groupby(["cadastro", "_dia_operacional"], sort=False)
                .apply(_dia_e_falso_verdadeiro)
    )
    grupos_descartar = set(grp_flag[grp_flag].index.tolist())

if grupos_descartar:
    day_["_k"] = list(zip(day_["cadastro"], day_["_dia_operacional"]))
    day_.loc[day_["_k"].isin(grupos_descartar), irregularidade_col] = False
    day_.drop(columns=["_k"], inplace=True, errors="ignore")

# (continua o fluxo normal após a validação)
keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()

row_ids = row_ids_from_keys(df, keys)
df_999_trajeto = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

# (opcional) manter em memória
if MANTER_DF_EM_MEMORIA:
    df_irregularidade_999_como_trajeto = df_999_trajeto

df_999_trajeto.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Ponto sem Portaria
# Regra: existe PONTO presencial no dia, mas NÃO existe ENTRADA na portaria em até 1 minuto do 1º ponto presencial.
# Não contar executivos. Não acusar em borda de contexto.
# Regra adicional: se houver pelo menos uma ENTRADA e pelo menos uma SAÍDA naquele dia,
#                  o dia já é removido da irregularidade.
# =========================
irregularidade_col = "Irregularidade - Ponto sem Portaria"
titulo = "Irregularidade – Ponto sem Portaria (sem ENTRADA de portaria em até 1 minuto do 1º ponto presencial; dias com ENTRADA+SAÍDA em portaria são desconsiderados)"

day_ = day.copy()

port = df_valid.loc[
    df_valid["Portaria"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]
].copy(deep=False)

# janela ancorada no primeiro ponto presencial (±1 minuto)
day_["w_start_ponto"] = day_["first_ponto_pres"] - pd.Timedelta(minutes=1)
day_["w_end_ponto"]   = day_["first_ponto_pres"] + pd.Timedelta(minutes=1)

day_["ok_entry_ponto"] = False
day_["tem_entrada_saida_portaria_dia"] = False

if not port.empty:
    # NOVA REGRA:
    # se houver ao menos uma entrada e ao menos uma saída em portaria no dia,
    # o dia é descartado da irregularidade
    port_flags = (
        port.assign(
            _is_entry=port["_dir_norm"].eq("E"),
            _is_saida=port["_dir_norm"].eq("S"),
        )
        .groupby(["cadastro", "_dia_operacional"], sort=False)
        .agg(
            tem_entrada_portaria_dia=("_is_entry", "any"),
            tem_saida_portaria_dia=("_is_saida", "any"),
        )
        .reset_index()
    )

    port_flags["_tem_entrada_saida_portaria_dia"] = (
        port_flags["tem_entrada_portaria_dia"] & port_flags["tem_saida_portaria_dia"]
    )

    day_ = day_.merge(
        port_flags[["cadastro", "_dia_operacional", "_tem_entrada_saida_portaria_dia"]],
        on=["cadastro", "_dia_operacional"],
        how="left",
    )

    day_["tem_entrada_saida_portaria_dia"] = (
        day_["_tem_entrada_saida_portaria_dia"].fillna(False).astype(bool)
    )
    day_.drop(columns=["_tem_entrada_saida_portaria_dia"], inplace=True, errors="ignore")

    port = port.merge(
        day_[["cadastro", "_dia_operacional", "w_start_ponto", "w_end_ponto", "first_ponto_pres"]],
        on=["cadastro", "_dia_operacional"],
        how="inner",
    )

    in_window = (port["_dt_ronda"] >= port["w_start_ponto"]) & (port["_dt_ronda"] <= port["w_end_ponto"])

    port["ok_entry_ponto"] = (
        in_window
        & port["_dir_norm"].eq("E")
        & port["first_ponto_pres"].notna()
    )

    ok = (
        port.groupby(["cadastro", "_dia_operacional"], sort=False)["ok_entry_ponto"]
            .any()
            .rename("ok_entry_ponto_portaria")
            .reset_index()
    )

    day_ = day_.merge(ok, on=["cadastro", "_dia_operacional"], how="left")
    day_["ok_entry_ponto"] = day_["ok_entry_ponto_portaria"].fillna(False).astype(bool)
    day_.drop(columns=["ok_entry_ponto_portaria"], inplace=True, errors="ignore")

day_[irregularidade_col] = (
    (~day_["SEM_COBERTURA_CONTEXTO"])
    & (~day_["IS_EXECUTIVO_DAY"])          # não incluir executivos (cargo == 'E')
    & (day_["first_ponto_pres"].notna())
    & (~day_["tem_entrada_saida_portaria_dia"])   # novo filtro
    & (~day_["ok_entry_ponto"])
).fillna(False)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()
row_ids = row_ids_from_keys(df, keys)
df_ponto_sem_portaria = materializar_e_exportar_por_rowid(
    row_ids,
    irregularidade_col=irregularidade_col,
    titulo=titulo,
)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_ponto_sem_portaria = df_ponto_sem_portaria

df_ponto_sem_portaria.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Registro sem Portaria
# =========================
irregularidade_col = "Irregularidade - Registro sem Portaria"
titulo = "Irregularidade – Registro sem Portaria"

day_ = day.copy()
day_[irregularidade_col] = (
    (~day_["SEM_COBERTURA_CONTEXTO"])
    & (~day_["IS_EXECUTIVO_DAY"])          # FIX: não incluir executivos (cargo == 'E')
    & (day_["SEM_PORTARIA_COERENTE"])
    & (day_["first_any_pres"].notna())
)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()

row_ids = row_ids_from_keys(df, keys)
df_registro_sem_portaria = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_registro_sem_portaria = df_registro_sem_portaria

df_registro_sem_portaria.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Executivos com Histórico de 999 como trajeto (redefinida)
# =========================
# Nova definição:
#   "Executivos (cargo E) que, ANTES de virarem executivos, cometiam a irregularidade '999 como Trajeto'."
#
# A saída retorna os DIAS PRÉ-PROMOÇÃO em que a mesma lógica da irregularidade 999 como trajeto é verdadeira.

irregularidade_col = "Irregularidade - Executivos com Histórico de 999 como Trajeto"
titulo = "Irregularidade – Executivos com histórico de 999 como Trajeto (pré promoção)"

# Data de transição para cargo E (primeira ocorrência de cargo E no histórico)
df_exec = df_valid.loc[df_valid["_is_executivo"].fillna(False), ["cadastro", "_dt_ronda"]].copy()
promo = (
    df_exec.groupby("cadastro")["_dt_ronda"].min()
          .rename("dt_promocao")
          .reset_index()
)

# Dia operacional da promoção (mesma lógica do corte operacional)
promo["dia_promocao_op"] = (promo["dt_promocao"] - pd.Timedelta(hours=hora_corte_operacional)).dt.normalize()

# Day base apenas para quem virou executivo
day_ = day.merge(promo, on="cadastro", how="inner")

# =========================
# Recalcula a irregularidade 999 (mesma lógica do bloco 999)
# =========================

# Base: precisa ter 999 no dia
day999_counts = (
    df_valid.loc[df_valid["999"].fillna(False), ["cadastro", "_dia_operacional"]]
      .groupby(["cadastro", "_dia_operacional"])
      .size()
      .rename("n_999")
      .reset_index()
)

day_ = day_.merge(day999_counts, on=["cadastro", "_dia_operacional"], how="left")
day_["n_999"] = day_["n_999"].fillna(0).astype(int)

# Anti-truncamento específico da janela 999
day_["ANTI_TRUNC_999"] = False
mask_mm = day_["min_dt_cad"].notna() & day_["max_dt_cad"].notna()
day_.loc[mask_mm, "ANTI_TRUNC_999"] = (
    (day_.loc[mask_mm, "first_any_pres"] - t_jan_999) < day_.loc[mask_mm, "min_dt_cad"]
) | (
    (day_.loc[mask_mm, "last_any_pres"] + t_jan_999) > day_.loc[mask_mm, "max_dt_cad"]
)

# Conta 999 antes/depois do presencial
ev_999 = df_valid.loc[df_valid["999"].fillna(False), ["cadastro", "_dia_operacional", "_dt_ronda"]].copy()
ev_999 = ev_999.merge(
    day_.loc[:, ["cadastro", "_dia_operacional", "first_any_pres", "last_any_pres"]],
    on=["cadastro", "_dia_operacional"],
    how="inner",
)

ev_999["in_before"] = (ev_999["_dt_ronda"] >= (ev_999["first_any_pres"] - t_jan_999)) & (ev_999["_dt_ronda"] < ev_999["first_any_pres"])
ev_999["in_after"]  = (ev_999["_dt_ronda"] >  ev_999["last_any_pres"]) & (ev_999["_dt_ronda"] <= (ev_999["last_any_pres"] + t_jan_999))

cnt = (
    ev_999.groupby(["cadastro", "_dia_operacional"], as_index=False)
          .agg(cnt_before=("in_before", "sum"), cnt_after=("in_after", "sum"))
)

day_ = day_.merge(cnt, on=["cadastro", "_dia_operacional"], how="left")
day_[["cnt_before", "cnt_after"]] = day_[["cnt_before", "cnt_after"]].fillna(0).astype(int)

odd_before = (day_["cnt_before"] % 2 == 1) & (day_["cnt_before"] > 0)
odd_after  = (day_["cnt_after"] % 2 == 1) & (day_["cnt_after"] > 0)

flag_999_irregularidade = (
    (day_["n_999"] > 0)
    & day_["HAS_PORTARIA_ANY"]
    & (odd_before | odd_after)
    & (~day_["ANTI_TRUNC_999"])
)

# =========================
# Irregularidade executivos: somente pré promoção
# =========================
dia_op = pd.to_datetime(day_["_dia_operacional"], errors="coerce").to_numpy(dtype="datetime64[ns]")
dia_prom_op = pd.to_datetime(day_["dia_promocao_op"], errors="coerce").to_numpy(dtype="datetime64[ns]")

day_[irregularidade_col] = (
    day_["dt_promocao"].notna().to_numpy()
    & (dia_op < dia_prom_op)
    & flag_999_irregularidade.to_numpy()
)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()

# Materialização do df_ronda filtrado pelo df temporário (keys)
row_ids = row_ids_from_keys(df, keys)
df_exec_hist_999 = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_ponto_de_executivos = df_exec_hist_999

df_exec_hist_999.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Ausência de Descanso entre Jornadas de Trabalho
# Regra: descanso mínimo 11h até a próxima jornada (usa Portaria E->S com fallback any_pres).
# Mantém também o dia anterior usado no cálculo.
# =========================
irregularidade_col = "Irregularidade - Ausência de Descanso entre Jornadas de Trabalho"
titulo = "Irregularidade – Ausência de Descanso entre Jornadas (mín. 11h)"

t_desc_min = pd.Timedelta(hours=11)

day_ = day.sort_values(["cadastro", "_dia_operacional"], kind="mergesort").copy()

port = df_valid.loc[
    df_valid["Portaria"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]
].copy(deep=False)

if port.empty:
    day_["first_port_E"] = pd.NaT
    day_["last_port_S"]  = pd.NaT
else:
    first_E = (
        port.loc[port["_dir_norm"].eq("E")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .min()
            .rename("first_port_E")
            .reset_index()
    )
    last_S = (
        port.loc[port["_dir_norm"].eq("S")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .max()
            .rename("last_port_S")
            .reset_index()
    )
    day_ = day_.merge(first_E, on=["cadastro", "_dia_operacional"], how="left")
    day_ = day_.merge(last_S,  on=["cadastro", "_dia_operacional"], how="left")

day_["j_start"] = day_["first_port_E"].where(day_["first_port_E"].notna(), day_["first_any_pres"])
day_["j_end"]   = day_["last_port_S"].where(day_["last_port_S"].notna(),  day_["last_any_pres"])

day_["prev_dia_operacional"] = day_.groupby("cadastro")["_dia_operacional"].shift(1)
day_["prev_j_end"] = day_.groupby("cadastro")["j_end"].shift(1)
day_["prev_dur"]   = day_.groupby("cadastro")["dur_any_pres"].shift(1)
day_["prev_sem_cob"] = day_.groupby("cadastro")["SEM_COBERTURA_CONTEXTO"].shift(1).fillna(False)

descanso = day_["j_start"] - day_["prev_j_end"]

day_[irregularidade_col] = (
    (~day_["SEM_COBERTURA_CONTEXTO"])
    & (~day_["prev_sem_cob"])
    & (day_["prev_j_end"].notna())
    & (day_["j_start"].notna())
    & (descanso < t_desc_min)
    & (day_["prev_dur"] >= t_jornada_min)
).fillna(False)

keys_main = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()
keys_prev = (
    day_.loc[day_[irregularidade_col], ["cadastro", "prev_dia_operacional"]]
        .rename(columns={"prev_dia_operacional": "_dia_operacional"})
        .dropna(subset=["_dia_operacional"])
)
keys = pd.concat([keys_main, keys_prev], ignore_index=True).drop_duplicates()

row_ids = row_ids_from_keys(df, keys)
df_descanso = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_ausencia_descanso = df_descanso

df_descanso.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Pessoas Trabalhando Além da Jornada de Trabalho
# Regra: presença na planta fora do intervalo Portaria E->S por >=2h.
# Não contar executivos. Não acusar em borda de contexto.
# =========================
irregularidade_col = "Irregularidade - Pessoas Trabalhando Além da Jornada de Trabalho"
titulo = "Irregularidade – Presencial fora da Portaria (>= 2h)"

day_ = day.copy()

port = df_valid.loc[
    df_valid["Portaria"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]
].copy(deep=False)

if port.empty:
    day_["first_port_E"] = pd.NaT
    day_["last_port_S"]  = pd.NaT
else:
    first_E = (
        port.loc[port["_dir_norm"].eq("E")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .min()
            .rename("first_port_E")
            .reset_index()
    )
    last_S = (
        port.loc[port["_dir_norm"].eq("S")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .max()
            .rename("last_port_S")
            .reset_index()
    )
    day_ = day_.merge(first_E, on=["cadastro", "_dia_operacional"], how="left")
    day_ = day_.merge(last_S,  on=["cadastro", "_dia_operacional"], how="left")

before_gap = day_["first_port_E"] - day_["first_any_pres"]
after_gap  = day_["last_any_pres"] - day_["last_port_S"]

day_[irregularidade_col] = (
    (~day_["SEM_COBERTURA_CONTEXTO"])
    & (~day_["IS_EXECUTIVO_DAY"])
    & (day_["first_any_pres"].notna())
    & (day_["last_any_pres"].notna())
    & (day_["first_port_E"].notna())
    & (day_["last_port_S"].notna())
    & ((before_gap >= t_gap_work) | (after_gap >= t_gap_work))
).fillna(False)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()
row_ids = row_ids_from_keys(df, keys)
df_alem_jornada = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_alem_jornada = df_alem_jornada

df_alem_jornada.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Pessoas Trabalhando sem Ganhar
# Regra: ficou na fábrica >=2h (Portaria E->S com fallback any_pres) e NÃO bateu ponto.
# Não contar executivos. Não acusar em borda de contexto.
# =========================
irregularidade_col = "Irregularidade - Pessoas Trabalhando sem Ganhar"
titulo = "Irregularidade – Trabalhando sem Ganhar (>=2h na fábrica e sem ponto)"

day_ = day.copy()

port = df_valid.loc[
    df_valid["Portaria"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]
].copy(deep=False)

if port.empty:
    day_["first_port_E"] = pd.NaT
    day_["last_port_S"]  = pd.NaT
else:
    first_E = (
        port.loc[port["_dir_norm"].eq("E")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .min()
            .rename("first_port_E")
            .reset_index()
    )
    last_S = (
        port.loc[port["_dir_norm"].eq("S")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .max()
            .rename("last_port_S")
            .reset_index()
    )
    day_ = day_.merge(first_E, on=["cadastro", "_dia_operacional"], how="left")
    day_ = day_.merge(last_S,  on=["cadastro", "_dia_operacional"], how="left")

dur_port = day_["last_port_S"] - day_["first_port_E"]
dur_fabrica = dur_port.where(dur_port.notna() & (dur_port >= pd.Timedelta(0)), day_["dur_any_pres"])

day_[irregularidade_col] = (
    (~day_["SEM_COBERTURA_CONTEXTO"])
    & (~day_["IS_EXECUTIVO_DAY"])
    & (dur_fabrica >= t_gap_work)  # t_gap_work = 2h
    & (~day_["HAS_PONTO_ANY"])
).fillna(False)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()
row_ids = row_ids_from_keys(df, keys)
df_sem_ganhar = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_sem_ganhar = df_sem_ganhar

df_sem_ganhar.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Registros de Pessoas Desligadas (se existir data de desligado)
# =========================
irregularidade_col = "Irregularidade - Registros de Pessoas Desligadas"
titulo = "Irregularidade – Registros de Pessoas Desligadas"

# Descobre coluna de desligado SEM carregar o parquet inteiro
dem_col = None
for cand in ["Data_Dem", "data_desligado", "DataDemissao", "data_demissao"]:
    try:
        tmp = pd.read_parquet(PARQUET_PATH, columns=["cadastro", cand])
        dem_col = cand
        break
    except Exception:
        continue

# limpa cache de row_id/hash para reduzir pico de memória nesta etapa
for _v in ("_ROWID_CACHE_HASH", "_ROWID_CACHE_HASH_INDEX", "_ROWID_CACHE_ROWID"):
    if _v in globals():
        del globals()[_v]
gc.collect()

day_ = day.loc[:, ["cadastro", "_dia_operacional"]]

if dem_col is None:
    print("⚠️ Nenhuma coluna de data de desligado encontrada. Irregularidade ficará vazia.")
    day_[irregularidade_col] = False
else:
    dem = tmp.copy()
    dem["cadastro"] = dem["cadastro"].astype(str).str.strip()
    dem[dem_col] = pd.to_datetime(dem[dem_col], dayfirst=True, errors="coerce")
    dem = dem.loc[dem[dem_col].notna()].copy()
    afast_map = dem.groupby("cadastro")[dem_col].max().rename("dt_dem").reset_index()

    day_ = day_.merge(afast_map, on="cadastro", how="left")
    day_[irregularidade_col] = day_["dt_dem"].notna() & (pd.to_datetime(day_["_dia_operacional"], errors="coerce") > day_["dt_dem"])

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()

row_ids = row_ids_from_keys(df, keys)
df_desligados = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_desligados = df_desligados

df_desligados.head(3)


In [ ]:
# =========================
# Copiando base
# =========================
df = df_min_raw.copy()

# cadastro como string
df["cadastro"] = df["cadastro"].astype(str).str.strip()

# descrição
desc = df["descricao coletor"].fillna("").astype(str)

df["Portaria"] = desc.str.contains(regex_portaria, case=False, na=False, regex=True)
df["Ponto"] = desc.str.contains(regex_ponto, case=False, na=False, regex=True)
df["_REFEITORIO"] = desc.str.contains(regex_refeitorio, case=False, na=False, regex=True)
df["_GRILL"] = desc.str.contains(regex_grill, case=False, na=False, regex=True)

# planta == 999 (robusto)
planta_num = pd.to_numeric(df["planta"], errors="coerce")
df["999"] = planta_num.eq(999)

# Presencial: planta válida e não 999
df["Presencial"] = planta_num.notna() & (~df["999"])

# Executivo: cargo == 'E'
cargo_norm = df["cargo"].fillna("").astype(str).str.strip().str.upper()
df["_is_executivo"] = cargo_norm.eq("E")

# dir normalizado
dir_norm = df["dir"].fillna("").astype(str).str.strip().str.upper()
dir_norm = dir_norm.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
dir_norm = dir_norm.where(~dir_norm.str.startswith("E"), "E")
dir_norm = dir_norm.where(~dir_norm.str.startswith("S"), "S")
df["_dir_norm"] = dir_norm

# dt_ronda robusto (data + hora)
data_dt = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")
hora_s = df["hora"].fillna("00:00").astype(str).str.strip()
hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
hora_td = pd.to_timedelta(hora_s, errors="coerce")

df["_dia_key"] = data_dt.dt.normalize()
df["_dt_ronda"] = df["_dia_key"] + hora_td

# Dia operacional base (corte às 11h)
df["_dia_operacional"] = (df["_dt_ronda"] - pd.Timedelta(hours=hora_corte_operacional)).dt.normalize()

print("OK: pré-processamento mínimo aplicado.")
df.head(3)

# =========================
# Ajuste de dia operacional por jornada (Portaria E->S) - versão otimizada
# =========================
if usar_jornada_por_portaria:
    cols_min = ["cadastro", "_dt_ronda", "Portaria", "_dir_norm", "999", "_dia_operacional"]
    df_min = df[cols_min].copy()

    df_sorted = (
        df_min.sort_values(["cadastro", "_dt_ronda"], kind="mergesort")
              .reset_index(drop=False)
    )

    orig_idx = df_sorted["index"].to_numpy()
    times    = df_sorted["_dt_ronda"].to_numpy()
    port     = df_sorted["Portaria"].fillna(False).to_numpy(dtype=bool)
    dirn     = df_sorted["_dir_norm"].to_numpy()
    is999    = df_sorted["999"].fillna(False).to_numpy(dtype=bool)

    dia_op = df_sorted["_dia_operacional"].to_numpy(copy=True)
    if not dia_op.flags.writeable:
        dia_op = dia_op.copy()
        dia_op.setflags(write=True)

    tmax = np.timedelta64(int(pd.Timedelta(t_max_jornada_port).value), "ns")
    tjan = np.timedelta64(int(pd.Timedelta(t_jan_999).value), "ns")

    grp_pos = df_sorted.groupby("cadastro", sort=False).indices

    for cad, pos in grp_pos.items():
        t = times[pos]
        valid = ~np.isnat(t)
        if not valid.any():
            continue

        posv  = pos[valid]
        tv    = t[valid]
        portv = port[posv]
        dirv  = dirn[posv]
        is9v  = is999[posv]

        m_ent = portv & (dirv == "E")
        m_sai = portv & (dirv == "S")

        ent = tv[m_ent]
        sai = tv[m_sai]

        if ent.size == 0 or sai.size == 0:
            continue

        i = j = 0
        while i < ent.size and j < sai.size:
            dt_ent = ent[i]
            dt_sai = sai[j]

            if dt_sai < dt_ent:
                j += 1
                continue

            if (dt_sai - dt_ent) > tmax:
                i += 1
                continue

            dia = dt_ent.astype("datetime64[D]").astype("datetime64[ns]")

            l = np.searchsorted(tv, dt_ent, side="left")
            r = np.searchsorted(tv, dt_sai, side="right")
            if r > l:
                dia_op[posv[l:r]] = dia

            l2 = np.searchsorted(tv, dt_ent - tjan, side="left")
            r2 = np.searchsorted(tv, dt_ent, side="left")
            if r2 > l2:
                sl = slice(l2, r2)
                m9 = is9v[sl]
                if m9.any():
                    dia_op[posv[sl][m9]] = dia

            l3 = np.searchsorted(tv, dt_sai, side="right")
            r3 = np.searchsorted(tv, dt_sai + tjan, side="right")
            if r3 > l3:
                sl = slice(l3, r3)
                m9 = is9v[sl]
                if m9.any():
                    dia_op[posv[sl][m9]] = dia

            i += 1
            j += 1

    s = pd.Series(dia_op, index=orig_idx, name="_dia_operacional").reindex(df.index)
    df["_dia_operacional"] = s.to_numpy()

for _v in ("df_min", "df_sorted", "orig_idx", "times", "port", "dirn", "is999", "dia_op", "grp_pos"):
    if _v in globals():
        del globals()[_v]
gc.collect()

print("OK: _dia_operacional ajustado por jornada (Portaria E->S) [otimizado].")

# =========================
# Considera apenas registros com dt válido (sem cópia profunda)
# =========================
mask_dt = df["_dt_ronda"].notna()

COLS_KEEP = [
    "cadastro", "_dia_operacional", "_dt_ronda",
    "_dia_key",  # <<< ALTERADO: necessário para Semana Feriado Prolongado
    "Presencial", "Ponto", "Portaria",
    "_REFEITORIO", "_GRILL", "_is_executivo",
    "_dir_norm",
    "999",
]

for _extra_col in ["nome", "área", "area"]:
    if (_extra_col in df.columns) and (_extra_col not in COLS_KEEP):
        COLS_KEEP.append(_extra_col)

df_valid = df.loc[mask_dt, COLS_KEEP]

cad_minmax = (
    df_valid.groupby("cadastro", sort=False)["_dt_ronda"]
      .agg(min_dt_cad="min", max_dt_cad="max")
      .reset_index()
)

pres_any = df_valid.loc[df_valid["Presencial"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda"]]
day_any = (
    pres_any.groupby(["cadastro", "_dia_operacional"], as_index=False, sort=False)
            .agg(
                first_any_pres=("_dt_ronda", "min"),
                last_any_pres=("_dt_ronda", "max"),
                n_any_pres=("_dt_ronda", "size"),
            )
)
day_any["dur_any_pres"] = day_any["last_any_pres"] - day_any["first_any_pres"]

pres_ponto = df_valid.loc[
    df_valid["Ponto"].eq(True) & df_valid["Presencial"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda"]
]
day_ponto = (
    pres_ponto.groupby(["cadastro", "_dia_operacional"], as_index=False, sort=False)
              .agg(first_ponto_pres=("_dt_ronda", "min"), last_ponto_pres=("_dt_ronda", "max"))
)

day_has_ponto_any = (
    df_valid.loc[df_valid["Ponto"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_PONTO_ANY=True)
)

day_has_port = (
    df_valid.loc[df_valid["Portaria"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_PORTARIA_ANY=True)
)

day_has_ref = (
    df_valid.loc[df_valid["_REFEITORIO"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_REFEITORIO_ANY=True)
)

day_has_grill = (
    df_valid.loc[df_valid["_GRILL"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_GRILL_ANY=True)
)

day_is_exec = (
    df_valid.loc[df_valid["_is_executivo"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(IS_EXECUTIVO_DAY=True)
)

day = day_any.merge(day_ponto, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_ponto_any, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_port, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_ref, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_grill, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_is_exec, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(cad_minmax, on="cadastro", how="left")

for c in ["HAS_PONTO_ANY", "HAS_PORTARIA_ANY", "HAS_REFEITORIO_ANY", "HAS_GRILL_ANY", "IS_EXECUTIVO_DAY"]:
    day[c] = day[c].fillna(False).astype(bool)

day["w_start_needed"] = day["first_any_pres"] - t_jan_port
day["w_end_needed"] = day["last_any_pres"] + t_jan_port
day["SEM_COBERTURA_CONTEXTO"] = False
mask_mm = day["min_dt_cad"].notna() & day["max_dt_cad"].notna()
day.loc[mask_mm, "SEM_COBERTURA_CONTEXTO"] = (day["w_start_needed"] < day["min_dt_cad"]) | (day["w_end_needed"] > day["max_dt_cad"])

port = df_valid.loc[df_valid["Portaria"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]].copy(deep=False)
if port.empty:
    day["ok_entry"] = False
    day["ok_exit"] = False
else:
    port = port.merge(
        day[["cadastro", "_dia_operacional", "w_start_needed", "w_end_needed", "first_any_pres", "last_any_pres"]],
        on=["cadastro", "_dia_operacional"],
        how="inner",
    )

    in_window = (port["_dt_ronda"] >= port["w_start_needed"]) & (port["_dt_ronda"] <= port["w_end_needed"])

    port["ok_entry"] = (
        in_window
        & port["_dir_norm"].eq("E")
        & (port["_dt_ronda"] <= (port["first_any_pres"] + t_tol_ent))
    )

    port["ok_exit"] = (
        in_window
        & port["_dir_norm"].eq("S")
        & (port["_dt_ronda"] >= (port["last_any_pres"] - t_tol_sai))
    )

    ok = port.groupby(["cadastro", "_dia_operacional"], sort=False)[["ok_entry", "ok_exit"]].any().reset_index()
    day = day.merge(ok, on=["cadastro", "_dia_operacional"], how="left")
    day[["ok_entry", "ok_exit"]] = day[["ok_entry", "ok_exit"]].fillna(False)

day["SEM_PORTARIA_COERENTE"] = np.where(
    day["IS_EXECUTIVO_DAY"],
    ~day["HAS_PORTARIA_ANY"],
    ~(day["ok_entry"] | day["ok_exit"]),
)

day.loc[day["SEM_COBERTURA_CONTEXTO"], "SEM_PORTARIA_COERENTE"] = False

first_grill = (
    df_valid.loc[df_valid["_GRILL"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda"]]
      .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
      .min()
      .rename("first_grill")
      .reset_index()
)

ref_rows = (
    df_valid.loc[df_valid["_REFEITORIO"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda"]]
      .merge(first_grill, on=["cadastro", "_dia_operacional"], how="inner")
)

ref_before = (
    ref_rows.loc[ref_rows["_dt_ronda"] < ref_rows["first_grill"], ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_REFEITORIO_BEFORE_GRILL=True)
)

day = day.merge(ref_before, on=["cadastro", "_dia_operacional"], how="left")
day["HAS_REFEITORIO_BEFORE_GRILL"] = day["HAS_REFEITORIO_BEFORE_GRILL"].fillna(False).astype(bool)

day["Irregularidade - Não entrou no refeitório, mas passou no grill"] = (
    day["HAS_GRILL_ANY"].fillna(False).astype(bool)
    & (~day["HAS_REFEITORIO_BEFORE_GRILL"])
)

print("OK: day construído.")
day.head(3)

# =========================
# Helpers (igual ao seu)
# =========================
def slugify(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return text.lower()

def _pick_col(df_: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df_.columns:
            return c
    return None


def gerar_relatorio_pdf(df_irregularidade: pd.DataFrame, titulo: str, pdf_path: Path) -> None:
    """
    Relatório enxuto e analítico.
    - Capa com data de geração + métricas + percentuais (Top 1/3/10 e recência)
    - Remove tabelas/listagens
    - Gerais: tendência mensal, DOW, e 3 gráficos separados (áreas, plantas, áreas|planta)
    - Top 3: rankings (quantidade, percentual, atualidade) + 4 gráficos por pessoa
      (mensal, DOW, ocorrências diárias, vazio)
    - Caso especial (999 trajeto): gráfico de "concentração" (Antes/Depois/Ambos) por pessoa com 3 bolinhas
    - Otimizações: trabalha com subconjuntos de colunas e evita cópias profundas grandes
    """
    import datetime as _dt
    import numpy as np

    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 150,
        "font.size": 12,
        "axes.titlesize": 18,
        "axes.labelsize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
    })

    def _wrap(s: str, width: int = 44) -> str:
        s = "" if s is None else str(s)
        s = s.strip()
        if len(s) <= width:
            return s
        words = s.split()
        out, line = [], ""
        for w in words:
            if len(line) + len(w) + 1 <= width:
                line = (line + " " + w).strip()
            else:
                if line:
                    out.append(line)
                line = w
        if line:
            out.append(line)
        if len(out) <= 4:
            return "\n".join(out)
        return "\n".join(out[:4]) + "…"

    def _as_series(df_: pd.DataFrame, col: str) -> pd.Series:
        x = df_.loc[:, col]
        if isinstance(x, pd.DataFrame):
            x = x.iloc[:, 0]
        return x

    def _pick_col_local(df_: pd.DataFrame, candidates: list[str]) -> str | None:
        for c in candidates:
            if c in df_.columns:
                return c
        return None

    def _annotate_barh(ax, values, total_base: int | None = None, fs: int = 9):
        # values: array aligned com barras (na ordem do ax)
        for i, v in enumerate(values):
            v = float(v)
            if v <= 0:
                continue
            if total_base and total_base > 0:
                ax.text(v, i, f"  {int(v)} ({(v/total_base):.1%})", va="center", fontsize=fs)
            else:
                ax.text(v, i, f"  {int(v)}", va="center", fontsize=fs)

    def _annotate_bar(ax, values, total_base: int | None = None, fs: int = 9):
        for i, v in enumerate(values):
            v = float(v)
            if v <= 0:
                continue
            if total_base and total_base > 0:
                ax.text(i, v, f"{int(v)}\n({(v/total_base):.1%})", ha="center", va="bottom", fontsize=fs)
            else:
                ax.text(i, v, f"{int(v)}", ha="center", va="bottom", fontsize=fs)

    gen_date = _dt.date.today().strftime("%Y-%m-%d")

    # Caso vazio
    if df_irregularidade is None or df_irregularidade.empty:
        with PdfPages(pdf_path) as pdf:
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)
            ax.axis("off")
            ax.text(0.02, 0.96, titulo, fontsize=24, fontweight="bold", va="top")
            ax.text(0.02, 0.91, f"Gerado em: {gen_date}", fontsize=12, va="top")
            ax.text(0.02, 0.84, "Sem linhas para gerar o relatório.", fontsize=14, va="top")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
        return

    # remove colunas duplicadas sem deep copy
    if getattr(df_irregularidade, "columns", None) is not None:
        df_irregularidade = df_irregularidade.loc[:, ~df_irregularidade.columns.duplicated()]

    # Colunas
    col_cad = _pick_col_local(df_irregularidade, ["cadastro"])
    col_nome = _pick_col_local(df_irregularidade, ["nome"])
    col_areas = "área" if "área" in df_irregularidade.columns else None
    col_area_fallback = _pick_col_local(df_irregularidade, ["area"])  # fallback real
    col_planta = "planta" if "planta" in df_irregularidade.columns else None

    # Determina _dia_operacional
    if "_dia_operacional" in df_irregularidade.columns:
        dia_op = pd.to_datetime(_as_series(df_irregularidade, "_dia_operacional"), errors="coerce")
    elif "dia_operacional" in df_irregularidade.columns:
        dia_op = pd.to_datetime(_as_series(df_irregularidade, "dia_operacional"), errors="coerce")
    else:
        data_dt = (
            pd.to_datetime(_as_series(df_irregularidade, "data"), dayfirst=True, errors="coerce")
            if "data" in df_irregularidade.columns
            else pd.Series(pd.NaT, index=df_irregularidade.index)
        )
        if "hora" not in df_irregularidade.columns:
            dia_op = data_dt.dt.normalize()
        else:
            hora_s = _as_series(df_irregularidade, "hora").fillna("00:00").astype(str).str.strip()
            hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
            hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
            hora_td = pd.to_timedelta(hora_s, errors="coerce")
            dia_op = (data_dt.dt.normalize() + hora_td - pd.Timedelta(hours=11)).dt.normalize()

    # subset mínimo para métricas gerais (evita carregar tudo)
    cols_keep = ["_dia_operacional"]
    if col_cad: cols_keep.append(col_cad)
    if col_nome: cols_keep.append(col_nome)
    if col_areas: cols_keep.append(col_areas)
    elif col_area_fallback: cols_keep.append(col_area_fallback)
    if col_planta: cols_keep.append(col_planta)

    dfv = df_irregularidade.loc[:, [c for c in cols_keep if c != "_dia_operacional"]].copy()
    dfv["_dia_operacional"] = pd.to_datetime(dia_op, errors="coerce")

    if col_cad and (col_cad in dfv.columns):
        dfv[col_cad] = dfv[col_cad].astype(str).str.strip()
        dfv = dfv[dfv[col_cad].ne("")]

    key_cols = [col_cad, "_dia_operacional"] if (col_cad and col_cad in dfv.columns) else ["_dia_operacional"]

    # ocorrência = pessoa-dia
    df_occ = (
        dfv.dropna(subset=["_dia_operacional"])
           .drop_duplicates(subset=key_cols, keep="first")
    )

    if df_occ.empty:
        with PdfPages(pdf_path) as pdf:
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)
            ax.axis("off")
            ax.text(0.02, 0.96, titulo, fontsize=24, fontweight="bold", va="top")
            ax.text(0.02, 0.91, f"Gerado em: {gen_date}", fontsize=12, va="top")
            ax.text(0.02, 0.84, "Sem ocorrências após normalização de _dia_operacional.", fontsize=14, va="top")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
        return

    total = int(len(df_occ))
    n_pessoas = int(df_occ[col_cad].nunique()) if (col_cad and col_cad in df_occ.columns) else 0
    n_dias = int(df_occ["_dia_operacional"].nunique())
    dt_min = df_occ["_dia_operacional"].min()
    dt_max = df_occ["_dia_operacional"].max()

    # nome_map por cadastro (mais frequente)
    nome_map = {}
    if col_cad and col_nome and (col_cad in dfv.columns) and (col_nome in dfv.columns):
        tmp = dfv[[col_cad, col_nome]].copy()
        tmp[col_nome] = tmp[col_nome].fillna("").astype(str).str.strip()
        tmp = tmp[tmp[col_nome].ne("")]
        if not tmp.empty:
            cnt = (
                tmp.groupby([col_cad, col_nome], sort=False)
                   .size()
                   .reset_index(name="n")
                   .sort_values([col_cad, "n"], ascending=[True, False], kind="mergesort")
            )
            nome_map = cnt.drop_duplicates(subset=[col_cad], keep="first").set_index(col_cad)[col_nome].to_dict()

    # labels de pessoa (cadastro + nome)
    if col_cad and (col_cad in df_occ.columns):
        cad_series = df_occ[col_cad].astype(str).str.strip()
        pessoa_label = cad_series.map(lambda c: f"{c} - {nome_map.get(c,'')}".strip(" -")) if nome_map else cad_series
        top_pessoas = pessoa_label.value_counts()
    else:
        top_pessoas = pd.Series(dtype=int)

    # Top 1/3/10
    top1_n = int(top_pessoas.head(1).sum()) if (not top_pessoas.empty) else 0
    top3_n = int(top_pessoas.head(3).sum()) if (not top_pessoas.empty) else 0
    top10_n = int(top_pessoas.head(10).sum()) if (not top_pessoas.empty) else 0
    top1_pct = (top1_n / total) if total else 0.0
    top3_pct = (top3_n / total) if total else 0.0
    top10_pct = (top10_n / total) if total else 0.0

    # Mensal / DOW (gerais)
    mes = df_occ["_dia_operacional"].dt.to_period("M").dt.to_timestamp()
    mensal = mes.value_counts().sort_index()

    dow_map = {0: "Seg", 1: "Ter", 2: "Qua", 3: "Qui", 4: "Sex", 5: "Sáb", 6: "Dom"}
    dow = (
        df_occ["_dia_operacional"].dt.dayofweek.map(dow_map)
            .value_counts()
            .reindex(["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"])
            .fillna(0).astype(int)
    )

    # Recência (atuais): últimos 30 dias (ou menor se período curto)
    dt_max_norm = pd.to_datetime(dt_max).normalize()
    window_days = 30
    if pd.notna(dt_min) and pd.notna(dt_max):
        span = (pd.to_datetime(dt_max) - pd.to_datetime(dt_min)).days
        if span < 45:
            window_days = 14
        if span < 21:
            window_days = 7

    dt_cut = dt_max_norm - pd.Timedelta(days=window_days)
    recent = df_occ.loc[df_occ["_dia_operacional"] >= dt_cut]
    total_recent = int(len(recent))
    if col_cad and (col_cad in df_occ.columns) and total_recent > 0:
        cad_recent = recent[col_cad].astype(str).str.strip()
        label_recent = cad_recent.map(lambda c: f"{c} - {nome_map.get(c,'')}".strip(" -")) if nome_map else cad_recent
        top_recent = label_recent.value_counts()
    else:
        top_recent = pd.Series(dtype=int)

    # Top 3 listas
    top3_qty = top_pessoas.head(3) if not top_pessoas.empty else pd.Series(dtype=int)
    top3_pct_ser = (top3_qty / total) if (total and not top3_qty.empty) else pd.Series(dtype=float)

    top3_recent = top_recent.head(3) if not top_recent.empty else pd.Series(dtype=int)
    top3_recent_pct = (top3_recent / total_recent) if (total_recent and not top3_recent.empty) else pd.Series(dtype=float)

    # ---------- Funções de representatividade (áreas / plantas / combo) por pessoa-dia
    def _rep_value_per_occ(df_in: pd.DataFrame, key_cols2: list[str], val_col: str) -> pd.Series:
        if (val_col is None) or (val_col not in df_in.columns):
            return pd.Series(dtype=str)
        tmp = df_in[key_cols2 + [val_col]].copy()
        tmp[val_col] = tmp[val_col].fillna("").astype(str).str.strip()
        tmp = tmp[tmp[val_col].ne("")]
        if tmp.empty:
            return pd.Series(dtype=str)
        cnt = (
            tmp.groupby(key_cols2 + [val_col], sort=False)
               .size()
               .reset_index(name="n")
               .sort_values(key_cols2 + ["n"], ascending=[True]*len(key_cols2) + [False], kind="mergesort")
        )
        rep = cnt.drop_duplicates(subset=key_cols2, keep="first").set_index(key_cols2)[val_col]
        return rep

    # área: prefer 'área', senão 'area'
    area_col_use = col_areas if col_areas else col_area_fallback
    area_rep = pd.Series(dtype=str)
    if area_col_use and col_cad and (col_cad in dfv.columns):
        area_rep = _rep_value_per_occ(dfv, [col_cad, "_dia_operacional"], area_col_use)

    # planta
    planta_rep = pd.Series(dtype=str)
    if col_planta and col_cad and (col_cad in dfv.columns):
        planta_rep = _rep_value_per_occ(dfv, [col_cad, "_dia_operacional"], col_planta)

    # combo
    combo_rep = pd.Series(dtype=str)
    if (not area_rep.empty) and (not planta_rep.empty):
        # constrói combo por ocorrência (índice MultiIndex)
        idx = area_rep.index.intersection(planta_rep.index)
        if len(idx) > 0:
            combo_rep = (area_rep.loc[idx].astype(str) + " | " + planta_rep.loc[idx].astype(str))

    top_areas = area_rep.value_counts() if not area_rep.empty else pd.Series(dtype=int)
    top_plantas = planta_rep.value_counts() if not planta_rep.empty else pd.Series(dtype=int)
    top_combo = combo_rep.value_counts() if not combo_rep.empty else pd.Series(dtype=int)

    # ---------- Top 3 pessoas (para painéis): por quantidade total
    top3_labels = list(top3_qty.index) if not top3_qty.empty else []
    top3_cads = [str(lbl).split(" - ", 1)[0].strip() for lbl in top3_labels]

    # ---------- PDF
    with PdfPages(pdf_path) as pdf:
        # ---------------- CAPA
        fig = plt.figure(figsize=(11.69, 8.27))
        ax = fig.add_subplot(111)
        ax.axis("off")

        ax.text(0.02, 0.96, titulo, fontsize=24, fontweight="bold", va="top")
        ax.text(0.02, 0.91, f"Gerado em: {gen_date}", fontsize=12, va="top")

        ax.text(0.02, 0.84, f"Ocorrências (pessoa-dia): {total:,}".replace(",", "."), fontsize=16, va="top")
        if n_pessoas:
            ax.text(0.02, 0.80, f"Pessoas (cadastro): {n_pessoas:,}".replace(",", "."), fontsize=16, va="top")
        ax.text(0.02, 0.76, f"Dias com ocorrência: {n_dias:,}".replace(",", "."), fontsize=16, va="top")

        if pd.notna(dt_min) and pd.notna(dt_max):
            ax.text(0.02, 0.72, f"Período: {pd.to_datetime(dt_min).date()} → {pd.to_datetime(dt_max).date()}",
                    fontsize=14, va="top")

        y = 0.64
        if total and not top_pessoas.empty:
            ax.text(0.02, y, f"Concentração Top 1: {top1_n:,} ({top1_pct:.1%})".replace(",", "."),
                    fontsize=14, va="top")
            y -= 0.05
            ax.text(0.02, y, f"Concentração Top 3: {top3_n:,} ({top3_pct:.1%})".replace(",", "."),
                    fontsize=14, va="top")
            y -= 0.05
            ax.text(0.02, y, f"Concentração Top 10: {top10_n:,} ({top10_pct:.1%})".replace(",", "."),
                    fontsize=13, va="top")
            y -= 0.07

            ax.text(0.02, y, "Top 3 piores (quantidade e % do total):", fontsize=14, fontweight="bold", va="top")
            y -= 0.04
            for i, (k, v) in enumerate(top3_qty.items(), start=1):
                pct = (int(v) / total) if total else 0.0
                ax.text(0.03, y, f"{i}. {_wrap(k, 60)} — {int(v)} ({pct:.1%})", fontsize=12.5, va="top")
                y -= 0.04

            y -= 0.02
            if total_recent > 0 and not top3_recent.empty:
                ax.text(0.02, y, f"Top 3 atuais (últimos {window_days} dias) — base: {total_recent} ({total_recent/total:.1%} do total)",
                        fontsize=13, fontweight="bold", va="top")
                y -= 0.04
                for i, (k, v) in enumerate(top3_recent.items(), start=1):
                    pct_r = (int(v) / total_recent) if total_recent else 0.0
                    ax.text(0.03, y, f"{i}. {_wrap(k, 60)} — {int(v)} ({pct_r:.1%} da janela)",
                            fontsize=11.8, va="top")
                    y -= 0.035

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # ---------------- GERAIS: mensal + DOW
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11.69, 8.27), gridspec_kw={"height_ratios": [2.1, 1.3]})

        if mensal.empty:
            ax1.axis("off")
            ax1.text(0.02, 0.95, "Tendência mensal", fontsize=18, fontweight="bold", va="top")
            ax1.text(0.02, 0.80, "Sem datas válidas para agregação mensal.", fontsize=13, va="top")
        else:
            ax1.plot(mensal.index, mensal.values, marker="o", linewidth=2.3)
            ax1.fill_between(mensal.index, mensal.values, alpha=0.15)
            ax1.set_title("Tendência mensal de ocorrências (pessoa-dia)")
            ax1.set_xlabel("Mês")
            ax1.set_ylabel("Ocorrências")
            ax1.grid(alpha=0.30)
            ax1.tick_params(axis="x", rotation=35)

            # anota pico com %
            peak_v = int(mensal.max())
            if peak_v > 0 and total > 0:
                peak_i = mensal.idxmax()
                ax1.annotate(f"Pico: {peak_v} ({peak_v/total:.1%})",
                             xy=(peak_i, peak_v),
                             xytext=(0, 10),
                             textcoords="offset points")

        ax2.bar(dow.index, dow.values)
        ax2.set_title("Ocorrências por dia da semana (pessoa-dia)")
        ax2.set_xlabel("Dia")
        ax2.set_ylabel("Ocorrências")
        ax2.grid(axis="y", alpha=0.30)
        _annotate_bar(ax2, dow.values, total_base=total, fs=9)

        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # ---------------- GERAIS: Áreas / Plantas / (Área | Planta)
        fig, axs = plt.subplots(3, 1, figsize=(11.69, 8.27), gridspec_kw={"height_ratios": [1.2, 1.2, 1.6]})
        axa, axp, axc = axs

        # Áreas
        if top_areas.empty:
            axa.axis("off")
            axa.text(0.02, 0.92, "Top áreas (por dias com ocorrência)", fontsize=15, fontweight="bold", va="top")
            axa.text(0.02, 0.74, "Sem coluna 'área'/'area' ou sem valores.", fontsize=12, va="top")
        else:
            s = top_areas.head(12).sort_values()
            axa.barh([_wrap(i, 44) for i in s.index], s.values)
            axa.set_title("Top áreas (por dias com ocorrência)")
            axa.set_xlabel("Ocorrências (pessoa-dia)")
            axa.grid(axis="x", alpha=0.25)
            _annotate_barh(axa, s.values, total_base=total, fs=9)

        # Plantas
        if top_plantas.empty:
            axp.axis("off")
            axp.text(0.02, 0.92, "Top plantas (por dias com ocorrência)", fontsize=15, fontweight="bold", va="top")
            axp.text(0.02, 0.74, "Sem coluna 'planta' ou sem valores.", fontsize=12, va="top")
        else:
            s = top_plantas.head(12).sort_values()
            axp.barh([_wrap(i, 44) for i in s.index], s.values)
            axp.set_title("Top plantas (por dias com ocorrência)")
            axp.set_xlabel("Ocorrências (pessoa-dia)")
            axp.grid(axis="x", alpha=0.25)
            _annotate_barh(axp, s.values, total_base=total, fs=9)

        # Área | Planta
        if top_combo.empty:
            axc.axis("off")
            axc.text(0.02, 0.92, "Top (Área | Planta) (por dias com ocorrência)", fontsize=15, fontweight="bold", va="top")
            axc.text(0.02, 0.74, "Sem dados suficientes para combinar área e planta.", fontsize=12, va="top")
        else:
            s = top_combo.head(15).sort_values()
            axc.barh([_wrap(i, 56) for i in s.index], s.values)
            axc.set_title("Top (Área | Planta) (por dias com ocorrência)")
            axc.set_xlabel("Ocorrências (pessoa-dia)")
            axc.grid(axis="x", alpha=0.25)
            _annotate_barh(axc, s.values, total_base=total, fs=8)

        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # ---------------- TOP 3: Quantidade / Percentual / Atualidade
        if not top_pessoas.empty:
            fig, axs = plt.subplots(3, 1, figsize=(11.69, 8.27), gridspec_kw={"height_ratios": [1.25, 1.1, 1.25]})
            axq, axpct, axr = axs

            # Quantidade
            s = top3_qty.iloc[::-1]  # invert para ficar maior no topo no barh
            axq.barh([_wrap(x, 60) for x in s.index], s.values)
            axq.set_title("Top 3 piores — por quantidade (pessoa-dia)")
            axq.set_xlabel("Ocorrências")
            axq.grid(axis="x", alpha=0.25)
            _annotate_barh(axq, s.values, total_base=total, fs=10)

            # Percentual (mesmos 3, mas eixo em %)
            sp = top3_pct_ser.loc[top3_qty.index].iloc[::-1]
            axpct.barh([_wrap(x, 60) for x in sp.index], sp.values)
            axpct.set_title("Top 3 piores — por percentual do total")
            axpct.set_xlabel("% do total")
            axpct.grid(axis="x", alpha=0.25)
            for i, v in enumerate(sp.values):
                axpct.text(v, i, f"  {v:.1%}", va="center", fontsize=10)

            # Atualidade
            if total_recent <= 0 or top3_recent.empty:
                axr.axis("off")
                axr.text(0.02, 0.92, f"Top 3 atuais (últimos {window_days} dias)", fontsize=15, fontweight="bold", va="top")
                axr.text(0.02, 0.74, "Sem dados suficientes na janela recente.", fontsize=12, va="top")
            else:
                sr = top3_recent.iloc[::-1]
                axr.barh([_wrap(x, 60) for x in sr.index], sr.values)
                axr.set_title(f"Top 3 atuais — últimos {window_days} dias")
                axr.set_xlabel("Ocorrências na janela")
                axr.grid(axis="x", alpha=0.25)
                # anota % na janela e % no total
                for i, v in enumerate(sr.values):
                    pr = (int(v) / total_recent) if total_recent else 0.0
                    # % do total geral também (para visão analítica)
                    # total por pessoa no geral (se existir no top_pessoas)
                    k = sr.index[i]
                    vg = int(top_pessoas.get(k, 0))
                    p_total = (vg / total) if total else 0.0
                    axr.text(v, i, f"  {int(v)} ({pr:.1%} da janela) | total: {vg} ({p_total:.1%})",
                             va="center", fontsize=9)

            fig.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

        # ---------------- TOP 3: 4 gráficos por pessoa (mensal / DOW / diário / vazio)
        if top3_cads and col_cad and (col_cad in df_occ.columns):
            # range mensal completo
            mes_min = pd.to_datetime(dt_min).to_period("M").to_timestamp()
            mes_max = pd.to_datetime(dt_max).to_period("M").to_timestamp()
            idx_all = pd.date_range(mes_min, mes_max, freq="MS")

            for cad in top3_cads[:3]:
                occ_p = df_occ.loc[df_occ[col_cad].astype(str).eq(str(cad)), ["_dia_operacional"]]
                if occ_p.empty:
                    continue

                total_p = int(len(occ_p))
                pct_p = (total_p / total) if total else 0.0
                nome = nome_map.get(str(cad), "") if nome_map else ""
                label = f"{cad} - {nome}".strip(" -")

                # mensal
                ser_m = (
                    occ_p.assign(_mes=occ_p["_dia_operacional"].dt.to_period("M").dt.to_timestamp())
                         .groupby("_mes", sort=False).size()
                         .reindex(idx_all, fill_value=0)
                )

                # DOW
                dow_p = (
                    occ_p["_dia_operacional"].dt.dayofweek.map(dow_map)
                        .value_counts()
                        .reindex(["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"])
                        .fillna(0).astype(int)
                )

                # diário (últimos 60 dias para legibilidade)
                occ_p_day = (
                    occ_p.groupby(occ_p["_dia_operacional"].dt.normalize(), sort=False).size().rename("n").sort_index()
                )
                dt_cut_p = pd.to_datetime(dt_max).normalize() - pd.Timedelta(days=60)
                occ_p_day_60 = occ_p_day.loc[occ_p_day.index >= dt_cut_p]
                total_p_60 = int(occ_p_day_60.sum()) if not occ_p_day_60.empty else 0

                fig, axs = plt.subplots(2, 2, figsize=(11.69, 8.27))
                (ax1, ax2), (ax3, ax4) = axs

                fig.suptitle(
                    f"Top 3 — {_wrap(label, 70)}  |  Total: {total_p} ({pct_p:.1%} do total)",
                    fontsize=16,
                    fontweight="bold",
                )

                # (1) mensal
                ax1.plot(ser_m.index, ser_m.values, marker="o", linewidth=2.0)
                ax1.fill_between(ser_m.index, ser_m.values, alpha=0.12)
                ax1.set_title("Evolução mensal (pessoa-dia)")
                ax1.set_xlabel("Mês")
                ax1.set_ylabel("Ocorrências")
                ax1.grid(alpha=0.25)
                ax1.tick_params(axis="x", rotation=35)
                if len(ser_m) > 0:
                    peak_v = int(ser_m.max())
                    if peak_v > 0 and total_p > 0:
                        peak_i = ser_m.idxmax()
                        ax1.annotate(f"Pico: {peak_v} ({peak_v/total_p:.1%})",
                                     xy=(peak_i, peak_v),
                                     xytext=(0, 10),
                                     textcoords="offset points")

                # (2) DOW
                ax2.bar(dow_p.index, dow_p.values)
                ax2.set_title("Ocorrências por dia da semana")
                ax2.set_xlabel("Dia")
                ax2.set_ylabel("Ocorrências")
                ax2.grid(axis="y", alpha=0.25)
                _annotate_bar(ax2, dow_p.values, total_base=total_p, fs=8)

                # (3) diário
                if occ_p_day_60.empty:
                    ax3.axis("off")
                    ax3.text(0.02, 0.92, "Ocorrências diárias (últimos 60 dias)", fontsize=14, fontweight="bold", va="top")
                    ax3.text(0.02, 0.78, "Sem ocorrências na janela recente.", fontsize=11, va="top")
                else:
                    ax3.plot(occ_p_day_60.index, occ_p_day_60.values, marker="o", linewidth=1.8)
                    ax3.fill_between(occ_p_day_60.index, occ_p_day_60.values, alpha=0.12)
                    ax3.set_title("Ocorrências diárias (últimos 60 dias)")
                    ax3.set_xlabel("Dia")
                    ax3.set_ylabel("Ocorrências")
                    ax3.grid(alpha=0.25)
                    ax3.tick_params(axis="x", rotation=35)
                    # anota máximo na janela
                    mx = int(occ_p_day_60.max())
                    if mx > 0 and total_p_60 > 0:
                        mi = occ_p_day_60.idxmax()
                        ax3.annotate(f"Máx: {mx} ({mx/total_p_60:.1%})",
                                     xy=(mi, mx),
                                     xytext=(0, 10),
                                     textcoords="offset points")

                # (4) vazio*
                ax4.axis("off")
                ax4.text(0.02, 0.92, "Reservado", fontsize=14, fontweight="bold", va="top")
                ax4.text(0.02, 0.78, "—", fontsize=12, va="top")

                fig.tight_layout(rect=[0, 0, 1, 0.94])
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

        # ---------------- Caso especial: 999 trajeto (3 bolinhas por pessoa)
        # Critério de ativação: "999" e "trajeto" no título ou nome do arquivo
        titulo_l = str(titulo).lower()
        if ("999" in titulo_l) and ("trajeto" in titulo_l or "999" in str(pdf_path).lower()):
            # precisa de colunas básicas para reconstruir dt e identificar portaria/999
            if all(c in df_irregularidade.columns for c in ["cadastro", "data", "hora", "dir", "planta", "descricao coletor"]):
                dfx = df_irregularidade[["cadastro", "data", "hora", "dir", "planta", "descricao coletor"]].copy()
                dfx["cadastro"] = dfx["cadastro"].astype(str).str.strip()

                # dt
                data_dt = pd.to_datetime(dfx["data"], dayfirst=True, errors="coerce")
                hora_s = dfx["hora"].fillna("00:00").astype(str).str.strip()
                hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
                hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
                hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
                hora_td = pd.to_timedelta(hora_s, errors="coerce")
                dfx["_dt"] = data_dt.dt.normalize() + hora_td
                dfx["_dia_key"] = data_dt.dt.normalize()

                # portaria (usa regex_portaria global se existir; senão fallback)
                _re_port = globals().get("regex_portaria", r"PORTARIA|PORT")
                desc = dfx["descricao coletor"].fillna("").astype(str)
                dfx["_is_port"] = desc.str.contains(_re_port, case=False, na=False, regex=True)

                # dir norm
                dirn = dfx["dir"].fillna("").astype(str).str.strip().str.upper()
                dirn = dirn.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
                dfx["_is_E"] = dirn.str.startswith("E")
                dfx["_is_S"] = dirn.str.startswith("S")

                # 999
                planta_num = pd.to_numeric(dfx["planta"], errors="coerce")
                dfx["_is_999"] = planta_num.eq(999)

                # portaria por pessoa-dia (dia_key)
                port = dfx.loc[dfx["_is_port"] & dfx["_dt"].notna(), ["cadastro", "_dia_key", "_dt", "_is_E", "_is_S"]]
                if not port.empty:
                    ent = (
                        port.loc[port["_is_E"]]
                            .groupby(["cadastro", "_dia_key"], sort=False)["_dt"].min()
                            .rename("t_ent")
                            .reset_index()
                    )
                    sai = (
                        port.loc[port["_is_S"]]
                            .groupby(["cadastro", "_dia_key"], sort=False)["_dt"].max()
                            .rename("t_sai")
                            .reset_index()
                    )
                    pe = ent.merge(sai, on=["cadastro", "_dia_key"], how="outer")

                    # 999 eventos por pessoa-dia
                    ev = dfx.loc[dfx["_is_999"] & dfx["_dt"].notna(), ["cadastro", "_dia_key", "_dt"]]
                    if (not ev.empty) and (not pe.empty):
                        ev = ev.merge(pe, on=["cadastro", "_dia_key"], how="left")
                        # classifica por evento
                        ev["_before"] = ev["t_ent"].notna() & (ev["_dt"] < ev["t_ent"])
                        ev["_after"]  = ev["t_sai"].notna() & (ev["_dt"] > ev["t_sai"])

                        day_flags = (
                            ev.groupby(["cadastro", "_dia_key"], sort=False)[["_before", "_after"]]
                              .any()
                              .reset_index()
                        )
                        day_flags["cat"] = np.select(
                            [
                                day_flags["_before"] & day_flags["_after"],
                                day_flags["_before"] & (~day_flags["_after"]),
                                (~day_flags["_before"]) & day_flags["_after"],
                            ],
                            ["Ambos", "Antes", "Depois"],
                            default="(sem class.)",
                        )

                        # agrega por pessoa
                        # considera apenas categorias esperadas
                        day_flags = day_flags[day_flags["cat"].isin(["Antes", "Depois", "Ambos"])]
                        if not day_flags.empty:
                            agg = (
                                day_flags.groupby(["cadastro", "cat"], sort=False)
                                         .size()
                                         .rename("n_dias")
                                         .reset_index()
                            )
                            tot = agg.groupby("cadastro", sort=False)["n_dias"].sum().rename("n_total").reset_index()
                            agg = agg.merge(tot, on="cadastro", how="left")
                            agg["pct"] = agg["n_dias"] / agg["n_total"].where(agg["n_total"] > 0, np.nan)

                            # top pessoas por total de dias com 999 classificado (até 10)
                            top_cads = (
                                tot.sort_values("n_total", ascending=False)
                                   .head(10)["cadastro"].astype(str).tolist()
                            )

                            # prepara matriz (cadastro x cat)
                            cats = ["Antes", "Depois", "Ambos"]
                            mat = (
                                agg.loc[agg["cadastro"].isin(top_cads)]
                                   .pivot_table(index="cadastro", columns="cat", values="pct", aggfunc="first")
                                   .reindex(columns=cats)
                                   .fillna(0.0)
                            )
                            # labels com nome
                            if nome_map:
                                y_labels = [f"{c} - {nome_map.get(str(c),'')}".strip(" -") for c in mat.index]
                            else:
                                y_labels = list(mat.index)

                            # plot "3 bolinhas"
                            fig = plt.figure(figsize=(11.69, 8.27))
                            ax = fig.add_subplot(111)
                            ax.set_title("999 como trajeto — concentração por período (Top 10 pessoas)")
                            ax.set_xlabel("Período do dia relativo à Portaria")
                            ax.set_yticks(range(len(y_labels)))
                            ax.set_yticklabels([_wrap(y, 60) for y in y_labels])
                            ax.set_xticks([0, 1, 2])
                            ax.set_xticklabels(["Antes", "Depois", "Ambos"])
                            ax.set_xlim(-0.5, 2.5)
                            ax.grid(axis="x", alpha=0.25)

                            # tamanho proporcional ao pct (mínimo visível)
                            for yi, cad in enumerate(mat.index):
                                for xi, cat in enumerate(cats):
                                    p = float(mat.loc[cad, cat])
                                    s = 40 + 900 * p  # escala
                                    ax.scatter([xi], [yi], s=s)
                                    # anota p% (somente se relevante)
                                    if p >= 0.15:
                                        ax.text(xi + 0.06, yi, f"{p:.0%}", va="center", fontsize=9)

                            pdf.savefig(fig, bbox_inches="tight")
                            plt.close(fig)
            # se faltar coluna, simplesmente não cria página especial

def row_ids_from_keys(df_min: pd.DataFrame, keys: pd.DataFrame) -> np.ndarray:
    if keys is None or keys.empty:
        return np.array([], dtype=np.int64)

    keys2 = keys.loc[:, ["cadastro", "_dia_operacional"]].drop_duplicates().copy()
    keys2["_dia_operacional"] = pd.to_datetime(keys2["_dia_operacional"], errors="coerce")

    if ("_ROWID_CACHE_HASH" not in globals()) or (len(globals()["_ROWID_CACHE_HASH"]) != len(df_min)):
        globals()["_ROWID_CACHE_ROWID"] = df_min["_row_id"].to_numpy(dtype=np.int64, copy=False)
        globals()["_ROWID_CACHE_HASH"] = pd.util.hash_pandas_object(
            df_min[["cadastro", "_dia_operacional"]], index=False
        ).to_numpy(dtype=np.uint64, copy=False)
        globals()["_ROWID_CACHE_HASH_INDEX"] = pd.Index(globals()["_ROWID_CACHE_HASH"])

    key_hash = pd.util.hash_pandas_object(
        keys2[["cadastro", "_dia_operacional"]], index=False
    ).to_numpy(dtype=np.uint64, copy=False)

    if key_hash.size == 0:
        return np.array([], dtype=np.int64)

    mask = globals()["_ROWID_CACHE_HASH_INDEX"].isin(key_hash)
    return globals()["_ROWID_CACHE_ROWID"][mask].astype(np.int64, copy=True)


n_total = len(df_min_raw)
dia_op_arr = np.full(n_total, np.datetime64("NaT"), dtype="datetime64[ns]")
valid_map = df.loc[df["_dt_ronda"].notna(), ["_row_id", "_dia_operacional"]]
dia_op_arr[valid_map["_row_id"].to_numpy()] = valid_map["_dia_operacional"].to_numpy()


def materializar_e_exportar_por_rowid(row_ids: np.ndarray, irregularidade_col: str, titulo: str) -> pd.DataFrame:
    slug = slugify(irregularidade_col)

    parquet_path = DFS_DIR / f"{slug}.parquet"
    pdf_path = PDF_DIR / f"{slug}.pdf"

    params_export = {
        "param_janela_portaria_min": globals().get("janela_portaria_min"),
        "param_tolerancia_entrada_min": globals().get("tolerancia_entrada_min"),
        "param_tolerancia_saida_min": globals().get("tolerancia_saida_min"),
        "param_janela_999_horas": globals().get("janela_999_horas"),
        "param_hora_corte_operacional": globals().get("hora_corte_operacional"),
        "param_descanso_min_horas": globals().get("descanso_min_horas"),
        "param_jornada_min_horas": globals().get("jornada_min_horas"),
        "param_jornada_min_minutos": globals().get("jornada_min_minutos"),
        "param_usar_jornada_por_portaria": globals().get("usar_jornada_por_portaria"),
        "param_max_jornada_portaria_horas": globals().get("max_jornada_portaria_horas"),
    }

    if row_ids.size == 0:
        df_out = df_min_raw.head(0).copy()
        df_out["dia_operacional"] = pd.to_datetime([], errors="coerce")
        df_out[irregularidade_col] = False
        for _k, _v in params_export.items():
            df_out[_k] = _v
        df_out.to_parquet(parquet_path, index=False)
        gerar_relatorio_pdf(df_out, titulo=titulo, pdf_path=pdf_path)
        gc.collect()
        print("Parquet:", parquet_path)
        print("PDF   :", pdf_path)
        return df_out

    row_ids = row_ids[(row_ids >= 0) & (row_ids < len(df_min_raw))]

    MAX_ROWS_IN_MEMORY = 2_000_000
    REPORT_SAMPLE_ROWS = 2_000

    if row_ids.size > MAX_ROWS_IN_MEMORY:
        import pyarrow as pa
        import pyarrow.parquet as pq

        row_ids = np.sort(row_ids.astype(np.int64, copy=False))
        chunk_size = 250_000

        if parquet_path.exists():
            parquet_path.unlink()

        writer = None
        report_parts = []
        report_count = 0

        for i in range(0, row_ids.size, chunk_size):
            ids = row_ids[i:i + chunk_size]
            part = df_min_raw.iloc[ids].copy()
            part["dia_operacional"] = pd.to_datetime(dia_op_arr[ids], errors="coerce")
            part[irregularidade_col] = True
            for _k, _v in params_export.items():
                part[_k] = _v
            part.drop(columns=["_row_id"], inplace=True, errors="ignore")

            table = pa.Table.from_pandas(part, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(str(parquet_path), table.schema, compression="snappy")
            writer.write_table(table)

            if report_count < REPORT_SAMPLE_ROWS:
                take = min(REPORT_SAMPLE_ROWS - report_count, len(part))
                report_parts.append(part.head(take).copy())
                report_count += take

            del ids, part, table
            gc.collect()

        if writer is not None:
            writer.close()

        if report_parts:
            df_report = pd.concat(report_parts, ignore_index=True)
        else:
            df_report = df_min_raw.head(0).copy()
            df_report["dia_operacional"] = pd.to_datetime([], errors="coerce")
            df_report[irregularidade_col] = False
            for _k, _v in params_export.items():
                df_report[_k] = _v

        gerar_relatorio_pdf(df_report, titulo=titulo, pdf_path=pdf_path)

        gc.collect()
        print("Parquet:", parquet_path)
        print("PDF   :", pdf_path)
        return df_report

    df_out = df_min_raw.iloc[row_ids].copy()
    df_out["dia_operacional"] = pd.to_datetime(dia_op_arr[row_ids], errors="coerce")
    df_out[irregularidade_col] = True
    for _k, _v in params_export.items():
        df_out[_k] = _v
    df_out.drop(columns=["_row_id"], inplace=True, errors="ignore")

    df_out.to_parquet(parquet_path, index=False)
    gerar_relatorio_pdf(df_out, titulo=titulo, pdf_path=pdf_path)

    gc.collect()
    print("Parquet:", parquet_path)
    print("PDF   :", pdf_path)
    return df_out


# =========================
# <<< ALTERADO: Semanas de feriado prolongado (2024, 2025, 2026) com feriado na TERÇA ou QUINTA
# =========================
from dateutil.easter import easter

FERIADOS_INFO = "Esses feriados são os de 2024, 2025 e 2026."
ANOS_FERIADOS = (2024, 2025, 2026)

def feriados_br_df(anos=ANOS_FERIADOS) -> pd.DataFrame:
    rows = []

    fixos = [
        (1, 1,  "Confraternização Universal"),
        (4, 21, "Tiradentes"),
        (5, 1,  "Dia do Trabalho"),
        (9, 7,  "Independência do Brasil"),
        (10, 12,"Nossa Senhora Aparecida"),
        (11, 2, "Finados"),
        (11, 15,"Proclamação da República"),
        (12, 25,"Natal"),
    ]

    for ano in anos:
        for m, d, nome in fixos:
            rows.append((pd.Timestamp(year=ano, month=m, day=d), nome))

        # Consciência Negra (feriado nacional a partir de 2024)
        if ano >= 2024:
            rows.append((pd.Timestamp(year=ano, month=11, day=20), "Consciência Negra"))

        pascoa = pd.Timestamp(easter(ano))
        rows.append((pascoa - pd.Timedelta(days=2), "Sexta-feira Santa"))
        rows.append((pascoa + pd.Timedelta(days=60), "Corpus Christi"))

        # Carnaval (terça) — comum em calendário corporativo; mantém terça conforme solicitado
        rows.append((pascoa - pd.Timedelta(days=47), "Carnaval (terça)"))

    fer = pd.DataFrame(rows, columns=["data", "nome"])
    fer["data"] = pd.to_datetime(fer["data"], errors="coerce").dt.normalize()
    fer = fer.dropna(subset=["data"]).drop_duplicates(subset=["data"]).sort_values("data").reset_index(drop=True)
    return fer

feriados = feriados_br_df()

# filtra feriados que caem na terça (1) ou quinta (3)
fer_ter_qui = feriados.loc[feriados["data"].dt.dayofweek.isin([1, 3])].copy()

# semana considerada: segunda..sexta (para a regra de 3 dias presenciais)
fer_ter_qui["semana_ini"] = (fer_ter_qui["data"] - pd.to_timedelta(fer_ter_qui["data"].dt.dayofweek, unit="D")).dt.normalize()
fer_ter_qui["semana_fim"] = fer_ter_qui["semana_ini"] + pd.Timedelta(days=4)

semanas_feriado_prolongado = (
    fer_ter_qui[["semana_ini", "semana_fim"]]
    .drop_duplicates()
    .sort_values("semana_ini")
    .itertuples(index=False, name=None)
)
semanas_feriado_prolongado = list(semanas_feriado_prolongado)

# Reusa os helpers modernos já definidos anteriormente no notebook.
if "_gerar_relatorio_pdf_moderno" in globals():
    gerar_relatorio_pdf = _gerar_relatorio_pdf_moderno
if "_row_ids_from_keys_moderno" in globals():
    row_ids_from_keys = _row_ids_from_keys_moderno
if "_materializar_e_exportar_por_rowid_moderno" in globals():
    materializar_e_exportar_por_rowid = _materializar_e_exportar_por_rowid_moderno


In [ ]:
# =========================
# IRREGULARIDADE: Semana Feriado Prolongado (menos de 3 dias presenciais na semana)
# =========================
irregularidade_col = "Irregularidade - Pessoas sem cumprir os 3 Dias Presenciais em Semana de Feriado Prolongado"
titulo = (
    "Irregularidade – Semana de Feriado Prolongado "
    "(menos de 3 dias presenciais na semana)\n"
    + FERIADOS_INFO
)

day_ = day.copy()
if "cadastro" in day_.columns:
    day_["cadastro"] = day_["cadastro"].astype(str).str.strip()

# Marca semana de feriado por data real do registro, sem copiar df inteiro
semana_ini = pd.Series(pd.NaT, index=df_valid.index)
dia_key_base = (
    pd.to_datetime(df_valid["_dia_key"], errors="coerce").dt.normalize()
    if "_dia_key" in df_valid.columns
    else pd.to_datetime(df_valid["_dt_ronda"], errors="coerce").dt.normalize()
)

for ini_w, fim_w in semanas_feriado_prolongado:
    m_w = dia_key_base.between(ini_w, fim_w)
    semana_ini.loc[m_w] = ini_w

m_feriado = semana_ini.notna()
cols_base = ["cadastro", "_dia_operacional", "_dt_ronda", "Presencial", "Portaria", "_dir_norm"]
gw = df_valid.loc[m_feriado, cols_base].copy()
gw["_semana_feriado_ini"] = semana_ini.loc[m_feriado].to_numpy()
gw["_dia_key"] = dia_key_base.loc[m_feriado].to_numpy()

gw["cadastro"] = gw["cadastro"].astype(str).str.strip()

# Tenta trazer nome/área se existirem em df_valid; fallback por cadastro usando df.
if "nome" in df_valid.columns:
    gw["nome"] = df_valid.loc[m_feriado, "nome"].to_numpy()
else:
    gw["nome"] = None

if "área" in df_valid.columns:
    gw["area"] = df_valid.loc[m_feriado, "área"].to_numpy()
elif "area" in df_valid.columns:
    gw["area"] = df_valid.loc[m_feriado, "area"].to_numpy()
else:
    gw["area"] = None

if ("nome" in df.columns) and (gw["nome"].isna().all() or gw["nome"].astype(str).str.strip().eq("").all()):
    tmp_nome_src = df.loc[df["nome"].notna(), ["cadastro", "nome"]].copy()
    if not tmp_nome_src.empty:
        tmp_nome_src["cadastro"] = tmp_nome_src["cadastro"].astype(str).str.strip()
        tmp_nome_src["nome"] = tmp_nome_src["nome"].astype(str).str.strip()
        tmp_nome_src = tmp_nome_src[(tmp_nome_src["cadastro"].ne("")) & (tmp_nome_src["nome"].ne(""))]
        if not tmp_nome_src.empty:
            cnt_nome_src = (
                tmp_nome_src.groupby(["cadastro", "nome"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
            )
            nome_map_src = cnt_nome_src.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["nome"].to_dict()
            gw["nome"] = gw["cadastro"].map(nome_map_src)

col_area_src = "área" if "área" in df.columns else ("area" if "area" in df.columns else None)
if (col_area_src is not None) and (gw["area"].isna().all() or gw["area"].astype(str).str.strip().eq("").all()):
    tmp_area_src = df.loc[df[col_area_src].notna(), ["cadastro", col_area_src]].copy()
    tmp_area_src = tmp_area_src.rename(columns={col_area_src: "area"})
    if not tmp_area_src.empty:
        tmp_area_src["cadastro"] = tmp_area_src["cadastro"].astype(str).str.strip()
        tmp_area_src["area"] = tmp_area_src["area"].astype(str).str.strip()
        tmp_area_src = tmp_area_src[(tmp_area_src["cadastro"].ne("")) & (tmp_area_src["area"].ne(""))]
        if not tmp_area_src.empty:
            cnt_area_src = (
                tmp_area_src.groupby(["cadastro", "area"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
            )
            area_map_src = cnt_area_src.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["area"].to_dict()
            gw["area"] = gw["cadastro"].map(area_map_src)

if gw.empty:
    day_[irregularidade_col] = False
    df_relatorio_feriado = pd.DataFrame(columns=[
        "cadastro", "nome", "area", "semana_inicio", "semana_fim",
        "dias_com_registro", "dias_presenciais", "horas_presenciais", "nao_cumpriu_3"
    ])
else:
    # Portaria E->S por dia (dentro da semana do feriado)
    port = gw.loc[gw["Portaria"].astype(bool) & gw["_dt_ronda"].notna()].copy()

    # quantidade de dias com qualquer registro na semana
    dias_com_registro = (
        gw.groupby(["cadastro", "_semana_feriado_ini"], sort=False)["_dia_operacional"]
          .nunique()
          .rename("dias_com_registro")
          .reset_index()
    )

    # nome/área mais representativos por cadastro
    nome_map = {}
    area_map = {}
    cads_semana = gw["cadastro"].dropna().astype(str).str.strip().unique()

    if "nome" in gw.columns:
        tmp_nome = gw.loc[gw["nome"].notna(), ["cadastro", "nome"]].copy()
        if not tmp_nome.empty:
            tmp_nome["nome"] = tmp_nome["nome"].astype(str).str.strip()
            tmp_nome = tmp_nome[tmp_nome["nome"].ne("")]
            if not tmp_nome.empty:
                cnt_nome = (
                    tmp_nome.groupby(["cadastro", "nome"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
                )
                nome_map = cnt_nome.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["nome"].to_dict()

    if "area" in gw.columns:
        tmp_area = gw.loc[gw["area"].notna(), ["cadastro", "area"]].copy()
        if not tmp_area.empty:
            tmp_area["area"] = tmp_area["area"].astype(str).str.strip()
            tmp_area = tmp_area[tmp_area["area"].ne("")]
            if not tmp_area.empty:
                cnt_area = (
                    tmp_area.groupby(["cadastro", "area"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
                )
                area_map = cnt_area.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["area"].to_dict()

    # Critério principal da irregularidade: quantidade de dias presenciais na semana.
    dias_presenciais_semana = (
        gw.loc[
            gw["Presencial"].fillna(False).astype(bool),
            ["cadastro", "_semana_feriado_ini", "_dia_operacional"]
        ]
        .dropna(subset=["_dia_operacional"])
        .drop_duplicates()
        .groupby(["cadastro", "_semana_feriado_ini"], sort=False)
        .size()
        .rename("dias_presenciais")
        .reset_index()
    )

    wk = dias_com_registro.merge(
        dias_presenciais_semana,
        on=["cadastro", "_semana_feriado_ini"],
        how="left",
    )
    wk["dias_presenciais"] = pd.to_numeric(wk["dias_presenciais"], errors="coerce").fillna(0).astype(int)

    # Horas presenciais ficam como métrica auxiliar (não entram no critério principal).
    if port.empty:
        wk["horas_presenciais"] = 0.0
    else:
        first_E = (
            port.loc[port["_dir_norm"].astype(str).eq("E")]
                .groupby(["cadastro", "_semana_feriado_ini", "_dia_key"], sort=False)["_dt_ronda"]
                .min()
                .rename("first_port_E")
                .reset_index()
        )
        last_S = (
            port.loc[port["_dir_norm"].astype(str).eq("S")]
                .groupby(["cadastro", "_semana_feriado_ini", "_dia_key"], sort=False)["_dt_ronda"]
                .max()
                .rename("last_port_S")
                .reset_index()
        )

        day_dur = first_E.merge(
            last_S,
            on=["cadastro", "_semana_feriado_ini", "_dia_key"],
            how="inner",
        )
        day_dur["dur"] = day_dur["last_port_S"] - day_dur["first_port_E"]
        day_dur["horas_dia"] = (day_dur["dur"] / pd.Timedelta(hours=1)).where(
            day_dur["dur"].notna() & (day_dur["dur"] > pd.Timedelta(0)),
            0.0,
        ).fillna(0.0)

        wk_horas = (
            day_dur.groupby(["cadastro", "_semana_feriado_ini"], sort=False)["horas_dia"]
                  .sum()
                  .rename("horas_presenciais")
                  .reset_index()
        )

        wk = wk.merge(wk_horas, on=["cadastro", "_semana_feriado_ini"], how="left")
        wk["horas_presenciais"] = pd.to_numeric(wk["horas_presenciais"], errors="coerce").fillna(0.0)

    wk["nao_cumpriu_3"] = wk["dias_presenciais"] < 3

    df_relatorio_feriado = (
        wk.loc[wk["nao_cumpriu_3"]].copy()
          .rename(columns={"_semana_feriado_ini": "semana_inicio"})
    )
    df_relatorio_feriado["semana_inicio"] = pd.to_datetime(df_relatorio_feriado["semana_inicio"], errors="coerce").dt.normalize()
    df_relatorio_feriado["semana_fim"] = df_relatorio_feriado["semana_inicio"] + pd.Timedelta(days=6)
    df_relatorio_feriado["cadastro"] = df_relatorio_feriado["cadastro"].astype(str).str.strip()
    df_relatorio_feriado["nome"] = df_relatorio_feriado["cadastro"].map(nome_map).fillna("Sem Nome")
    df_relatorio_feriado["area"] = df_relatorio_feriado["cadastro"].map(area_map).fillna("Sem Área")

    df_relatorio_feriado = df_relatorio_feriado.loc[:, [
        "cadastro", "nome", "area", "semana_inicio", "semana_fim",
        "dias_com_registro", "dias_presenciais", "horas_presenciais", "nao_cumpriu_3"
    ]].copy()

    # marca os dias (dia_operacional) que caem nessas semanas irregulares
    wk_irregularidade = (
        df_relatorio_feriado.loc[:, ["cadastro", "semana_inicio"]]
                            .rename(columns={"semana_inicio": "_semana_feriado_ini"})
                            .drop_duplicates()
    )

    if wk_irregularidade.empty:
        day_[irregularidade_col] = False
    else:
        dias_flag = (
            gw.merge(wk_irregularidade, on=["cadastro", "_semana_feriado_ini"], how="inner")
              .loc[:, ["cadastro", "_dia_operacional"]]
              .drop_duplicates()
        )

        day_ = day_.merge(
            dias_flag.assign(_flag_feriado=True),
            on=["cadastro", "_dia_operacional"],
            how="left"
        )
        day_[irregularidade_col] = day_["_flag_feriado"].fillna(False).astype(bool)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()

row_ids = row_ids_from_keys(df, keys)

df_feriado = materializar_e_exportar_por_rowid(
    row_ids,
    irregularidade_col=irregularidade_col,
    titulo=titulo,
)

# Enriquecimento final de nome/área usando a base materializada da própria irregularidade.
if isinstance(df_relatorio_feriado, pd.DataFrame) and (not df_relatorio_feriado.empty) and isinstance(df_feriado, pd.DataFrame) and (not df_feriado.empty):
    if "cadastro" in df_relatorio_feriado.columns:
        df_relatorio_feriado["cadastro"] = df_relatorio_feriado["cadastro"].astype(str).str.strip()

    if "nome" in df_feriado.columns:
        tmp_nome = df_feriado.loc[df_feriado["nome"].notna(), ["cadastro", "nome"]].copy()
        if not tmp_nome.empty:
            tmp_nome["cadastro"] = tmp_nome["cadastro"].astype(str).str.strip()
            tmp_nome["nome"] = tmp_nome["nome"].astype(str).str.strip()
            tmp_nome = tmp_nome[tmp_nome["nome"].ne("")]
            if not tmp_nome.empty:
                cnt_nome = (
                    tmp_nome.groupby(["cadastro", "nome"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
                )
                nome_map_final = cnt_nome.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["nome"].to_dict()
                if nome_map_final:
                    nome_atual = df_relatorio_feriado["nome"].fillna("Sem Nome").astype(str)
                    df_relatorio_feriado["nome"] = (
                        df_relatorio_feriado["cadastro"].map(nome_map_final)
                        .where(df_relatorio_feriado["cadastro"].map(nome_map_final).notna(), nome_atual)
                        .fillna("Sem Nome")
                    )

    col_area_mat = "área" if "área" in df_feriado.columns else ("area" if "area" in df_feriado.columns else None)
    if col_area_mat is not None:
        tmp_area = df_feriado.loc[df_feriado[col_area_mat].notna(), ["cadastro", col_area_mat]].copy()
        tmp_area = tmp_area.rename(columns={col_area_mat: "area"})
        if not tmp_area.empty:
            tmp_area["cadastro"] = tmp_area["cadastro"].astype(str).str.strip()
            tmp_area["area"] = tmp_area["area"].astype(str).str.strip()
            tmp_area = tmp_area[tmp_area["area"].ne("")]
            if not tmp_area.empty:
                cnt_area = (
                    tmp_area.groupby(["cadastro", "area"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
                )
                area_map_final = cnt_area.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["area"].to_dict()
                if area_map_final:
                    area_atual = df_relatorio_feriado["area"].fillna("Sem Área").astype(str)
                    df_relatorio_feriado["area"] = (
                        df_relatorio_feriado["cadastro"].map(area_map_final)
                        .where(df_relatorio_feriado["cadastro"].map(area_map_final).notna(), area_atual)
                        .fillna("Sem Área")
                    )

# sobrescreve o PDF com o relatório semanal
pdf_path = PDF_DIR / f"{slugify(irregularidade_col)}.pdf"
gerar_relatorio_pdf(
    df_relatorio_feriado,
    titulo=titulo,
    pdf_path=pdf_path,
)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_feriado_prolongado = df_feriado

df_feriado.head(3)


In [ ]:
# =========================
# IRREGULARIDADE: Não entrou no refeitório, mas passou no grill
# Regra:
# - houve GRILL no dia
# - NÃO houve REFEITÓRIO NORMAL antes do primeiro GRILL
# - não acusar em borda de contexto
# =========================
irregularidade_col = "Irregularidade - Não entrou no refeitório, mas passou no grill"
titulo = "Irregularidade – Não entrou no refeitório, mas passou no grill"

day_ = day.copy()

# Usa o pré-cálculo já feito no day (evita normalização pesada da base completa).
if {"HAS_GRILL_ANY", "HAS_REFEITORIO_BEFORE_GRILL"}.issubset(day_.columns):
    day_[irregularidade_col] = (
        (~day_["SEM_COBERTURA_CONTEXTO"].fillna(False).astype(bool))
        & day_["HAS_GRILL_ANY"].fillna(False).astype(bool)
        & (~day_["HAS_REFEITORIO_BEFORE_GRILL"].fillna(False).astype(bool))
    )
else:
    # Fallback mínimo caso o day não tenha as flags calculadas.
    day_[irregularidade_col] = False

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()
row_ids = row_ids_from_keys(df, keys)

df_ref_grill = materializar_e_exportar_por_rowid(
    row_ids,
    irregularidade_col=irregularidade_col,
    titulo=titulo,
)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_ref_grill = df_ref_grill

df_ref_grill.head(3)


In [ ]:
# =========================
# IRREGULARIDADE: Ausência das horas Presenciais Mínimas (SEMANAL)
# Regra: horas presenciais semanais = soma(Portaria S - Portaria E) por dia.
# Não contar executivos. Não acusar em borda de contexto.
#
# NECESSÁRIO: definir min_horas_semana (default 24.0)
# =========================
irregularidade_col = "Irregularidade - Ausência das horas Presenciais Mínimas"
titulo = "Irregularidade – Ausência de Horas Presenciais Mínimas (semanal; Portaria E→S)"

MIN_HORAS_SEMANA = float(globals().get("min_horas_semana", 24.0))

day_ = day.copy()

port = df_valid.loc[
    df_valid["Portaria"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]
].copy(deep=False)

first_E = (
    port.loc[port["_dir_norm"].eq("E")]
        .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
        .min()
        .rename("first_port_E")
        .reset_index()
)
last_S = (
    port.loc[port["_dir_norm"].eq("S")]
        .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
        .max()
        .rename("last_port_S")
        .reset_index()
)

day_ = day_.merge(first_E, on=["cadastro", "_dia_operacional"], how="left")
day_ = day_.merge(last_S,  on=["cadastro", "_dia_operacional"], how="left")

dur_port = day_["last_port_S"] - day_["first_port_E"]
day_["_horas_pres_dia"] = (dur_port / pd.Timedelta(hours=1)).where(
    dur_port.notna() & (dur_port >= pd.Timedelta(0)),
    0.0
).fillna(0.0)

dia = pd.to_datetime(day_["_dia_operacional"], errors="coerce")
day_["_sem_ini"] = (dia - pd.to_timedelta(dia.dt.dayofweek, unit="D")).dt.normalize()

base = day_.loc[
    (~day_["IS_EXECUTIVO_DAY"].astype(bool))
    & (~day_["SEM_COBERTURA_CONTEXTO"]),
    ["cadastro", "_dia_operacional", "_sem_ini", "_horas_pres_dia"]
].copy()

semanal = (
    base.groupby(["cadastro", "_sem_ini"], as_index=False)["_horas_pres_dia"]
        .sum()
        .rename(columns={"_horas_pres_dia": "horas_pres_semana"})
)

semanal["FLAG_MIN_HORAS"] = semanal["horas_pres_semana"] < MIN_HORAS_SEMANA

keys_sem = semanal.loc[semanal["FLAG_MIN_HORAS"], ["cadastro", "_sem_ini"]].drop_duplicates()

if keys_sem.empty:
    keys = day_.head(0)[["cadastro", "_dia_operacional"]].copy()
else:
    day_flag = day_.merge(keys_sem.assign(_flag=True), on=["cadastro", "_sem_ini"], how="inner")
    keys = day_flag.loc[:, ["cadastro", "_dia_operacional"]].drop_duplicates()

row_ids = row_ids_from_keys(df, keys)
df_min_horas = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_horas_minimas = df_min_horas

df_min_horas.head(3)

## 8) Resumo de arquivos gerados


In [ ]:
# Lista PDFs e parquets gerados
pdfs = sorted(PDF_DIR.glob("*.pdf"))
pars = sorted(DFS_DIR.glob("*/*.parquet"))

print(f"PDFs gerados: {len(pdfs)}")
for p in pdfs:
    print(" -", p)

print(f"Parquets gerados: {len(pars)}")
for p in pars:
    print(" -", p)
